In [1]:
# ============================================================
# V2 - CELL 1
# IMPORTS & CONFIGURATION
# ============================================================

import os
import gc
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ks_2samp

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

print("=" * 70)
print("DS KAGGLE MIT-WPU 2026 — OPTIMIZATION V2")
print("=" * 70)

print("\nRandom state:", RANDOM_STATE)
print("V2 objective:")
print("1. Better validation")
print("2. Exploit repeated sensor patterns")
print("3. Exploit date-level anomaly behavior")
print("4. Improve leaderboard F1")

DS KAGGLE MIT-WPU 2026 — OPTIMIZATION V2

Random state: 42
V2 objective:
1. Better validation
2. Exploit repeated sensor patterns
3. Exploit date-level anomaly behavior
4. Improve leaderboard F1


In [2]:
# ============================================================
# V2 - CELL 2
# LOAD COMPETITION DATA
# ============================================================

DATA_DIR = "/kaggle/input/competitions/ds-kaggle-mit-wpu"

TRAIN_PATH = os.path.join(
    DATA_DIR,
    "train.parquet"
)

TEST_PATH = os.path.join(
    DATA_DIR,
    "test.parquet"
)

SAMPLE_PATH = os.path.join(
    DATA_DIR,
    "sample_submission.parquet"
)


print("=" * 70)
print("LOADING COMPETITION DATA")
print("=" * 70)


train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)
sample_submission = pd.read_parquet(SAMPLE_PATH)


print("\nTrain shape:")
print(train.shape)

print("\nTest shape:")
print(test.shape)

print("\nSample submission shape:")
print(sample_submission.shape)


print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

print("\nSample columns:")
print(sample_submission.columns.tolist())


# ------------------------------------------------------------
# Target
# ------------------------------------------------------------

target_column = "target"

assert target_column in train.columns

print("\nTarget column:", target_column)

print("\nTarget distribution:")
display(
    train[target_column]
    .value_counts()
    .sort_index()
)


print("\nTarget percentages:")
display(
    train[target_column]
    .value_counts(
        normalize=True
    )
    .sort_index()
    .mul(100)
    .round(4)
)


# ------------------------------------------------------------
# Sensor columns
# ------------------------------------------------------------

sensor_columns = [
    "X1",
    "X2",
    "X3",
    "X4",
    "X5"
]

print("\nSensor columns:")
print(sensor_columns)


# ------------------------------------------------------------
# Clean memory
# ------------------------------------------------------------

gc.collect()

print("\n✓ Data loaded successfully.")

LOADING COMPETITION DATA

Train shape:
(1639424, 7)

Test shape:
(409856, 7)

Sample submission shape:
(50, 2)

Train columns:
['Date', 'X1', 'X2', 'X3', 'X4', 'X5', 'target']

Test columns:
['ID', 'Date', 'X1', 'X2', 'X3', 'X4', 'X5']

Sample columns:
['ID', 'target']

Target column: target

Target distribution:


target
0    1625386
1      14038
Name: count, dtype: int64


Target percentages:


target
0    99.1437
1     0.8563
Name: proportion, dtype: float64


Sensor columns:
['X1', 'X2', 'X3', 'X4', 'X5']

✓ Data loaded successfully.


In [3]:
# ============================================================
# V2 - CELL 3
# DATE FEATURE ENGINEERING
# ============================================================

print("=" * 70)
print("DATE FEATURE ENGINEERING")
print("=" * 70)


def add_date_features(df):
    
    df = df.copy()
    
    # Make sure Date is datetime
    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="coerce"
    )
    
    # Calendar features
    df["year"] = df["Date"].dt.year.astype("int16")
    df["month"] = df["Date"].dt.month.astype("int8")
    df["day"] = df["Date"].dt.day.astype("int8")
    df["day_of_week"] = (
        df["Date"]
        .dt.dayofweek
        .astype("int8")
    )
    
    df["day_of_year"] = (
        df["Date"]
        .dt.dayofyear
        .astype("int16")
    )
    
    df["quarter"] = (
        df["Date"]
        .dt.quarter
        .astype("int8")
    )
    
    
    # Cyclic encoding
    df["month_sin"] = np.sin(
        2 * np.pi * df["month"] / 12
    )
    
    df["month_cos"] = np.cos(
        2 * np.pi * df["month"] / 12
    )
    
    df["day_of_week_sin"] = np.sin(
        2 * np.pi * df["day_of_week"] / 7
    )
    
    df["day_of_week_cos"] = np.cos(
        2 * np.pi * df["day_of_week"] / 7
    )
    
    
    # Cyclic day-of-year
    df["day_of_year_sin"] = np.sin(
        2 * np.pi * df["day_of_year"] / 365.25
    )
    
    df["day_of_year_cos"] = np.cos(
        2 * np.pi * df["day_of_year"] / 365.25
    )
    
    return df


train_v2 = add_date_features(train)
test_v2 = add_date_features(test)


print("\nTrain shape after date features:")
print(train_v2.shape)


print("\nTest shape after date features:")
print(test_v2.shape)


print("\nNew date features:")

date_features = [
    "year",
    "month",
    "day",
    "day_of_week",
    "day_of_year",
    "quarter",
    "month_sin",
    "month_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "day_of_year_sin",
    "day_of_year_cos"
]

print(date_features)


print("\nDate range:")
print(
    train_v2["Date"].min(),
    "→",
    train_v2["Date"].max()
)


print("\n✓ Date features created.")

DATE FEATURE ENGINEERING

Train shape after date features:
(1639424, 19)

Test shape after date features:
(409856, 19)

New date features:
['year', 'month', 'day', 'day_of_week', 'day_of_year', 'quarter', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin', 'day_of_year_cos']

Date range:
2020-12-16 00:00:00 → 2024-12-11 00:00:00

✓ Date features created.


In [4]:
# ============================================================
# V2 - CELL 4
# SENSOR REGIME FEATURES
# ============================================================

print("=" * 70)
print("SENSOR REGIME FEATURE ENGINEERING")
print("=" * 70)


def add_sensor_regime_features(df):
    
    df = df.copy()
    
    for col in sensor_columns:
        
        # Absolute magnitude
        abs_value = np.abs(
            df[col].astype("float64")
        )
        
        # Stable signed logarithm
        df[f"{col}_log_abs"] = (
            np.sign(
                df[col]
            )
            *
            np.log1p(abs_value)
        )
        
        # Magnitude order
        df[f"{col}_log10_abs"] = np.log10(
            abs_value + 1e-12
        )
        
        # Useful regime indicators
        df[f"{col}_is_zero"] = (
            df[col] == 0
        ).astype("int8")
        
        df[f"{col}_is_one"] = (
            np.isclose(
                df[col],
                1.0
            )
        ).astype("int8")
        
        # Large-magnitude indicators
        df[f"{col}_large"] = (
            abs_value > 100
        ).astype("int8")
        
        df[f"{col}_extreme"] = (
            abs_value > 1e6
        ).astype("int8")
    
    return df


train_v2 = add_sensor_regime_features(
    train_v2
)

test_v2 = add_sensor_regime_features(
    test_v2
)


regime_features = []

for col in sensor_columns:
    
    regime_features.extend([
        f"{col}_log_abs",
        f"{col}_log10_abs",
        f"{col}_is_zero",
        f"{col}_is_one",
        f"{col}_large",
        f"{col}_extreme"
    ])


print(
    "\nNumber of regime features:",
    len(regime_features)
)


print("\nRegime features:")
print(regime_features)


print("\n✓ Extreme values preserved.")
print("✓ No outliers removed.")
print("✓ Magnitude information converted into model features.")

SENSOR REGIME FEATURE ENGINEERING

Number of regime features: 30

Regime features:
['X1_log_abs', 'X1_log10_abs', 'X1_is_zero', 'X1_is_one', 'X1_large', 'X1_extreme', 'X2_log_abs', 'X2_log10_abs', 'X2_is_zero', 'X2_is_one', 'X2_large', 'X2_extreme', 'X3_log_abs', 'X3_log10_abs', 'X3_is_zero', 'X3_is_one', 'X3_large', 'X3_extreme', 'X4_log_abs', 'X4_log10_abs', 'X4_is_zero', 'X4_is_one', 'X4_large', 'X4_extreme', 'X5_log_abs', 'X5_log10_abs', 'X5_is_zero', 'X5_is_one', 'X5_large', 'X5_extreme']

✓ Extreme values preserved.
✓ No outliers removed.
✓ Magnitude information converted into model features.


In [5]:
# ============================================================
# V2 - CELL 5
# REPEATED SENSOR / SENSOR-DATE COVERAGE
# ============================================================

print("=" * 70)
print("REPEATED SENSOR PATTERN ANALYSIS")
print("=" * 70)


# ------------------------------------------------------------
# Sensor-only hash
# ------------------------------------------------------------

train_sensor_hash = pd.util.hash_pandas_object(
    train[sensor_columns],
    index=False
).astype("uint64")


test_sensor_hash = pd.util.hash_pandas_object(
    test[sensor_columns],
    index=False
).astype("uint64")


train_sensor_hash_set = set(
    train_sensor_hash.unique()
)


sensor_seen_mask = (
    test_sensor_hash
    .isin(train_sensor_hash_set)
)


sensor_seen_count = sensor_seen_mask.sum()

sensor_seen_percentage = (
    sensor_seen_mask.mean() * 100
)


print("\nSENSOR-ONLY MATCHING")
print("-" * 50)

print(
    f"Test rows with sensor combination seen in train: "
    f"{sensor_seen_count:,}"
)

print(
    f"Percentage of test rows: "
    f"{sensor_seen_percentage:.4f}%"
)


# ------------------------------------------------------------
# Sensor + Date hash
# ------------------------------------------------------------

train_sensor_date = train[
    sensor_columns + ["Date"]
].copy()

test_sensor_date = test[
    sensor_columns + ["Date"]
].copy()


train_sensor_date_hash = (
    pd.util.hash_pandas_object(
        train_sensor_date,
        index=False
    )
    .astype("uint64")
)


test_sensor_date_hash = (
    pd.util.hash_pandas_object(
        test_sensor_date,
        index=False
    )
    .astype("uint64")
)


train_sensor_date_hash_set = set(
    train_sensor_date_hash.unique()
)


sensor_date_seen_mask = (
    test_sensor_date_hash
    .isin(
        train_sensor_date_hash_set
    )
)


sensor_date_seen_count = (
    sensor_date_seen_mask.sum()
)


sensor_date_seen_percentage = (
    sensor_date_seen_mask.mean() * 100
)


print("\nSENSOR + DATE MATCHING")
print("-" * 50)

print(
    f"Test rows with sensor+date combination seen in train: "
    f"{sensor_date_seen_count:,}"
)

print(
    f"Percentage of test rows: "
    f"{sensor_date_seen_percentage:.4f}%"
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

coverage_summary = pd.DataFrame({
    
    "pattern": [
        "Sensor combination",
        "Sensor + Date combination"
    ],
    
    "test_rows_matched": [
        sensor_seen_count,
        sensor_date_seen_count
    ],
    
    "test_coverage_%": [
        sensor_seen_percentage,
        sensor_date_seen_percentage
    ]
})


print("\nCOVERAGE SUMMARY")
display(coverage_summary)


# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

del train_sensor_hash
del test_sensor_hash
del train_sensor_date_hash
del test_sensor_date_hash

gc.collect()


print(
    "\n✓ Repeated-pattern analysis completed."
)

REPEATED SENSOR PATTERN ANALYSIS

SENSOR-ONLY MATCHING
--------------------------------------------------
Test rows with sensor combination seen in train: 253,018
Percentage of test rows: 61.7334%

SENSOR + DATE MATCHING
--------------------------------------------------
Test rows with sensor+date combination seen in train: 27,183
Percentage of test rows: 6.6323%

COVERAGE SUMMARY


,pattern,test_rows_matched,test_coverage_%
0,Sensor combination,253018,61.733389
1,Sensor + Date combination,27183,6.632329



✓ Repeated-pattern analysis completed.


In [6]:
# ============================================================
# V2 - CELL 6
# COMPETITION-LIKE DATE-WISE VALIDATION SPLIT
# ============================================================

print("=" * 70)
print("CREATING DATE-WISE VALIDATION SPLIT")
print("=" * 70)


# ------------------------------------------------------------
# Preserve original row order
# ------------------------------------------------------------

train_v2["_original_row"] = np.arange(
    len(train_v2),
    dtype=np.int32
)


# ------------------------------------------------------------
# Position within each date
# ------------------------------------------------------------

train_v2["_row_in_date"] = (
    train_v2
    .groupby("Date", sort=False)
    .cumcount()
)


# ------------------------------------------------------------
# Deterministic 80/20 split
# ------------------------------------------------------------

# Every 5th observation goes to validation.
#
# This keeps ALL dates represented in both sets.

validation_mask = (
    train_v2["_row_in_date"] % 5 == 0
)


train_mask = ~validation_mask


train_indices = np.flatnonzero(
    train_mask
)

validation_indices = np.flatnonzero(
    validation_mask
)


print("\nTraining observations:")
print(
    f"{len(train_indices):,}"
)


print("\nValidation observations:")
print(
    f"{len(validation_indices):,}"
)


print("\nTraining percentage:")
print(
    f"{len(train_indices) / len(train_v2) * 100:.2f}%"
)


print("\nValidation percentage:")
print(
    f"{len(validation_indices) / len(train_v2) * 100:.2f}%"
)


# ------------------------------------------------------------
# Target distribution
# ------------------------------------------------------------

y_train_v2 = (
    train_v2
    .iloc[train_indices][target_column]
    .astype(str)
)


y_valid_v2 = (
    train_v2
    .iloc[validation_indices][target_column]
    .astype(str)
)


validation_distribution = pd.DataFrame({
    
    "train_count":
        y_train_v2.value_counts(),
    
    "validation_count":
        y_valid_v2.value_counts()
    
}).fillna(0).astype(int)


validation_distribution[
    "train_%"
] = (
    validation_distribution[
        "train_count"
    ]
    /
    validation_distribution[
        "train_count"
    ].sum()
    * 100
)


validation_distribution[
    "validation_%"
] = (
    validation_distribution[
        "validation_count"
    ]
    /
    validation_distribution[
        "validation_count"
    ].sum()
    * 100
)


print("\nTARGET DISTRIBUTION")
display(
    validation_distribution
)


# ------------------------------------------------------------
# Date coverage check
# ------------------------------------------------------------

train_dates = set(
    train_v2.iloc[
        train_indices
    ]["Date"]
)


valid_dates = set(
    train_v2.iloc[
        validation_indices
    ]["Date"]
)


print(
    "\nUnique dates in training:",
    len(train_dates)
)


print(
    "Unique dates in validation:",
    len(valid_dates)
)


print(
    "Dates shared between train/validation:",
    len(
        train_dates.intersection(
            valid_dates
        )
    )
)


assert (
    len(
        train_dates.intersection(
            valid_dates
        )
    )
    ==
    len(train_dates)
)


print(
    "\n✓ Every validation date is represented "
    "in the training portion."
)


print(
    "\n✓ Date-wise validation split created."
)

CREATING DATE-WISE VALIDATION SPLIT

Training observations:
1,310,948

Validation observations:
328,476

Training percentage:
79.96%

Validation percentage:
20.04%

TARGET DISTRIBUTION


,train_count,validation_count,train_%,validation_%
target,,,,
0,1299728,325658,99.144131,99.142099
1,11220,2818,0.855869,0.857901



Unique dates in training: 1432
Unique dates in validation: 1432
Dates shared between train/validation: 1432

✓ Every validation date is represented in the training portion.

✓ Date-wise validation split created.


In [7]:
# ============================================================
# V2 - CELL 7
# LEAKAGE-SAFE DATE TARGET ENCODING
# ============================================================

print("=" * 70)
print("LEAKAGE-SAFE DATE TARGET ENCODING")
print("=" * 70)


# ------------------------------------------------------------
# Convert target to numeric
# ------------------------------------------------------------

y_numeric = (
    train_v2[target_column]
    .astype(str)
    .astype(int)
)


# ------------------------------------------------------------
# Build training-fold date statistics
# ------------------------------------------------------------

train_fold = train_v2.iloc[
    train_indices
].copy()


train_fold["_target_numeric"] = (
    y_numeric.iloc[
        train_indices
    ].values
)


date_statistics = (
    train_fold
    .groupby("Date")["_target_numeric"]
    .agg(
        date_anomaly_count="sum",
        date_observation_count="count"
    )
)


date_statistics[
    "date_anomaly_rate"
] = (
    date_statistics[
        "date_anomaly_count"
    ]
    /
    date_statistics[
        "date_observation_count"
    ]
)


# ------------------------------------------------------------
# Global training anomaly rate
# ------------------------------------------------------------

global_anomaly_rate = (
    train_fold[
        "_target_numeric"
    ].mean()
)


print(
    "\nGlobal training anomaly rate:",
    f"{global_anomaly_rate:.6%}"
)


# ------------------------------------------------------------
# Apply date statistics to validation
# ------------------------------------------------------------

valid_dates_series = (
    train_v2
    .iloc[
        validation_indices
    ]["Date"]
)


valid_date_features = (
    valid_dates_series
    .map(
        date_statistics[
            "date_anomaly_rate"
        ]
    )
)


# All dates should have training observations
missing_date_encoding = (
    valid_date_features
    .isna()
    .sum()
)


print(
    "\nValidation rows without date encoding:",
    missing_date_encoding
)


# ------------------------------------------------------------
# Apply date statistics to training
#
# IMPORTANT:
# For training rows, use leave-one-out encoding.
# This prevents a row's own target from entering its feature.
# ------------------------------------------------------------

train_dates_series = (
    train_v2
    .iloc[
        train_indices
    ]["Date"]
)


train_target_values = (
    y_numeric.iloc[
        train_indices
    ].values
)


train_date_counts = (
    train_dates_series
    .map(
        date_statistics[
            "date_observation_count"
        ]
    )
    .values
)


train_date_anomaly_counts = (
    train_dates_series
    .map(
        date_statistics[
            "date_anomaly_count"
        ]
    )
    .values
)


# Leave-one-out anomaly rate
#
# If a row itself is anomalous:
# remove 1 anomaly from numerator.
#
# If normal:
# numerator stays unchanged.

loo_numerator = (
    train_date_anomaly_counts
    -
    train_target_values
)


loo_denominator = (
    train_date_counts
    - 1
)


train_date_te = (
    loo_numerator
    /
    np.maximum(
        loo_denominator,
        1
    )
)


# ------------------------------------------------------------
# Basic diagnostics
# ------------------------------------------------------------

print(
    "\nTraining date-TE range:"
)

print(
    f"Min: {np.nanmin(train_date_te):.6f}"
)

print(
    f"Max: {np.nanmax(train_date_te):.6f}"
)

print(
    "\nValidation date-TE range:"
)

print(
    f"Min: {valid_date_features.min():.6f}"
)

print(
    f"Max: {valid_date_features.max():.6f}"
)


# ------------------------------------------------------------
# Show strongest dates
# ------------------------------------------------------------

top_dates = (
    date_statistics
    .sort_values(
        "date_anomaly_rate",
        ascending=False
    )
    .head(15)
    .reset_index()
)


print(
    "\nTOP DATE ANOMALY RATES "
    "(calculated using training portion only)"
)

display(top_dates)


# ------------------------------------------------------------
# Store features
# ------------------------------------------------------------

train_v2["date_target_rate_loo"] = (
    global_anomaly_rate
)

train_v2.loc[
    train_indices,
    "date_target_rate_loo"
] = train_date_te


train_v2.loc[
    validation_indices,
    "date_target_rate_loo"
] = (
    valid_date_features
    .fillna(global_anomaly_rate)
    .values
)


print(
    "\n✓ Leakage-safe date target encoding created."
)


# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

del train_fold
del date_statistics
del train_date_counts
del train_date_anomaly_counts
del loo_numerator
del loo_denominator
del train_date_te
del valid_date_features

gc.collect()


print(
    "\n✓ Cells 1–7 completed."
)

LEAKAGE-SAFE DATE TARGET ENCODING

Global training anomaly rate: 0.855869%

Validation rows without date encoding: 0

Training date-TE range:
Min: 0.000000
Max: 0.127844

Validation date-TE range:
Min: 0.000000
Max: 0.127706

TOP DATE ANOMALY RATES (calculated using training portion only)


,Date,date_anomaly_count,date_observation_count,date_anomaly_rate
0,2022-11-23,118,924,0.127706
1,2021-01-21,100,924,0.108225
2,2021-01-13,95,914,0.103939
3,2023-01-18,92,901,0.102109
4,2023-12-24,88,911,0.096597
5,2021-12-28,84,933,0.090032
6,2020-12-23,76,922,0.082430
7,2021-02-18,76,928,0.081897
8,2021-03-02,74,926,0.079914
9,2023-01-16,73,916,0.079694



✓ Leakage-safe date target encoding created.

✓ Cells 1–7 completed.


In [8]:
# ============================================================
# V2 - CELL 8
# SENSOR COMBINATION KEY
# ============================================================

print("=" * 70)
print("BUILDING SENSOR COMBINATION KEYS")
print("=" * 70)


# Use strings rather than rounded values.
# We want exact combinations.

def make_sensor_key(df):
    
    return (
        df["X1"].astype(str)
        + "|"
        + df["X2"].astype(str)
        + "|"
        + df["X3"].astype(str)
        + "|"
        + df["X4"].astype(str)
        + "|"
        + df["X5"].astype(str)
    )


train_v2["_sensor_key"] = make_sensor_key(train_v2)

test_v2["_sensor_key"] = make_sensor_key(test_v2)


print("\nUnique sensor combinations:")

print(
    "Train:",
    train_v2["_sensor_key"].nunique()
)

print(
    "Test:",
    test_v2["_sensor_key"].nunique()
)


# ------------------------------------------------------------
# Coverage
# ------------------------------------------------------------

train_sensor_keys = set(
    train_v2.iloc[
        train_indices
    ]["_sensor_key"].unique()
)


validation_sensor_keys = (
    train_v2.iloc[
        validation_indices
    ]["_sensor_key"]
)


validation_seen = (
    validation_sensor_keys
    .isin(train_sensor_keys)
)


print("\nValidation rows with sensor combination")
print("already present in training:")

print(
    f"{validation_seen.sum():,}"
)

print(
    f"{validation_seen.mean() * 100:.4f}%"
)


print("\n✓ Sensor keys created.")

BUILDING SENSOR COMBINATION KEYS

Unique sensor combinations:
Train: 844829
Test: 281488

Validation rows with sensor combination
already present in training:
192,816
58.7002%

✓ Sensor keys created.


In [9]:
# ============================================================
# V2 - CELL 9
# SENSOR COMBINATION TARGET STATISTICS
# ============================================================

print("=" * 70)
print("SENSOR COMBINATION TARGET STATISTICS")
print("=" * 70)


# ------------------------------------------------------------
# Training portion only
# ------------------------------------------------------------

sensor_train = train_v2.iloc[
    train_indices
][
    ["_sensor_key", target_column]
].copy()


sensor_train["_target_numeric"] = (
    sensor_train[target_column]
    .astype(str)
    .astype(int)
)


# ------------------------------------------------------------
# Aggregate
# ------------------------------------------------------------

sensor_stats = (
    sensor_train
    .groupby("_sensor_key")[
        "_target_numeric"
    ]
    .agg(
        sensor_count="count",
        sensor_anomaly_count="sum"
    )
)


sensor_stats[
    "sensor_anomaly_rate"
] = (
    sensor_stats["sensor_anomaly_count"]
    /
    sensor_stats["sensor_count"]
)


# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

print(
    "\nUnique sensor groups in training:",
    len(sensor_stats)
)


print(
    "\nSensor groups with anomalies:"
)

print(
    (
        sensor_stats[
            "sensor_anomaly_count"
        ] > 0
    ).sum()
)


print(
    "\nSensor groups containing BOTH classes:"
)

# A group is mixed when its anomaly rate is neither 0 nor 1
mixed_groups = (
    (
        sensor_stats["sensor_anomaly_rate"] > 0
    )
    &
    (
        sensor_stats["sensor_anomaly_rate"] < 1
    )
).sum()

print(mixed_groups)


# ------------------------------------------------------------
# Most predictive groups
# ------------------------------------------------------------

print(
    "\nHighest anomaly-rate sensor groups:"
)

display(
    sensor_stats
    .sort_values(
        ["sensor_anomaly_rate", "sensor_count"],
        ascending=[False, False]
    )
    .head(20)
)


print(
    "\n✓ Training-only sensor statistics created."
)

SENSOR COMBINATION TARGET STATISTICS

Unique sensor groups in training: 714371

Sensor groups with anomalies:
11190

Sensor groups containing BOTH classes:
549

Highest anomaly-rate sensor groups:


,sensor_count,sensor_anomaly_count,sensor_anomaly_rate
_sensor_key,,,
1.0512710963760241|5.45677305224001|4311231547115195.0|1.0|2.8903717578961645,2,2,1.0
1.0768068054962199|5.465059864211477|1.0|5.052393630276104e+31|2.8903717578961645,2,2,1.0
1.0832870676749586|5.468102334337813|1.1719142372802612e+16|1.0|2.8903717578961645,2,2,1.0
1.0|5.445788645906245|1.0|2.515438670919167e+30|2.833213344056216,2,2,1.0
1.0|5.447039050294667|4311231547115195.0|1.0|2.833213344056216,2,2,1.0
1.0|5.447426789135878|4311231547115195.0|1.0|2.8903717578961645,2,2,1.0
1.0|5.45992588174445|1.1719142372802612e+16|1.0|2.833213344056216,2,2,1.0
1.0|5.460818795892685|4311231547115195.0|1.0|2.8903717578961645,2,2,1.0
1.0|5.462262778136282|1.1719142372802612e+16|2.718281828459045|2.8903717578961645,2,2,1.0



✓ Training-only sensor statistics created.


In [10]:
# ============================================================
# V2 - CELL 10
# SENSOR LOOKUP VALIDATION
# ============================================================

print("=" * 70)
print("EXPERIMENT 1 — SENSOR LOOKUP")
print("=" * 70)


# ------------------------------------------------------------
# Validation sensor probabilities
# ------------------------------------------------------------

valid_keys = (
    train_v2.iloc[
        validation_indices
    ]["_sensor_key"]
)


sensor_valid_probability = (
    valid_keys
    .map(
        sensor_stats[
            "sensor_anomaly_rate"
        ]
    )
)


# Unseen combinations receive global rate
sensor_valid_probability = (
    sensor_valid_probability
    .fillna(
        global_anomaly_rate
    )
    .astype(float)
)


y_valid_numeric = (
    y_numeric.iloc[
        validation_indices
    ].values
)


# ------------------------------------------------------------
# Threshold search
# ------------------------------------------------------------

threshold_results = []


for threshold in np.arange(
    0.05,
    0.96,
    0.01
):
    
    predictions = (
        sensor_valid_probability
        .values
        >= threshold
    ).astype(int)
    
    
    threshold_results.append({
        
        "threshold": threshold,
        
        "accuracy": accuracy_score(
            y_valid_numeric,
            predictions
        ),
        
        "precision": precision_score(
            y_valid_numeric,
            predictions,
            zero_division=0
        ),
        
        "recall": recall_score(
            y_valid_numeric,
            predictions,
            zero_division=0
        ),
        
        "f1": f1_score(
            y_valid_numeric,
            predictions,
            zero_division=0
        )
    })


sensor_lookup_results = (
    pd.DataFrame(
        threshold_results
    )
    .sort_values(
        "f1",
        ascending=False
    )
    .reset_index(drop=True)
)


best_sensor_lookup = (
    sensor_lookup_results.iloc[0]
)


print("\nBEST SENSOR LOOKUP RESULT")

display(
    sensor_lookup_results.head(10)
)


print(
    f"\nBest threshold: "
    f"{best_sensor_lookup['threshold']:.3f}"
)

print(
    f"F1: "
    f"{best_sensor_lookup['f1']:.6f}"
)

print(
    f"Precision: "
    f"{best_sensor_lookup['precision']:.6f}"
)

print(
    f"Recall: "
    f"{best_sensor_lookup['recall']:.6f}"
)

print(
    f"Accuracy: "
    f"{best_sensor_lookup['accuracy']:.6f}"
)

EXPERIMENT 1 — SENSOR LOOKUP

BEST SENSOR LOOKUP RESULT


,threshold,accuracy,precision,recall,f1
0,0.15,0.990861,0.070093,0.005323,0.009894
1,0.16,0.990861,0.070093,0.005323,0.009894
2,0.13,0.990800,0.064103,0.005323,0.009830
3,0.14,0.990800,0.064103,0.005323,0.009830
4,0.12,0.990730,0.058366,0.005323,0.009756
5,0.11,0.990715,0.057252,0.005323,0.009740
6,0.10,0.990696,0.055970,0.005323,0.009721
7,0.09,0.990642,0.052448,0.005323,0.009665
8,0.08,0.990632,0.051903,0.005323,0.009656
9,0.07,0.990468,0.043732,0.005323,0.009491



Best threshold: 0.150
F1: 0.009894
Precision: 0.070093
Recall: 0.005323
Accuracy: 0.990861


In [11]:
# ============================================================
# V2 - CELL 11
# SENSOR LOOKUP — SEEN VS UNSEEN ANALYSIS
# ============================================================

print("=" * 70)
print("SENSOR LOOKUP — SEEN VS UNSEEN")
print("=" * 70)


seen_mask = (
    valid_keys
    .isin(
        sensor_stats.index
    )
    .values
)


# ------------------------------------------------------------
# Evaluate separately
# ------------------------------------------------------------

def evaluate_subset(
    mask,
    name
):
    
    y_true = y_valid_numeric[mask]
    
    probabilities = (
        sensor_valid_probability
        .values[mask]
    )
    
    predictions = (
        probabilities
        >= best_sensor_lookup[
            "threshold"
        ]
    ).astype(int)
    
    
    return {
        
        "subset": name,
        
        "rows": len(y_true),
        
        "anomaly_rate_%":
            y_true.mean() * 100,
        
        "accuracy":
            accuracy_score(
                y_true,
                predictions
            ),
        
        "precision":
            precision_score(
                y_true,
                predictions,
                zero_division=0
            ),
        
        "recall":
            recall_score(
                y_true,
                predictions,
                zero_division=0
            ),
        
        "f1":
            f1_score(
                y_true,
                predictions,
                zero_division=0
            )
    }


coverage_results = pd.DataFrame([
    
    evaluate_subset(
        np.ones(
            len(y_valid_numeric),
            dtype=bool
        ),
        "All validation"
    ),
    
    evaluate_subset(
        seen_mask,
        "Seen sensor combination"
    ),
    
    evaluate_subset(
        ~seen_mask,
        "Unseen sensor combination"
    )
])


display(
    coverage_results
)


print(
    "\nValidation sensor coverage:"
)

print(
    f"Seen: "
    f"{seen_mask.mean() * 100:.4f}%"
)

print(
    f"Unseen: "
    f"{(~seen_mask).mean() * 100:.4f}%"
)

SENSOR LOOKUP — SEEN VS UNSEEN


,subset,rows,anomaly_rate_%,accuracy,precision,recall,f1
0,All validation,328476,0.857901,0.990861,0.070093,0.005323,0.009894
1,Seen sensor combination,192816,0.078832,0.998257,0.070093,0.098684,0.081967
2,Unseen sensor combination,135660,1.965207,0.980348,0.000000,0.000000,0.000000



Validation sensor coverage:
Seen: 58.7002%
Unseen: 41.2998%


In [12]:
# ============================================================
# V2 - CELL 12
# SENSOR + DATE COMBINATION LOOKUP
# ============================================================

print("=" * 70)
print("EXPERIMENT 2 — SENSOR + DATE LOOKUP")
print("=" * 70)


# ------------------------------------------------------------
# Create sensor + date key
# ------------------------------------------------------------

def make_sensor_date_key(df):
    
    return (
        df["Date"].astype(str)
        + "|"
        + df["_sensor_key"]
    )


train_v2["_sensor_date_key"] = (
    make_sensor_date_key(
        train_v2
    )
)

test_v2["_sensor_date_key"] = (
    make_sensor_date_key(
        test_v2
    )
)


# ------------------------------------------------------------
# Training-only statistics
# ------------------------------------------------------------

sensor_date_train = train_v2.iloc[
    train_indices
][
    [
        "_sensor_date_key",
        target_column
    ]
].copy()


sensor_date_train[
    "_target_numeric"
] = (
    sensor_date_train[
        target_column
    ]
    .astype(str)
    .astype(int)
)


sensor_date_stats = (
    sensor_date_train
    .groupby("_sensor_date_key")[
        "_target_numeric"
    ]
    .agg(
        count="count",
        anomaly_count="sum"
    )
)


sensor_date_stats[
    "anomaly_rate"
] = (
    sensor_date_stats[
        "anomaly_count"
    ]
    /
    sensor_date_stats[
        "count"
    ]
)


# ------------------------------------------------------------
# Validation lookup
# ------------------------------------------------------------

valid_sensor_date_keys = (
    train_v2.iloc[
        validation_indices
    ]["_sensor_date_key"]
)


sensor_date_probability = (
    valid_sensor_date_keys
    .map(
        sensor_date_stats[
            "anomaly_rate"
        ]
    )
    .fillna(
        global_anomaly_rate
    )
    .astype(float)
)


sensor_date_seen = (
    valid_sensor_date_keys
    .isin(
        sensor_date_stats.index
    )
    .values
)


print(
    "\nUnique sensor+date groups in training:",
    len(sensor_date_stats)
)


print(
    "Validation rows with known sensor+date:",
    f"{sensor_date_seen.sum():,}"
)


print(
    "Validation coverage:",
    f"{sensor_date_seen.mean() * 100:.4f}%"
)


# ------------------------------------------------------------
# Threshold optimization
# ------------------------------------------------------------

sensor_date_results = []


for threshold in np.arange(
    0.05,
    0.96,
    0.01
):
    
    pred = (
        sensor_date_probability.values
        >= threshold
    ).astype(int)
    
    
    sensor_date_results.append({
        
        "threshold": threshold,
        
        "accuracy": accuracy_score(
            y_valid_numeric,
            pred
        ),
        
        "precision": precision_score(
            y_valid_numeric,
            pred,
            zero_division=0
        ),
        
        "recall": recall_score(
            y_valid_numeric,
            pred,
            zero_division=0
        ),
        
        "f1": f1_score(
            y_valid_numeric,
            pred,
            zero_division=0
        )
    })


sensor_date_results = (
    pd.DataFrame(
        sensor_date_results
    )
    .sort_values(
        "f1",
        ascending=False
    )
    .reset_index(drop=True)
)


best_sensor_date = (
    sensor_date_results.iloc[0]
)


print("\nBEST SENSOR + DATE LOOKUP")

display(
    sensor_date_results.head(10)
)


print(
    f"\nBest threshold: "
    f"{best_sensor_date['threshold']:.3f}"
)

print(
    f"F1: "
    f"{best_sensor_date['f1']:.6f}"
)

print(
    f"Precision: "
    f"{best_sensor_date['precision']:.6f}"
)

print(
    f"Recall: "
    f"{best_sensor_date['recall']:.6f}"
)

EXPERIMENT 2 — SENSOR + DATE LOOKUP

Unique sensor+date groups in training: 1273705
Validation rows with known sensor+date: 18,519
Validation coverage: 5.6379%

BEST SENSOR + DATE LOOKUP


,threshold,accuracy,precision,recall,f1
0,0.05,0.991424,0.666667,0.00071,0.001418
1,0.06,0.991424,0.666667,0.00071,0.001418
2,0.07,0.991424,0.666667,0.00071,0.001418
3,0.08,0.991424,0.666667,0.00071,0.001418
4,0.09,0.991424,0.666667,0.00071,0.001418
5,0.10,0.991424,0.666667,0.00071,0.001418
6,0.11,0.991424,0.666667,0.00071,0.001418
7,0.12,0.991424,0.666667,0.00071,0.001418
8,0.13,0.991424,0.666667,0.00071,0.001418
9,0.14,0.991424,0.666667,0.00071,0.001418



Best threshold: 0.050
F1: 0.001418
Precision: 0.666667
Recall: 0.000710


In [13]:
# ============================================================
# V2 - CELL 13
# COMBINATION DETERMINISM ANALYSIS
# ============================================================

print("=" * 70)
print("COMBINATION DETERMINISM ANALYSIS")
print("=" * 70)


# ------------------------------------------------------------
# Sensor combination determinism
# ------------------------------------------------------------

sensor_group_purity = (
    sensor_stats
    .assign(
        purity=lambda x: np.maximum(
            x["sensor_anomaly_rate"],
            1 - x["sensor_anomaly_rate"]
        )
    )
)


# ------------------------------------------------------------
# Sensor + date determinism
# ------------------------------------------------------------

sensor_date_group_purity = (
    sensor_date_stats
    .assign(
        purity=lambda x: np.maximum(
            x["anomaly_rate"],
            1 - x["anomaly_rate"]
        )
    )
)


print(
    "\nSENSOR COMBINATIONS"
)

print(
    "Total groups:",
    len(sensor_group_purity)
)

print(
    "Perfectly pure groups:",
    (
        sensor_group_purity["purity"] == 1
    ).sum()
)

print(
    "Purity >= 99%:",
    (
        sensor_group_purity["purity"] >= 0.99
    ).sum()
)


print(
    "\nSENSOR + DATE COMBINATIONS"
)

print(
    "Total groups:",
    len(sensor_date_group_purity)
)

print(
    "Perfectly pure groups:",
    (
        sensor_date_group_purity["purity"] == 1
    ).sum()
)

print(
    "Purity >= 99%:",
    (
        sensor_date_group_purity["purity"] >= 0.99
    ).sum()
)


# ------------------------------------------------------------
# Distribution of group sizes
# ------------------------------------------------------------

print(
    "\nSensor group size distribution:"
)

display(
    sensor_stats[
        "sensor_count"
    ]
    .describe()
)


print(
    "\nSensor+date group size distribution:"
)

display(
    sensor_date_stats[
        "count"
    ]
    .describe()
)


print(
    "\n✓ Determinism analysis complete."
)

COMBINATION DETERMINISM ANALYSIS

SENSOR COMBINATIONS
Total groups: 714371
Perfectly pure groups: 713822
Purity >= 99%: 713825

SENSOR + DATE COMBINATIONS
Total groups: 1273705
Perfectly pure groups: 1273695
Purity >= 99%: 1273695

Sensor group size distribution:


count    714371.000000
mean          1.835108
std           5.986197
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         251.000000
Name: sensor_count, dtype: float64


Sensor+date group size distribution:


count    1.273705e+06
mean     1.029240e+00
std      2.879617e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      2.510000e+02
Name: count, dtype: float64


✓ Determinism analysis complete.


In [14]:
# ============================================================
# V2 - CELL 14
# EXPERIMENT DECISION SUMMARY
# ============================================================

print("=" * 70)
print("V2 EXPERIMENT DECISION SUMMARY")
print("=" * 70)


experiment_summary = pd.DataFrame({

    "experiment": [
        "V1 XGBoost OOF",
        "V1 XGB + LGB OOF",
        "V2 Sensor Lookup",
        "V2 Sensor + Date Lookup"
    ],
    
    "f1": [
        0.799669,
        0.800956,
        best_sensor_lookup["f1"],
        best_sensor_date["f1"]
    ],
    
    "threshold": [
        0.345,
        0.330,
        best_sensor_lookup["threshold"],
        best_sensor_date["threshold"]
    ]
})


experiment_summary[
    "improvement_vs_V1_ensemble"
] = (
    experiment_summary["f1"]
    - 0.800956
)


display(
    experiment_summary
)


# ------------------------------------------------------------
# Best new experiment
# ------------------------------------------------------------

new_experiments = (
    experiment_summary.iloc[2:]
)


best_new_experiment = (
    new_experiments
    .sort_values(
        "f1",
        ascending=False
    )
    .iloc[0]
)


print(
    "\nBEST NEW EXPERIMENT:"
)

print(
    best_new_experiment[
        "experiment"
    ]
)


print(
    f"F1: "
    f"{best_new_experiment['f1']:.6f}"
)


print(
    f"Improvement vs V1 ensemble: "
    f"{best_new_experiment['improvement_vs_V1_ensemble']:+.6f}"
)


print("\n" + "=" * 70)
print("STOPPING BEFORE EXPENSIVE MODEL TRAINING")
print("=" * 70)

print(
    "\nWe will only proceed with the new signal "
    "if validation evidence supports it."
)

V2 EXPERIMENT DECISION SUMMARY


,experiment,f1,threshold,improvement_vs_V1_ensemble
0,V1 XGBoost OOF,0.799669,0.345,-0.001287
1,V1 XGB + LGB OOF,0.800956,0.330,0.000000
2,V2 Sensor Lookup,0.009894,0.150,-0.791062
3,V2 Sensor + Date Lookup,0.001418,0.050,-0.799538



BEST NEW EXPERIMENT:
V2 Sensor Lookup
F1: 0.009894
Improvement vs V1 ensemble: -0.791062

STOPPING BEFORE EXPENSIVE MODEL TRAINING

We will only proceed with the new signal if validation evidence supports it.


In [15]:
# ============================================================
# V2 - CELL 15
# DATE TARGET ENCODING
# ============================================================

print("=" * 70)
print("V2 — DATE TARGET ENCODING")
print("=" * 70)


# ------------------------------------------------------------
# Training-fold data ONLY
# ------------------------------------------------------------

te_train = train_v2.iloc[
    train_indices
].copy()

te_valid = train_v2.iloc[
    validation_indices
].copy()


te_train["_target_num"] = (
    te_train[target_column]
    .astype(str)
    .astype(int)
)


# ------------------------------------------------------------
# Global anomaly rate
# ------------------------------------------------------------

global_rate = (
    te_train["_target_num"]
    .mean()
)


print(
    f"\nTraining-fold global anomaly rate: "
    f"{global_rate:.6f}"
)


# ------------------------------------------------------------
# Date statistics
# ------------------------------------------------------------

date_stats = (
    te_train
    .groupby("Date")["_target_num"]
    .agg(
        date_count="count",
        date_anomalies="sum"
    )
)


date_stats["date_rate"] = (
    date_stats["date_anomalies"]
    /
    date_stats["date_count"]
)


# ------------------------------------------------------------
# Validation mapping
# ------------------------------------------------------------

valid_date_rate = (
    te_valid["Date"]
    .map(
        date_stats["date_rate"]
    )
    .fillna(global_rate)
)


# ------------------------------------------------------------
# Evaluate date-rate signal
# ------------------------------------------------------------

date_threshold_results = []


for threshold in np.arange(
    0.01,
    0.50,
    0.005
):

    pred = (
        valid_date_rate.values
        >= threshold
    ).astype(int)

    date_threshold_results.append({

        "threshold": threshold,

        "precision": precision_score(
            y_valid_numeric,
            pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_valid_numeric,
            pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_valid_numeric,
            pred,
            zero_division=0
        )
    })


date_results = (
    pd.DataFrame(
        date_threshold_results
    )
    .sort_values(
        "f1",
        ascending=False
    )
    .reset_index(drop=True)
)


best_date_te = date_results.iloc[0]


print("\nDATE TARGET ENCODING RESULT")

display(
    date_results.head(10)
)


print(
    f"\nBest threshold: "
    f"{best_date_te['threshold']:.3f}"
)

print(
    f"F1: "
    f"{best_date_te['f1']:.6f}"
)

print(
    f"Precision: "
    f"{best_date_te['precision']:.6f}"
)

print(
    f"Recall: "
    f"{best_date_te['recall']:.6f}"
)


print(
    "\nUnique dates:",
    date_stats.shape[0]
)

print(
    "Validation dates covered:",
    te_valid["Date"]
    .isin(date_stats.index)
    .mean() * 100,
    "%"
)

V2 — DATE TARGET ENCODING

Training-fold global anomaly rate: 0.008559

DATE TARGET ENCODING RESULT


,threshold,precision,recall,f1
0,0.055,0.074790,0.208659,0.110112
1,0.060,0.079352,0.175656,0.109320
2,0.065,0.081890,0.161107,0.108586
3,0.050,0.067525,0.260468,0.107247
4,0.045,0.062387,0.306955,0.103698
5,0.040,0.058937,0.347764,0.100792
6,0.075,0.092627,0.106104,0.098908
7,0.070,0.092627,0.106104,0.098908
8,0.035,0.053641,0.427963,0.095332
9,0.030,0.050341,0.479773,0.091120



Best threshold: 0.055
F1: 0.110112
Precision: 0.074790
Recall: 0.208659

Unique dates: 1432
Validation dates covered: 100.0 %


In [16]:
# ============================================================
# V2 - CELL 16
# INDIVIDUAL SENSOR TARGET ENCODING
# ============================================================

print("=" * 70)
print("V2 — INDIVIDUAL SENSOR TARGET ENCODING")
print("=" * 70)


sensor_te_results = []


sensor_columns = [
    "X1",
    "X2",
    "X3",
    "X4",
    "X5"
]


# Smoothing strength
SMOOTHING = 20


for feature in sensor_columns:

    # --------------------------------------------------------
    # Training statistics
    # --------------------------------------------------------

    stats = (
        te_train
        .groupby(feature)["_target_num"]
        .agg(
            count="count",
            anomalies="sum"
        )
    )


    # --------------------------------------------------------
    # Smoothed probability
    # --------------------------------------------------------

    stats["smoothed_rate"] = (
        stats["anomalies"]
        +
        SMOOTHING * global_rate
    ) / (
        stats["count"]
        +
        SMOOTHING
    )


    # --------------------------------------------------------
    # Validation probabilities
    # --------------------------------------------------------

    valid_probability = (
        te_valid[feature]
        .map(
            stats["smoothed_rate"]
        )
        .fillna(global_rate)
    )


    # --------------------------------------------------------
    # Threshold search
    # --------------------------------------------------------

    best = None


    for threshold in np.arange(
        0.01,
        0.50,
        0.005
    ):

        pred = (
            valid_probability.values
            >= threshold
        ).astype(int)


        f1 = f1_score(
            y_valid_numeric,
            pred,
            zero_division=0
        )


        if (
            best is None
            or f1 > best["f1"]
        ):

            best = {

                "feature": feature,

                "threshold": threshold,

                "precision":
                    precision_score(
                        y_valid_numeric,
                        pred,
                        zero_division=0
                    ),

                "recall":
                    recall_score(
                        y_valid_numeric,
                        pred,
                        zero_division=0
                    ),

                "f1": f1,

                "unique_values":
                    len(stats)
            }


    sensor_te_results.append(best)


sensor_te_results = (
    pd.DataFrame(
        sensor_te_results
    )
    .sort_values(
        "f1",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nINDIVIDUAL SENSOR TARGET-ENCODING RESULTS")

display(
    sensor_te_results
)

V2 — INDIVIDUAL SENSOR TARGET ENCODING

INDIVIDUAL SENSOR TARGET-ENCODING RESULTS


,feature,threshold,precision,recall,f1,unique_values
0,X4,0.160,0.385325,0.326118,0.353258,81
1,X3,0.125,0.268488,0.380057,0.314676,88
2,X2,0.060,0.086055,0.294535,0.133194,2789
3,X5,0.090,0.139913,0.125266,0.132185,32
4,X1,0.030,0.055403,0.207949,0.087495,520


In [17]:
# ============================================================
# V2 - CELL 17
# COMBINED TARGET-ENCODING SIGNAL
# ============================================================

print("=" * 70)
print("V2 — COMBINED TARGET ENCODING")
print("=" * 70)


te_features_valid = pd.DataFrame(
    index=te_valid.index
)


# ------------------------------------------------------------
# Date encoding
# ------------------------------------------------------------

te_features_valid["date_te"] = (
    te_valid["Date"]
    .map(
        date_stats["date_rate"]
    )
    .fillna(global_rate)
)


# ------------------------------------------------------------
# Sensor encodings
# ------------------------------------------------------------

for feature in sensor_columns:

    stats = (
        te_train
        .groupby(feature)["_target_num"]
        .agg(
            count="count",
            anomalies="sum"
        )
    )

    stats["smoothed_rate"] = (
        stats["anomalies"]
        +
        SMOOTHING * global_rate
    ) / (
        stats["count"]
        +
        SMOOTHING
    )

    te_features_valid[
        f"{feature}_te"
    ] = (
        te_valid[feature]
        .map(
            stats["smoothed_rate"]
        )
        .fillna(global_rate)
    )


# ------------------------------------------------------------
# Inspect
# ------------------------------------------------------------

print("\nTarget-encoding features:")

display(
    te_features_valid.head()
)


print("\nCorrelation with target:")

te_corr = (
    te_features_valid
    .assign(
        target=y_valid_numeric
    )
    .corr(
        method="spearman"
    )["target"]
    .drop("target")
    .sort_values(
        ascending=False
    )
)


display(te_corr)


# ------------------------------------------------------------
# Max TE signal
# ------------------------------------------------------------

max_te_probability = (
    te_features_valid
    .drop(
        columns=["date_te"]
    )
    .max(axis=1)
)


best_combined_te = None


for threshold in np.arange(
    0.01,
    0.50,
    0.005
):

    pred = (
        max_te_probability.values
        >= threshold
    ).astype(int)


    f1 = f1_score(
        y_valid_numeric,
        pred,
        zero_division=0
    )


    if (
        best_combined_te is None
        or f1 > best_combined_te["f1"]
    ):

        best_combined_te = {

            "threshold": threshold,

            "precision":
                precision_score(
                    y_valid_numeric,
                    pred,
                    zero_division=0
                ),

            "recall":
                recall_score(
                    y_valid_numeric,
                    pred,
                    zero_division=0
                ),

            "f1": f1
        }


print(
    "\nMAX SENSOR-TE SIGNAL"
)

print(
    f"F1: {best_combined_te['f1']:.6f}"
)

print(
    f"Precision: "
    f"{best_combined_te['precision']:.6f}"
)

print(
    f"Recall: "
    f"{best_combined_te['recall']:.6f}"
)

V2 — COMBINED TARGET ENCODING

Target-encoding features:


,date_te,X1_te,X2_te,X3_te,X4_te,X5_te
0,0.037344,0.033537,0.051105,0.003174,0.004009,0.051275
5,0.037344,0.038763,0.049111,0.003174,0.004009,0.051275
10,0.037344,0.045575,0.067653,0.003174,0.004009,0.051275
15,0.037344,0.004001,0.020649,0.003174,0.004033,0.011503
20,0.037344,0.004001,0.041052,0.003174,0.004033,0.051275



Correlation with target:


X3_te      0.206420
X2_te      0.123285
date_te    0.121506
X5_te      0.116945
X4_te      0.103593
X1_te      0.068607
Name: target, dtype: float64


MAX SENSOR-TE SIGNAL
F1: 0.419076
Precision: 0.516918
Recall: 0.352378


In [18]:
# ============================================================
# V2 - CELL 18
# EXPERIMENT 2 DECISION
# ============================================================

print("=" * 70)
print("V2 — TARGET ENCODING EXPERIMENT SUMMARY")
print("=" * 70)


summary_v2 = pd.DataFrame({

    "experiment": [

        "V1 XGB + LGB Ensemble",

        "V2 Date Target Encoding",

        "V2 X1 Target Encoding",

        "V2 X2 Target Encoding",

        "V2 X3 Target Encoding",

        "V2 X4 Target Encoding",

        "V2 X5 Target Encoding",

        "V2 Max Sensor TE"
    ],

    "f1": [

        0.800956,

        best_date_te["f1"],

        sensor_te_results.loc[
            sensor_te_results["feature"] == "X1",
            "f1"
        ].iloc[0],

        sensor_te_results.loc[
            sensor_te_results["feature"] == "X2",
            "f1"
        ].iloc[0],

        sensor_te_results.loc[
            sensor_te_results["feature"] == "X3",
            "f1"
        ].iloc[0],

        sensor_te_results.loc[
            sensor_te_results["feature"] == "X4",
            "f1"
        ].iloc[0],

        sensor_te_results.loc[
            sensor_te_results["feature"] == "X5",
            "f1"
        ].iloc[0],

        best_combined_te["f1"]
    ]
})


summary_v2["improvement_vs_V1"] = (
    summary_v2["f1"]
    - 0.800956
)


display(
    summary_v2.sort_values(
        "f1",
        ascending=False
    )
)


print("\n" + "=" * 70)
print("NEXT DECISION")
print("=" * 70)

best_v2 = (
    summary_v2
    .sort_values(
        "f1",
        ascending=False
    )
    .iloc[0]
)

print(
    f"\nBest experiment: "
    f"{best_v2['experiment']}"
)

print(
    f"F1: {best_v2['f1']:.6f}"
)

print(
    f"Improvement: "
    f"{best_v2['improvement_vs_V1']:+.6f}"
)

V2 — TARGET ENCODING EXPERIMENT SUMMARY


,experiment,f1,improvement_vs_V1
0,V1 XGB + LGB Ensemble,0.800956,0.000000
7,V2 Max Sensor TE,0.419076,-0.381880
5,V2 X4 Target Encoding,0.353258,-0.447698
4,V2 X3 Target Encoding,0.314676,-0.486280
3,V2 X2 Target Encoding,0.133194,-0.667762
6,V2 X5 Target Encoding,0.132185,-0.668771
1,V2 Date Target Encoding,0.110112,-0.690844
2,V2 X1 Target Encoding,0.087495,-0.713461



NEXT DECISION

Best experiment: V1 XGB + LGB Ensemble
F1: 0.800956
Improvement: +0.000000


In [19]:
# ============================================================
# V3 - CELL 19
# TARGET-FREE SENSOR RARITY FEATURES
# ============================================================

print("=" * 70)
print("V3 — TARGET-FREE SENSOR RARITY FEATURES")
print("=" * 70)

sensor_columns = ["X1", "X2", "X3", "X4", "X5"]

rarity_train = train_v2.iloc[train_indices].copy()
rarity_valid = train_v2.iloc[validation_indices].copy()

rarity_features_valid = pd.DataFrame(
    index=rarity_valid.index
)

rarity_features_train = pd.DataFrame(
    index=rarity_train.index
)


# ------------------------------------------------------------
# Frequency / rarity of individual sensor values
# ------------------------------------------------------------

for col in sensor_columns:

    frequencies = (
        rarity_train[col]
        .value_counts(normalize=True)
    )

    train_freq = (
        rarity_train[col]
        .map(frequencies)
        .fillna(0)
    )

    valid_freq = (
        rarity_valid[col]
        .map(frequencies)
        .fillna(0)
    )

    rarity_features_train[
        f"{col}_frequency"
    ] = train_freq

    rarity_features_valid[
        f"{col}_frequency"
    ] = valid_freq

    # Negative log frequency.
    # Larger = rarer.
    rarity_features_train[
        f"{col}_rarity"
    ] = -np.log(
        train_freq.clip(lower=1e-8)
    )

    rarity_features_valid[
        f"{col}_rarity"
    ] = -np.log(
        valid_freq.clip(lower=1e-8)
    )


# ------------------------------------------------------------
# Overall rarity
# ------------------------------------------------------------

rarity_features_valid[
    "overall_rarity"
] = rarity_features_valid[
    [
        f"{c}_rarity"
        for c in sensor_columns
    ]
].mean(axis=1)


rarity_features_valid[
    "max_sensor_rarity"
] = rarity_features_valid[
    [
        f"{c}_rarity"
        for c in sensor_columns
    ]
].max(axis=1)


print("\nValidation rarity features:")

display(
    rarity_features_valid.head()
)


print("\nRarity feature statistics:")

display(
    rarity_features_valid[
        [
            "overall_rarity",
            "max_sensor_rarity"
        ]
    ].describe()
)

print("\n✓ Target-free rarity features created.")

V3 — TARGET-FREE SENSOR RARITY FEATURES

Validation rarity features:


,X1_frequency,X1_rarity,X2_frequency,X2_rarity,X3_frequency,X3_rarity,X4_frequency,X4_rarity,X5_frequency,X5_rarity,overall_rarity,max_sensor_rarity
0,0.000375,7.887782,0.000241,8.330519,0.917474,0.086131,0.184225,1.691599,0.076336,2.572616,4.113729,8.330519
5,0.000126,8.980316,0.000267,8.228328,0.917474,0.086131,0.184225,1.691599,0.076336,2.572616,4.311798,8.980316
10,0.000205,8.491550,0.000066,9.631914,0.917474,0.086131,0.184225,1.691599,0.076336,2.572616,4.494762,9.631914
15,0.235084,1.447812,0.000397,7.830511,0.917474,0.086131,0.701382,0.354703,0.172347,1.758245,2.295480,7.830511
20,0.235084,1.447812,0.000341,7.983703,0.917474,0.086131,0.701382,0.354703,0.076336,2.572616,2.488993,7.983703



Rarity feature statistics:


,overall_rarity,max_sensor_rarity
count,328476.000000,328476.000000
mean,2.932949,7.236628
std,0.715191,0.870947
min,1.829318,6.380999
25%,2.544413,6.621751
50%,2.843474,6.933992
75%,3.225797,7.576003
max,9.228414,18.420681



✓ Target-free rarity features created.


In [20]:
# ============================================================
# V3 - CELL 20
# RARITY VS ANOMALIES
# ============================================================

print("=" * 70)
print("V3 — RARITY SIGNAL EVALUATION")
print("=" * 70)


y_v3 = y_valid_numeric


rarity_signal = (
    rarity_features_valid[
        "overall_rarity"
    ].values
)


# ------------------------------------------------------------
# Rank correlation
# ------------------------------------------------------------

from scipy.stats import spearmanr

rho, p_value = spearmanr(
    rarity_signal,
    y_v3
)

print(
    f"\nSpearman correlation "
    f"(rarity vs target): {rho:.6f}"
)

print(
    f"p-value: {p_value:.6e}"
)


# ------------------------------------------------------------
# Threshold search
# ------------------------------------------------------------

rarity_results = []


for threshold in np.quantile(
    rarity_signal,
    np.linspace(
        0.50,
        0.999,
        100
    )
):

    pred = (
        rarity_signal >= threshold
    ).astype(int)

    rarity_results.append({

        "threshold": threshold,

        "precision":
            precision_score(
                y_v3,
                pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_v3,
                pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_v3,
                pred,
                zero_division=0
            )
    })


rarity_results = (
    pd.DataFrame(
        rarity_results
    )
    .sort_values(
        "f1",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nBEST RARITY-ONLY PERFORMANCE")

display(
    rarity_results.head(10)
)

V3 — RARITY SIGNAL EVALUATION

Spearman correlation (rarity vs target): 0.142674
p-value: 0.000000e+00

BEST RARITY-ONLY PERFORMANCE


,threshold,precision,recall,f1
0,5.228050,0.327198,0.422640,0.368845
1,5.009232,0.278701,0.523776,0.363816
2,4.837343,0.240506,0.593329,0.342272
3,5.573384,0.411083,0.289567,0.339788
4,4.701642,0.212966,0.650461,0.320875
5,4.595544,0.189905,0.691625,0.297989
6,4.507169,0.169561,0.717175,0.274276
7,4.430586,0.155297,0.748048,0.257199
8,4.359380,0.142022,0.767566,0.239694
9,4.294209,0.131715,0.789212,0.225752


In [21]:
# ============================================================
# V3 - CELL 21
# ROBUST SENSOR DEVIATION
# ============================================================

print("=" * 70)
print("V3 — ROBUST SENSOR DEVIATION FEATURES")
print("=" * 70)


robust_stats = {}


for col in sensor_columns:

    median = (
        rarity_train[col]
        .median()
    )

    mad = (
        np.median(
            np.abs(
                rarity_train[col]
                - median
            )
        )
    )

    # Protect against zero MAD
    mad = max(
        mad,
        1e-12
    )

    robust_stats[col] = {
        "median": median,
        "mad": mad
    }


    rarity_features_valid[
        f"{col}_robust_z"
    ] = (
        np.abs(
            rarity_valid[col]
            - median
        )
        /
        (1.4826 * mad)
    )


# ------------------------------------------------------------
# Aggregate robust abnormality
# ------------------------------------------------------------

robust_z_columns = [
    f"{c}_robust_z"
    for c in sensor_columns
]


rarity_features_valid[
    "robust_mean_z"
] = rarity_features_valid[
    robust_z_columns
].mean(axis=1)


rarity_features_valid[
    "robust_max_z"
] = rarity_features_valid[
    robust_z_columns
].max(axis=1)


print("\nRobust deviation statistics:")

display(
    rarity_features_valid[
        [
            "robust_mean_z",
            "robust_max_z"
        ]
    ].describe()
)


print("\n✓ Robust sensor-distance features created.")

V3 — ROBUST SENSOR DEVIATION FEATURES

Robust deviation statistics:


,robust_mean_z,robust_max_z
count,3.284760e+05,3.284760e+05
mean,2.639678e+43,1.319839e+44
std,1.431902e+46,7.159511e+46
min,6.945544e-04,3.472772e-03
25%,3.589761e-01,8.867639e-01
50%,6.497527e-01,2.190697e+00
75%,2.317930e+11,1.158965e+12
max,8.196452e+48,4.098226e+49



✓ Robust sensor-distance features created.


In [22]:
# ============================================================
# V3 - CELL 22
# ROBUST DEVIATION EVALUATION
# ============================================================

print("=" * 70)
print("V3 — ROBUST DEVIATION SIGNAL")
print("=" * 70)


robust_signal = (
    rarity_features_valid[
        "robust_max_z"
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
    .values
)


rho, p_value = spearmanr(
    robust_signal,
    y_v3
)


print(
    f"\nSpearman correlation: "
    f"{rho:.6f}"
)

print(
    f"p-value: "
    f"{p_value:.6e}"
)


robust_results = []


for threshold in np.quantile(
    robust_signal,
    np.linspace(
        0.50,
        0.999,
        100
    )
):

    pred = (
        robust_signal >= threshold
    ).astype(int)

    robust_results.append({

        "threshold": threshold,

        "precision":
            precision_score(
                y_v3,
                pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_v3,
                pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_v3,
                pred,
                zero_division=0
            )
    })


robust_results = (
    pd.DataFrame(
        robust_results
    )
    .sort_values(
        "f1",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nBEST ROBUST-DEVIATION PERFORMANCE")

display(
    robust_results.head(10)
)

V3 — ROBUST DEVIATION SIGNAL

Spearman correlation: 0.144175
p-value: 0.000000e+00

BEST ROBUST-DEVIATION PERFORMANCE


,threshold,precision,recall,f1
0,5.840645e+28,0.315286,0.338893,0.326663
1,1.069751e+27,0.194879,0.834634,0.315980
2,2.907886e+27,0.195744,0.793116,0.313993
3,2.651649e+24,0.179704,0.882896,0.298626
4,7.904453e+27,0.191088,0.665011,0.296871
5,7.904453e+27,0.191088,0.665011,0.296871
6,2.148653e+28,0.207717,0.473740,0.288805
7,2.148653e+28,0.207717,0.473740,0.288805
8,1.786667e+22,0.165391,0.897090,0.279291
9,4.428704e+19,0.149744,0.903833,0.256922


In [23]:
# ============================================================
# V3 - CELL 23
# RARITY SEPARATION BY CLASS
# ============================================================

print("=" * 70)
print("V3 — RARITY SEPARATION BY CLASS")
print("=" * 70)


class_signal = pd.DataFrame({

    "target": y_v3,

    "overall_rarity":
        rarity_features_valid[
            "overall_rarity"
        ].values,

    "max_sensor_rarity":
        rarity_features_valid[
            "max_sensor_rarity"
        ].values,

    "robust_mean_z":
        rarity_features_valid[
            "robust_mean_z"
        ].values,

    "robust_max_z":
        rarity_features_valid[
            "robust_max_z"
        ].values
})


print("\nMean signal by class:")

display(
    class_signal
    .groupby("target")
    .mean()
)


print("\nMedian signal by class:")

display(
    class_signal
    .groupby("target")
    .median()
)


print("\n95th percentile by class:")

display(
    class_signal
    .groupby("target")
    .quantile(0.95)
)

V3 — RARITY SEPARATION BY CLASS

Mean signal by class:


,overall_rarity,max_sensor_rarity,robust_mean_z,robust_max_z
target,,,,
0,2.914659,7.221780,3.338703e+39,1.669352e+40
1,5.046718,8.952549,3.076516e+45,1.538258e+46



Median signal by class:


,overall_rarity,max_sensor_rarity,robust_mean_z,robust_max_z
target,,,,
0,2.837779,6.929305,6.447160e-01,2.190697e+00
1,5.062114,8.670161,1.794841e+27,7.904453e+27



95th percentile by class:


,overall_rarity,max_sensor_rarity,robust_mean_z,robust_max_z
target,,,,
0,4.235488,8.898875,5.451869e+13,2.714345e+14
1,6.849493,11.521312,1.852668e+43,9.263341e+43


In [24]:
# ============================================================
# V3 - CELL 24
# DATE-RELATIVE SENSOR DEVIATION
# ============================================================

print("=" * 70)
print("V3 — DATE-RELATIVE SENSOR DEVIATION")
print("=" * 70)


date_train = rarity_train.copy()
date_valid = rarity_valid.copy()


date_relative_features = pd.DataFrame(
    index=date_valid.index
)


for col in sensor_columns:

    # --------------------------------------------------------
    # Per-date robust statistics from TRAINING ONLY
    # --------------------------------------------------------

    grouped = (
        date_train
        .groupby("Date")[col]
        .agg(
            date_median="median"
        )
    )


    # --------------------------------------------------------
    # Per-date MAD
    # --------------------------------------------------------

    date_mad = (
        date_train
        .groupby("Date")[col]
        .apply(
            lambda x:
            np.median(
                np.abs(
                    x - np.median(x)
                )
            )
        )
    )


    date_stats_local = grouped.join(
        date_mad.rename("date_mad")
    )


    date_stats_local[
        "date_mad"
    ] = date_stats_local[
        "date_mad"
    ].clip(
        lower=1e-12
    )


    # --------------------------------------------------------
    # Map to validation
    # --------------------------------------------------------

    temp = date_valid[
        ["Date", col]
    ].copy()


    temp = temp.merge(
        date_stats_local,
        left_on="Date",
        right_index=True,
        how="left"
    )


    date_relative_features[
        f"{col}_date_z"
    ] = (
        np.abs(
            temp[col]
            -
            temp["date_median"]
        )
        /
        (
            1.4826
            *
            temp["date_mad"]
        )
    )


# ------------------------------------------------------------
# Aggregate
# ------------------------------------------------------------

date_z_cols = [
    f"{c}_date_z"
    for c in sensor_columns
]


date_relative_features[
    "date_relative_mean_z"
] = date_relative_features[
    date_z_cols
].mean(axis=1)


date_relative_features[
    "date_relative_max_z"
] = date_relative_features[
    date_z_cols
].max(axis=1)


print("\nDate-relative feature statistics:")

display(
    date_relative_features[
        [
            "date_relative_mean_z",
            "date_relative_max_z"
        ]
    ].describe()
)


print(
    "\n✓ Date-relative features created."
)

V3 — DATE-RELATIVE SENSOR DEVIATION

Date-relative feature statistics:


,date_relative_mean_z,date_relative_max_z
count,3.284760e+05,3.284760e+05
mean,2.639540e+43,1.319770e+44
std,1.431902e+46,7.159511e+46
min,0.000000e+00,0.000000e+00
25%,3.035647e-01,9.355110e-01
50%,8.880916e-01,2.497135e+00
75%,3.252973e+11,1.262513e+12
max,8.196452e+48,4.098226e+49



✓ Date-relative features created.


In [25]:
# ============================================================
# V3 - CELL 25
# DATE-RELATIVE SIGNAL EVALUATION
# ============================================================

print("=" * 70)
print("V3 — DATE-RELATIVE SENSOR SIGNAL")
print("=" * 70)


date_signal = (
    date_relative_features[
        "date_relative_max_z"
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
    .values
)


rho, p_value = spearmanr(
    date_signal,
    y_v3
)


print(
    f"\nSpearman correlation: "
    f"{rho:.6f}"
)

print(
    f"p-value: "
    f"{p_value:.6e}"
)


date_relative_results = []


# Use percentile thresholds
# because absolute robust-z values can be huge.

percentiles = np.linspace(
    50,
    99.9,
    150
)


for percentile in percentiles:

    threshold = np.percentile(
        date_signal,
        percentile
    )


    pred = (
        date_signal >= threshold
    ).astype(int)


    date_relative_results.append({

        "percentile": percentile,

        "threshold": threshold,

        "precision":
            precision_score(
                y_v3,
                pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_v3,
                pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_v3,
                pred,
                zero_division=0
            )
    })


date_relative_results = (
    pd.DataFrame(
        date_relative_results
    )
    .sort_values(
        "f1",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\nBEST DATE-RELATIVE RESULTS:"
)

display(
    date_relative_results.head(10)
)


print("\n" + "=" * 70)
print("V3 EXPERIMENT CHECKPOINT")
print("=" * 70)

print(
    "\nCompare these against our current "
    "OOF ensemble F1 = 0.800956"
)

print(
    "\nIf the date-relative signal is strong, "
    "we will inject it into XGBoost/LightGBM."
)

print(
    "\nIf it is weak, we discard it and move "
    "to the next experiment."
)

V3 — DATE-RELATIVE SENSOR SIGNAL

Spearman correlation: 0.145925
p-value: 0.000000e+00

BEST DATE-RELATIVE RESULTS:


,percentile,threshold,precision,recall,f1
0,99.565101,1.587652e+29,0.530323,0.291696,0.376374
1,99.230201,5.840645e+28,0.319236,0.332150,0.325565
2,96.551007,1.069751e+27,0.198577,0.821859,0.319867
3,96.885906,2.907886e+27,0.199150,0.781405,0.317405
4,96.216107,5.325979e+25,0.192212,0.853087,0.313736
5,95.881208,9.754870e+23,0.181128,0.871895,0.299945
6,97.220805,7.904453e+27,0.194002,0.656494,0.299498
7,97.555705,7.904453e+27,0.194002,0.656494,0.299498
8,97.890604,7.904453e+27,0.194002,0.656494,0.299498
9,98.560403,2.148653e+28,0.210013,0.465933,0.289526



V3 EXPERIMENT CHECKPOINT

Compare these against our current OOF ensemble F1 = 0.800956

If the date-relative signal is strong, we will inject it into XGBoost/LightGBM.

If it is weak, we discard it and move to the next experiment.


In [26]:
# ============================================================
# V4 - CELL 26
# HISTOGRAM GRADIENT BOOSTING DATA
# ============================================================

print("=" * 70)
print("V4 — HISTOGRAM GRADIENT BOOSTING")
print("=" * 70)

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# Automatically select numeric engineered features
# ------------------------------------------------------------

exclude_columns = {
    target_column,
    "_sensor_key",
    "_sensor_date_key",
    "Date"
}

hgb_features = [
    col
    for col in train_v2.columns
    if col not in exclude_columns
    and pd.api.types.is_numeric_dtype(
        train_v2[col]
    )
]


print(
    "\nNumber of HGB features:",
    len(hgb_features)
)

print(
    "\nFeatures:"
)

print(hgb_features)


# ------------------------------------------------------------
# Build train / validation matrices
# ------------------------------------------------------------

X_hgb_train = (
    train_v2.iloc[
        train_indices
    ][hgb_features]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

X_hgb_valid = (
    train_v2.iloc[
        validation_indices
    ][hgb_features]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)


y_hgb_train = (
    train_v2.iloc[
        train_indices
    ][target_column]
    .astype(str)
    .astype(int)
    .values
)

y_hgb_valid = (
    train_v2.iloc[
        validation_indices
    ][target_column]
    .astype(str)
    .astype(int)
    .values
)


print(
    "\nTrain shape:",
    X_hgb_train.shape
)

print(
    "Validation shape:",
    X_hgb_valid.shape
)

print("\n✓ HGB dataset ready.")

V4 — HISTOGRAM GRADIENT BOOSTING

Number of HGB features: 50

Features:
['X1', 'X2', 'X3', 'X4', 'X5', 'year', 'month', 'day', 'day_of_week', 'day_of_year', 'quarter', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin', 'day_of_year_cos', 'X1_log_abs', 'X1_log10_abs', 'X1_is_zero', 'X1_is_one', 'X1_large', 'X1_extreme', 'X2_log_abs', 'X2_log10_abs', 'X2_is_zero', 'X2_is_one', 'X2_large', 'X2_extreme', 'X3_log_abs', 'X3_log10_abs', 'X3_is_zero', 'X3_is_one', 'X3_large', 'X3_extreme', 'X4_log_abs', 'X4_log10_abs', 'X4_is_zero', 'X4_is_one', 'X4_large', 'X4_extreme', 'X5_log_abs', 'X5_log10_abs', 'X5_is_zero', 'X5_is_one', 'X5_large', 'X5_extreme', '_original_row', '_row_in_date', 'date_target_rate_loo']

Train shape: (1310948, 50)
Validation shape: (328476, 50)

✓ HGB dataset ready.


In [27]:
# ============================================================
# V4 - CELL 27
# STRATIFIED HGB TRAINING SAMPLE
# ============================================================

print("=" * 70)
print("BUILDING STRATIFIED HGB TRAINING SAMPLE")
print("=" * 70)

HGB_SAMPLE_SIZE = 500_000

rng = np.random.RandomState(
    RANDOM_STATE
)


# ------------------------------------------------------------
# Preserve all positive examples
# ------------------------------------------------------------

positive_idx = np.where(
    y_hgb_train == 1
)[0]

negative_idx = np.where(
    y_hgb_train == 0
)[0]


# Keep every positive example
# and randomly sample negatives.

remaining_size = (
    HGB_SAMPLE_SIZE
    - len(positive_idx)
)


sampled_negative_idx = (
    rng.choice(
        negative_idx,
        size=min(
            remaining_size,
            len(negative_idx)
        ),
        replace=False
    )
)


hgb_sample_idx = np.concatenate([
    positive_idx,
    sampled_negative_idx
])


rng.shuffle(
    hgb_sample_idx
)


X_hgb_sample = (
    X_hgb_train
    .iloc[hgb_sample_idx]
)

y_hgb_sample = (
    y_hgb_train[
        hgb_sample_idx
    ]
)


print(
    "\nOriginal training rows:",
    len(y_hgb_train)
)

print(
    "HGB training rows:",
    len(y_hgb_sample)
)

print(
    "\nClass distribution:"
)

print(
    pd.Series(
        y_hgb_sample
    ).value_counts(
        normalize=True
    )
    .sort_index()
    * 100
)

print("\n✓ Stratified sample ready.")

BUILDING STRATIFIED HGB TRAINING SAMPLE

Original training rows: 1310948
HGB training rows: 500000

Class distribution:
0    97.756
1     2.244
Name: proportion, dtype: float64

✓ Stratified sample ready.


In [28]:
# ============================================================
# V4 - CELL 28
# TRAIN HISTGRADIENTBOOSTING
# ============================================================

print("=" * 70)
print("TRAINING HISTGRADIENTBOOSTING")
print("=" * 70)

hgb_model = HistGradientBoostingClassifier(

    learning_rate=0.08,

    max_iter=350,

    max_leaf_nodes=31,

    max_depth=None,

    min_samples_leaf=30,

    l2_regularization=1.0,

    max_bins=255,

    early_stopping=True,

    validation_fraction=0.10,

    n_iter_no_change=30,

    random_state=RANDOM_STATE,

    verbose=1
)


print(
    "\nTraining HGB..."
)

hgb_model.fit(
    X_hgb_sample,
    y_hgb_sample
)


print(
    "\n✓ HGB training completed."
)

print(
    "Iterations used:",
    hgb_model.n_iter_
)

TRAINING HISTGRADIENTBOOSTING

Training HGB...
Binning 0.180 GB of training data: 1.413 s
Binning 0.020 GB of validation data: 0.072 s
Fitting gradient boosted rounds:
Fit 350 trees in 40.622 s, (10850 total leaves)
Time spent computing histograms: 28.747s
Time spent finding best splits:  0.862s
Time spent applying splits:      2.390s
Time spent predicting:           0.465s

✓ HGB training completed.
Iterations used: 350


In [29]:
# ============================================================
# V4 - CELL 29
# HGB VALIDATION + THRESHOLD OPTIMIZATION
# ============================================================

print("=" * 70)
print("HGB VALIDATION")
print("=" * 70)


hgb_valid_probability = (
    hgb_model
    .predict_proba(
        X_hgb_valid
    )[:, 1]
)


hgb_threshold_results = []


for threshold in np.arange(
    0.10,
    0.70,
    0.005
):

    pred = (
        hgb_valid_probability
        >= threshold
    ).astype(int)


    hgb_threshold_results.append({

        "threshold": threshold,

        "accuracy":
            accuracy_score(
                y_hgb_valid,
                pred
            ),

        "precision":
            precision_score(
                y_hgb_valid,
                pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_hgb_valid,
                pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_hgb_valid,
                pred,
                zero_division=0
            )
    })


hgb_threshold_results = (
    pd.DataFrame(
        hgb_threshold_results
    )
    .sort_values(
        "f1",
        ascending=False
    )
    .reset_index(drop=True)
)


best_hgb = (
    hgb_threshold_results.iloc[0]
)


print("\nBEST HGB THRESHOLD")

display(
    hgb_threshold_results.head(10)
)


print(
    f"\nBest threshold: "
    f"{best_hgb['threshold']:.3f}"
)

print(
    f"Accuracy: "
    f"{best_hgb['accuracy']:.6f}"
)

print(
    f"Precision: "
    f"{best_hgb['precision']:.6f}"
)

print(
    f"Recall: "
    f"{best_hgb['recall']:.6f}"
)

print(
    f"F1: "
    f"{best_hgb['f1']:.6f}"
)

HGB VALIDATION

BEST HGB THRESHOLD


,threshold,accuracy,precision,recall,f1
0,0.620,0.997026,0.871319,0.766501,0.815556
1,0.615,0.997017,0.869076,0.767921,0.815373
2,0.590,0.996986,0.859559,0.775373,0.815299
3,0.595,0.996989,0.860750,0.774308,0.815244
4,0.625,0.997029,0.873782,0.764017,0.815222
5,0.640,0.997047,0.880874,0.758339,0.815027
6,0.630,0.997032,0.875662,0.762243,0.815026
7,0.635,0.997038,0.878850,0.759404,0.814773
8,0.580,0.996962,0.854361,0.778566,0.814705
9,0.600,0.996986,0.862124,0.772179,0.814676



Best threshold: 0.620
Accuracy: 0.997026
Precision: 0.871319
Recall: 0.766501
F1: 0.815556


In [30]:
# ============================================================
# V4 - CELL 30
# HGB VS CURRENT CHAMPIONS
# ============================================================

print("=" * 70)
print("HGB VS CURRENT CHAMPIONS")
print("=" * 70)


hgb_comparison = pd.DataFrame({

    "model": [

        "XGBoost",

        "LightGBM",

        "XGB + LightGBM",

        "HistGradientBoosting"
    ],

    "OOF_F1": [

        0.799669,

        0.795778,

        0.800956,

        best_hgb["f1"]
    ],

    "threshold": [

        0.345,

        0.345,

        0.330,

        best_hgb["threshold"]
    ]
})


hgb_comparison[
    "improvement_vs_current"
] = (
    hgb_comparison["OOF_F1"]
    - 0.800956
)


display(
    hgb_comparison
    .sort_values(
        "OOF_F1",
        ascending=False
    )
)


print(
    "\nHGB improvement over current ensemble:"
)

print(
    f"{best_hgb['f1'] - 0.800956:+.6f}"
)

HGB VS CURRENT CHAMPIONS


,model,OOF_F1,threshold,improvement_vs_current
3,HistGradientBoosting,0.815556,0.620,0.014600
2,XGB + LightGBM,0.800956,0.330,0.000000
0,XGBoost,0.799669,0.345,-0.001287
1,LightGBM,0.795778,0.345,-0.005178



HGB improvement over current ensemble:
+0.014600


In [31]:
# ============================================================
# V4 - CELL 31
# MODEL PREDICTION DIVERSITY
# ============================================================

print("=" * 70)
print("PREDICTION DIVERSITY ANALYSIS")
print("=" * 70)


# ------------------------------------------------------------
# Expected existing variables:
#
# xgb_oof_probability
# lgb_oof_probability
#
# If your notebook used different names, this cell
# will identify that immediately.
# ------------------------------------------------------------

required_vars = [
    "xgb_oof_probability",
    "lgb_oof_probability"
]


missing_vars = [
    v
    for v in required_vars
    if v not in globals()
]


if missing_vars:

    print(
        "\n⚠ Missing variables:"
    )

    print(
        missing_vars
    )

    print(
        "\nDo NOT continue to Cell 32."
    )

else:

    diversity_df = pd.DataFrame({

        "XGBoost":
            xgb_oof_probability,

        "LightGBM":
            lgb_oof_probability,

        "HistGradientBoosting":
            hgb_valid_probability
    })


    print(
        "\nPrediction correlation:"
    )

    display(
        diversity_df.corr(
            method="pearson"
        )
    )


    print(
        "\nMean predictions:"
    )

    display(
        diversity_df.mean()
    )


    print(
        "\n✓ Diversity analysis complete."
    )

PREDICTION DIVERSITY ANALYSIS

⚠ Missing variables:
['xgb_oof_probability', 'lgb_oof_probability']

Do NOT continue to Cell 32.


In [32]:
# ============================================================
# CELL 33 — SAME-VALIDATION PREDICTIONS
# ============================================================

print("=" * 60)
print("RECOVERING XGB / LIGHTGBM PREDICTIONS")
print("=" * 60)

print("Validation shape:", X_hgb_valid.shape)
print("HGB probability shape:", hgb_valid_probability.shape)

# Try to locate trained models from earlier notebook cells
model_candidates = {
    "XGB": ["xgb_model", "xgb_best_model", "xgb_clf"],
    "LGB": ["lgb_model", "lgb_best_model", "lgb_clf", "lightgbm_model"]
}

found_models = {}

for model_name, candidates in model_candidates.items():
    for var_name in candidates:
        if var_name in globals():
            found_models[model_name] = globals()[var_name]
            print(f"✓ Found {model_name}: {var_name}")
            break

print("\nModels found:", list(found_models.keys()))

if "XGB" in found_models:
    xgb_valid_probability_same = found_models["XGB"].predict_proba(
        X_hgb_valid
    )[:, 1]
    print("✓ XGB validation predictions generated")

if "LGB" in found_models:
    lgb_valid_probability_same = found_models["LGB"].predict_proba(
        X_hgb_valid
    )[:, 1]
    print("✓ LightGBM validation predictions generated")

print("\nDone.")

RECOVERING XGB / LIGHTGBM PREDICTIONS
Validation shape: (328476, 50)
HGB probability shape: (328476,)

Models found: []

Done.


In [33]:
# ============================================================
# CELL 34 — FAIR MODEL COMPARISON
# ============================================================

from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np
import pandas as pd

def find_best_threshold(y_true, probabilities,
                        low=0.10, high=0.90, step=0.005):

    thresholds = np.arange(low, high + step, step)

    best = {
        "f1": -1,
        "threshold": None,
        "precision": None,
        "recall": None
    }

    for threshold in thresholds:
        pred = (probabilities >= threshold).astype(int)

        f1 = f1_score(y_true, pred)

        if f1 > best["f1"]:
            best["f1"] = f1
            best["threshold"] = threshold
            best["precision"] = precision_score(y_true, pred, zero_division=0)
            best["recall"] = recall_score(y_true, pred, zero_division=0)

    return best


same_split_results = []

# HGB
hgb_best_same = find_best_threshold(
    y_hgb_valid,
    hgb_valid_probability
)

same_split_results.append([
    "HistGradientBoosting",
    hgb_best_same["f1"],
    hgb_best_same["threshold"],
    hgb_best_same["precision"],
    hgb_best_same["recall"]
])

# XGB
if "xgb_valid_probability_same" in globals():

    xgb_best_same = find_best_threshold(
        y_hgb_valid,
        xgb_valid_probability_same
    )

    same_split_results.append([
        "XGBoost",
        xgb_best_same["f1"],
        xgb_best_same["threshold"],
        xgb_best_same["precision"],
        xgb_best_same["recall"]
    ])

# LGB
if "lgb_valid_probability_same" in globals():

    lgb_best_same = find_best_threshold(
        y_hgb_valid,
        lgb_valid_probability_same
    )

    same_split_results.append([
        "LightGBM",
        lgb_best_same["f1"],
        lgb_best_same["threshold"],
        lgb_best_same["precision"],
        lgb_best_same["recall"]
    ])

comparison_df = pd.DataFrame(
    same_split_results,
    columns=[
        "model",
        "F1",
        "threshold",
        "precision",
        "recall"
    ]
).sort_values("F1", ascending=False)

print("\n" + "=" * 75)
print("FAIR SAME-VALIDATION COMPARISON")
print("=" * 75)

display(comparison_df)

print("\nHGB F1:", round(hgb_best_same["f1"], 6))


FAIR SAME-VALIDATION COMPARISON


,model,F1,threshold,precision,recall
0,HistGradientBoosting,0.815556,0.62,0.871319,0.766501



HGB F1: 0.815556


In [34]:
# ============================================================
# CELL 35 — PREDICTION DIVERSITY
# ============================================================

print("=" * 60)
print("PREDICTION DIVERSITY")
print("=" * 60)

probability_dict = {
    "HGB": hgb_valid_probability
}

if "xgb_valid_probability_same" in globals():
    probability_dict["XGB"] = xgb_valid_probability_same

if "lgb_valid_probability_same" in globals():
    probability_dict["LGB"] = lgb_valid_probability_same

prob_df = pd.DataFrame(probability_dict)

print("\nProbability correlations:")
display(prob_df.corr().round(4))

print("\nProbability summary:")
display(prob_df.describe().T)

# Pairwise prediction agreement at each model's optimized threshold
print("\nPrediction disagreement:")

thresholds = {
    "HGB": hgb_best_same["threshold"]
}

if "xgb_best_same" in globals():
    thresholds["XGB"] = xgb_best_same["threshold"]

if "lgb_best_same" in globals():
    thresholds["LGB"] = lgb_best_same["threshold"]

prediction_df = pd.DataFrame({
    model: (prob_df[model] >= thresholds[model]).astype(int)
    for model in prob_df.columns
})

display(prediction_df.mean().rename("predicted_anomaly_rate").to_frame())

print("\nThis tells us whether HGB is genuinely adding new information.")

PREDICTION DIVERSITY

Probability correlations:


,HGB
HGB,1.0



Probability summary:


,count,mean,std,min,25%,50%,75%,max
HGB,328476.0,0.010682,0.083567,2.198689e-07,0.000009,0.000047,0.00031,0.999995



Prediction disagreement:


,predicted_anomaly_rate
HGB,0.007547



This tells us whether HGB is genuinely adding new information.


In [35]:
# ============================================================
# CELL 36 — HGB + XGB BLEND SEARCH
# ============================================================

if "xgb_valid_probability_same" not in globals():

    print("XGB model not available in memory.")
    print("Skipping XGB blend.")

else:

    print("=" * 60)
    print("HGB + XGB BLEND SEARCH")
    print("=" * 60)

    best_blend = {
        "f1": -1,
        "hgb_weight": None,
        "xgb_weight": None,
        "threshold": None,
        "precision": None,
        "recall": None
    }

    # HGB weight from 0.00 -> 1.00
    for hgb_weight in np.arange(0.0, 1.01, 0.025):

        xgb_weight = 1.0 - hgb_weight

        blend_probability = (
            hgb_weight * hgb_valid_probability +
            xgb_weight * xgb_valid_probability_same
        )

        for threshold in np.arange(0.20, 0.81, 0.005):

            pred = (blend_probability >= threshold).astype(int)

            f1 = f1_score(
                y_hgb_valid,
                pred
            )

            if f1 > best_blend["f1"]:

                best_blend.update({
                    "f1": f1,
                    "hgb_weight": hgb_weight,
                    "xgb_weight": xgb_weight,
                    "threshold": threshold,
                    "precision": precision_score(
                        y_hgb_valid,
                        pred,
                        zero_division=0
                    ),
                    "recall": recall_score(
                        y_hgb_valid,
                        pred,
                        zero_division=0
                    )
                })

    print("\nBEST HGB + XGB:")
    print(best_blend)

    print(
        "\nImprovement over HGB:",
        round(
            best_blend["f1"] - hgb_best_same["f1"],
            6
        )
    )

XGB model not available in memory.
Skipping XGB blend.


In [36]:
# ============================================================
# CELL 37 — HGB + LIGHTGBM BLEND SEARCH
# ============================================================

if "lgb_valid_probability_same" not in globals():

    print("LightGBM model not available in memory.")
    print("Skipping LGB blend.")

else:

    print("=" * 60)
    print("HGB + LIGHTGBM BLEND SEARCH")
    print("=" * 60)

    best_hgb_lgb = {
        "f1": -1,
        "hgb_weight": None,
        "lgb_weight": None,
        "threshold": None,
        "precision": None,
        "recall": None
    }

    for hgb_weight in np.arange(0.0, 1.01, 0.025):

        lgb_weight = 1.0 - hgb_weight

        blend_probability = (
            hgb_weight * hgb_valid_probability +
            lgb_weight * lgb_valid_probability_same
        )

        for threshold in np.arange(0.20, 0.81, 0.005):

            pred = (blend_probability >= threshold).astype(int)

            f1 = f1_score(
                y_hgb_valid,
                pred
            )

            if f1 > best_hgb_lgb["f1"]:

                best_hgb_lgb.update({
                    "f1": f1,
                    "hgb_weight": hgb_weight,
                    "lgb_weight": lgb_weight,
                    "threshold": threshold,
                    "precision": precision_score(
                        y_hgb_valid,
                        pred,
                        zero_division=0
                    ),
                    "recall": recall_score(
                        y_hgb_valid,
                        pred,
                        zero_division=0
                    )
                })

    print("\nBEST HGB + LGB:")
    print(best_hgb_lgb)

    print(
        "\nImprovement over HGB:",
        round(
            best_hgb_lgb["f1"] - hgb_best_same["f1"],
            6
        )
    )

LightGBM model not available in memory.
Skipping LGB blend.


In [37]:
# ============================================================
# CELL 38 — THREE-MODEL SAME-VALIDATION BLEND
# ============================================================

available_models = ["HGB"]

if "xgb_valid_probability_same" in globals():
    available_models.append("XGB")

if "lgb_valid_probability_same" in globals():
    available_models.append("LGB")

print("=" * 60)
print("THREE-MODEL BLEND")
print("=" * 60)
print("Available:", available_models)

if len(available_models) == 3:

    best_three = {
        "f1": -1
    }

    # Search HGB weight, remaining  weight split XGB/LGB 3:1
    for hgb_weight in np.arange(0.0, 0.81, 0.025):

        remaining = 1.0 - hgb_weight

        xgb_weight = remaining * 0.75
        lgb_weight = remaining * 0.25

        blend_probability = (
            hgb_weight * hgb_valid_probability +
            xgb_weight * xgb_valid_probability_same +
            lgb_weight * lgb_valid_probability_same
        )

        for threshold in np.arange(0.20, 0.81, 0.005):

            pred = (blend_probability >= threshold).astype(int)

            f1 = f1_score(
                y_hgb_valid,
                pred
            )

            if f1 > best_three["f1"]:

                best_three = {
                    "f1": f1,
                    "hgb_weight": hgb_weight,
                    "xgb_weight": xgb_weight,
                    "lgb_weight": lgb_weight,
                    "threshold": threshold,
                    "precision": precision_score(
                        y_hgb_valid,
                        pred,
                        zero_division=0
                    ),
                    "recall": recall_score(
                        y_hgb_valid,
                        pred,
                        zero_division=0
                    )
                }

    print("\nBEST THREE-MODEL BLEND:")
    print(best_three)

    print(
        "\nImprovement over HGB:",
        round(
            best_three["f1"] - hgb_best_same["f1"],
            6
        )
    )

else:

    print("\nNot enough models available for 3-model blend.")

THREE-MODEL BLEND
Available: ['HGB']

Not enough models available for 3-model blend.


In [38]:
# ============================================================
# CELL 39 — HGB V2: MORE EXPRESSIVE MODEL
# ============================================================

print("=" * 60)
print("TRAINING HGB V2")
print("=" * 60)

hgb_v2_model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=500,
    max_leaf_nodes=63,
    max_depth=None,
    min_samples_leaf=25,
    l2_regularization=1.5,
    max_bins=255,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=35,
    random_state=RANDOM_STATE + 100,
    verbose=1
)

hgb_v2_model.fit(
    X_hgb_sample,
    y_hgb_sample
)

hgb_v2_valid_probability = hgb_v2_model.predict_proba(
    X_hgb_valid
)[:, 1]

hgb_v2_best = find_best_threshold(
    y_hgb_valid,
    hgb_v2_valid_probability,
    low=0.10,
    high=0.90,
    step=0.005
)

print("\n" + "=" * 60)
print("HGB V2 RESULT")
print("=" * 60)

print("Best F1      :", round(hgb_v2_best["f1"], 6))
print("Threshold    :", hgb_v2_best["threshold"])
print("Precision    :", round(hgb_v2_best["precision"], 6))
print("Recall       :", round(hgb_v2_best["recall"], 6))

print("\nComparison:")
print("HGB V1 F1    :", round(hgb_best_same["f1"], 6))
print("HGB V2 F1    :", round(hgb_v2_best["f1"], 6))
print(
    "Improvement  :",
    round(hgb_v2_best["f1"] - hgb_best_same["f1"], 6)
)

TRAINING HGB V2
Binning 0.180 GB of training data: 0.978 s
Binning 0.020 GB of validation data: 0.060 s
Fitting gradient boosted rounds:
Fit 500 trees in 61.971 s, (31500 total leaves)
Time spent computing histograms: 45.383s
Time spent finding best splits:  1.887s
Time spent applying splits:      4.090s
Time spent predicting:           0.668s

HGB V2 RESULT
Best F1      : 0.826883
Threshold    : 0.5850000000000004
Precision    : 0.883421
Recall       : 0.777147

Comparison:
HGB V1 F1    : 0.815556
HGB V2 F1    : 0.826883
Improvement  : 0.011327


In [39]:
# ============================================================
# CELL 40 — HGB V1 vs V2 DIVERSITY
# ============================================================

print("=" * 70)
print("HGB V1 vs V2 — PREDICTION DIVERSITY")
print("=" * 70)

from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np
import pandas as pd

print("V1 predictions:", hgb_valid_probability.shape)
print("V2 predictions:", hgb_v2_valid_probability.shape)

corr = np.corrcoef(
    hgb_valid_probability,
    hgb_v2_valid_probability
)[0, 1]

print("\nProbability correlation:", round(corr, 6))

v1_pred = (
    hgb_valid_probability >= hgb_best_same["threshold"]
).astype(int)

v2_pred = (
    hgb_v2_valid_probability >= hgb_v2_best["threshold"]
).astype(int)

disagreement = np.mean(v1_pred != v2_pred)

print("Prediction disagreement:", round(disagreement, 6))
print("V1 anomaly rate:", round(v1_pred.mean(), 6))
print("V2 anomaly rate:", round(v2_pred.mean(), 6))

print("\nInterpretation:")
if disagreement < 0.01:
    print("Very similar models — ensemble may add little.")
elif disagreement < 0.03:
    print("Moderate diversity — ensemble may help.")
else:
    print("GOOD diversity — ensemble has real potential.")

HGB V1 vs V2 — PREDICTION DIVERSITY
V1 predictions: (328476,)
V2 predictions: (328476,)

Probability correlation: 0.980415
Prediction disagreement: 0.00109
V1 anomaly rate: 0.007547
V2 anomaly rate: 0.007547

Interpretation:
Very similar models — ensemble may add little.


In [41]:
# ============================================================
# CELL 41 — TEMPORAL ROBUSTNESS — FIXED
# ============================================================

print("=" * 70)
print("TEMPORAL ROBUSTNESS — V1 vs V2")
print("=" * 70)

# Get validation dates
valid_indices = X_hgb_valid.index

valid_dates = pd.to_datetime(
    train_v2.loc[valid_indices, "Date"]
)

# y_hgb_valid is already a NumPy array
temporal_check = pd.DataFrame({
    "year": valid_dates.dt.year.to_numpy(),
    "y_true": np.asarray(y_hgb_valid),
    "v1_prob": np.asarray(hgb_valid_probability),
    "v2_prob": np.asarray(hgb_v2_valid_probability)
})

print("Temporal validation data shape:", temporal_check.shape)

rows = []

for year, group in temporal_check.groupby("year"):

    v1_pred = (
        group["v1_prob"] >= hgb_best_same["threshold"]
    ).astype(int)

    v2_pred = (
        group["v2_prob"] >= hgb_v2_best["threshold"]
    ).astype(int)

    rows.append({
        "year": year,
        "rows": len(group),
        "anomalies": int(group["y_true"].sum()),
        "anomaly_rate": group["y_true"].mean(),

        "V1_F1": f1_score(
            group["y_true"],
            v1_pred,
            zero_division=0
        ),

        "V2_F1": f1_score(
            group["y_true"],
            v2_pred,
            zero_division=0
        )
    })

temporal_results = (
    pd.DataFrame(rows)
    .sort_values("year")
    .reset_index(drop=True)
)

display(
    temporal_results.style.format({
        "anomaly_rate": "{:.4%}",
        "V1_F1": "{:.4f}",
        "V2_F1": "{:.4f}"
    })
)

print("\nAverage V1 F1:",
      round(temporal_results["V1_F1"].mean(), 6))

print("Average V2 F1:",
      round(temporal_results["V2_F1"].mean(), 6))

print(
    "\nV2 - V1 average yearly F1 improvement:",
    round(
        temporal_results["V2_F1"].mean()
        - temporal_results["V1_F1"].mean(),
        6
    )
)

TEMPORAL ROBUSTNESS — V1 vs V2
Temporal validation data shape: (328476, 4)


,year,rows,anomalies,anomaly_rate,V1_F1,V2_F1
0,2020,3530,85,2.4079%,0.7673,0.8258
1,2021,83605,1045,1.2499%,0.8179,0.8223
2,2022,84440,737,0.8728%,0.8214,0.8250
3,2023,83567,609,0.7288%,0.7948,0.8263
4,2024,73334,342,0.4664%,0.8451,0.8464



Average V1 F1: 0.809267
Average V2 F1: 0.829153

V2 - V1 average yearly F1 improvement: 0.019886


In [42]:
# ============================================================
# CELL 42 — HGB V1 + V2 BLEND
# ============================================================

print("=" * 70)
print("HGB V1 + V2 BLEND SEARCH")
print("=" * 70)

best_v1_v2 = {
    "f1": -1
}

for v2_weight in np.arange(0.0, 1.001, 0.025):

    v1_weight = 1.0 - v2_weight

    blend_probability = (
        v1_weight * hgb_valid_probability +
        v2_weight * hgb_v2_valid_probability
    )

    for threshold in np.arange(0.20, 0.801, 0.005):

        pred = (
            blend_probability >= threshold
        ).astype(int)

        f1 = f1_score(
            y_hgb_valid,
            pred,
            zero_division=0
        )

        if f1 > best_v1_v2["f1"]:

            best_v1_v2 = {
                "f1": f1,
                "v1_weight": v1_weight,
                "v2_weight": v2_weight,
                "threshold": threshold,
                "precision": precision_score(
                    y_hgb_valid,
                    pred,
                    zero_division=0
                ),
                "recall": recall_score(
                    y_hgb_valid,
                    pred,
                    zero_division=0
                )
            }

print("\nBEST V1 + V2:")
print(best_v1_v2)

print(
    "\nImprovement over V2:",
    round(
        best_v1_v2["f1"] - hgb_v2_best["f1"],
        6
    )
)

HGB V1 + V2 BLEND SEARCH

BEST V1 + V2:
{'f1': 0.8280021933832937, 'v1_weight': np.float64(0.29999999999999993), 'v2_weight': np.float64(0.7000000000000001), 'threshold': np.float64(0.5150000000000003), 'precision': 0.8537504711647191, 'recall': 0.8037615330021292}

Improvement over V2: 0.001119


In [43]:
# ============================================================
# CELL 43 — HGB V3
# ============================================================

print("=" * 70)
print("TRAINING HGB V3")
print("=" * 70)

hgb_v3_model = HistGradientBoostingClassifier(
    learning_rate=0.04,
    max_iter=650,
    max_leaf_nodes=127,
    max_depth=None,
    min_samples_leaf=35,
    l2_regularization=2.0,
    max_bins=255,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=40,
    random_state=RANDOM_STATE + 200,
    verbose=1
)

hgb_v3_model.fit(
    X_hgb_sample,
    y_hgb_sample
)

hgb_v3_valid_probability = (
    hgb_v3_model.predict_proba(X_hgb_valid)[:, 1]
)

hgb_v3_best = find_best_threshold(
    y_hgb_valid,
    hgb_v3_valid_probability,
    low=0.10,
    high=0.90,
    step=0.005
)

print("\n" + "=" * 70)
print("HGB V3 RESULT")
print("=" * 70)

print("Best F1     :", round(hgb_v3_best["f1"], 6))
print("Threshold   :", hgb_v3_best["threshold"])
print("Precision   :", round(hgb_v3_best["precision"], 6))
print("Recall      :", round(hgb_v3_best["recall"], 6))

print("\nMODEL RANKING")

ranking = pd.DataFrame([
    ["HGB V1", hgb_best_same["f1"]],
    ["HGB V2", hgb_v2_best["f1"]],
    ["HGB V3", hgb_v3_best["f1"]]
], columns=["model", "F1"]).sort_values(
    "F1",
    ascending=False
)

display(ranking)

TRAINING HGB V3
Binning 0.180 GB of training data: 1.403 s
Binning 0.020 GB of validation data: 0.105 s
Fitting gradient boosted rounds:
Fit 650 trees in 200.682 s, (82550 total leaves)
Time spent computing histograms: 139.680s
Time spent finding best splits:  14.095s
Time spent applying splits:      21.263s
Time spent predicting:           1.811s

HGB V3 RESULT
Best F1     : 0.836033
Threshold   : 0.4500000000000003
Precision   : 0.85805
Recall      : 0.815117

MODEL RANKING


,model,F1
2,HGB V3,0.836033
1,HGB V2,0.826883
0,HGB V1,0.815556


In [44]:
# ============================================================
# CELL 44 — V2 vs V3 DIVERSITY
# ============================================================

print("=" * 70)
print("HGB V2 vs V3 DIVERSITY")
print("=" * 70)

v2_v3_corr = np.corrcoef(
    hgb_v2_valid_probability,
    hgb_v3_valid_probability
)[0, 1]

v2_pred = (
    hgb_v2_valid_probability >= hgb_v2_best["threshold"]
).astype(int)

v3_pred = (
    hgb_v3_valid_probability >= hgb_v3_best["threshold"]
).astype(int)

v2_v3_disagreement = np.mean(
    v2_pred != v3_pred
)

print("Probability correlation:",
      round(v2_v3_corr, 6))

print("Prediction disagreement:",
      round(v2_v3_disagreement, 6))

print("V2 anomaly rate:",
      round(v2_pred.mean(), 6))

print("V3 anomaly rate:",
      round(v3_pred.mean(), 6))

print("\nV2 F1:", round(hgb_v2_best["f1"], 6))
print("V3 F1:", round(hgb_v3_best["f1"], 6))

HGB V2 vs V3 DIVERSITY
Probability correlation: 0.97868
Prediction disagreement: 0.00123
V2 anomaly rate: 0.007547
V3 anomaly rate: 0.00815

V2 F1: 0.826883
V3 F1: 0.836033


In [45]:
# ============================================================
# CELL 45 — HGB V2 + V3 BLEND
# ============================================================

print("=" * 70)
print("HGB V2 + V3 BLEND SEARCH")
print("=" * 70)

best_v2_v3 = {
    "f1": -1
}

for v3_weight in np.arange(0.0, 1.001, 0.025):

    v2_weight = 1.0 - v3_weight

    blend_probability = (
        v2_weight * hgb_v2_valid_probability +
        v3_weight * hgb_v3_valid_probability
    )

    for threshold in np.arange(0.20, 0.801, 0.005):

        pred = (
            blend_probability >= threshold
        ).astype(int)

        f1 = f1_score(
            y_hgb_valid,
            pred,
            zero_division=0
        )

        if f1 > best_v2_v3["f1"]:

            best_v2_v3 = {
                "f1": f1,
                "v2_weight": v2_weight,
                "v3_weight": v3_weight,
                "threshold": threshold,
                "precision": precision_score(
                    y_hgb_valid,
                    pred,
                    zero_division=0
                ),
                "recall": recall_score(
                    y_hgb_valid,
                    pred,
                    zero_division=0
                )
            }

print("\nBEST V2 + V3:")
print(best_v2_v3)

print(
    "\nImprovement over V2:",
    round(
        best_v2_v3["f1"] - hgb_v2_best["f1"],
        6
    )
)

HGB V2 + V3 BLEND SEARCH

BEST V2 + V3:
{'f1': 0.8384768092402094, 'v2_weight': np.float64(0.32499999999999996), 'v3_weight': np.float64(0.675), 'threshold': np.float64(0.4400000000000002), 'precision': 0.8531031950055087, 'recall': 0.8243435060326473}

Improvement over V2: 0.011594


In [46]:
# ============================================================
# CELL 46 — V1 + V2 + V3 ENSEMBLE
# ============================================================

print("=" * 70)
print("HGB THREE-MODEL ENSEMBLE")
print("=" * 70)

best_hgb_ensemble = {
    "f1": -1
}

# Coarse but efficient grid
weights = np.arange(0.0, 1.001, 0.05)

for v1_weight in weights:

    for v2_weight in weights:

        v3_weight = 1.0 - v1_weight - v2_weight

        if v3_weight < 0 or v3_weight > 1:
            continue

        blend_probability = (
            v1_weight * hgb_valid_probability +
            v2_weight * hgb_v2_valid_probability +
            v3_weight * hgb_v3_valid_probability
        )

        for threshold in np.arange(0.30, 0.751, 0.005):

            pred = (
                blend_probability >= threshold
            ).astype(int)

            f1 = f1_score(
                y_hgb_valid,
                pred,
                zero_division=0
            )

            if f1 > best_hgb_ensemble["f1"]:

                best_hgb_ensemble = {
                    "f1": f1,
                    "v1_weight": v1_weight,
                    "v2_weight": v2_weight,
                    "v3_weight": v3_weight,
                    "threshold": threshold,
                    "precision": precision_score(
                        y_hgb_valid,
                        pred,
                        zero_division=0
                    ),
                    "recall": recall_score(
                        y_hgb_valid,
                        pred,
                        zero_division=0
                    )
                }

print("\n" + "=" * 70)
print("BEST HGB ENSEMBLE")
print("=" * 70)

print(best_hgb_ensemble)

print("\nComparison:")
print("HGB V1 :", round(hgb_best_same["f1"], 6))
print("HGB V2 :", round(hgb_v2_best["f1"], 6))
print("HGB V3 :", round(hgb_v3_best["f1"], 6))
print("V1+V2  :", round(best_v1_v2["f1"], 6))
print("V2+V3  :", round(best_v2_v3["f1"], 6))
print("ENSEMBLE:", round(best_hgb_ensemble["f1"], 6))

print(
    "\nEnsemble improvement over V2:",
    round(
        best_hgb_ensemble["f1"] - hgb_v2_best["f1"],
        6
    )
)

HGB THREE-MODEL ENSEMBLE

BEST HGB ENSEMBLE
{'f1': 0.8386738284867568, 'v1_weight': np.float64(0.30000000000000004), 'v2_weight': np.float64(0.1), 'v3_weight': np.float64(0.6), 'threshold': np.float64(0.5200000000000002), 'precision': 0.8771793878341728, 'recall': 0.8034066713981547}

Comparison:
HGB V1 : 0.815556
HGB V2 : 0.826883
HGB V3 : 0.836033
V1+V2  : 0.828002
V2+V3  : 0.838477
ENSEMBLE: 0.838674

Ensemble improvement over V2: 0.011791


In [47]:
# ============================================================
# CELL 47 — LOCK CURRENT CHAMPION
# ============================================================

print("=" * 70)
print("CURRENT CHAMPION")
print("=" * 70)

CURRENT_BEST_F1 = best_hgb_ensemble["f1"]
CURRENT_V1_WEIGHT = best_hgb_ensemble["v1_weight"]
CURRENT_V2_WEIGHT = best_hgb_ensemble["v2_weight"]
CURRENT_V3_WEIGHT = best_hgb_ensemble["v3_weight"]
CURRENT_THRESHOLD = best_hgb_ensemble["threshold"]

print("F1        :", round(CURRENT_BEST_F1, 6))
print("V1 weight :", CURRENT_V1_WEIGHT)
print("V2 weight :", CURRENT_V2_WEIGHT)
print("V3 weight :", CURRENT_V3_WEIGHT)
print("Threshold :", CURRENT_THRESHOLD)

print("\nThis is our current experimental champion.")

CURRENT CHAMPION
F1        : 0.838674
V1 weight : 0.30000000000000004
V2 weight : 0.1
V3 weight : 0.6
Threshold : 0.5200000000000002

This is our current experimental champion.


In [48]:
# ============================================================
# CELL 48 — TEMPORAL ROBUSTNESS OF ALL 3 HGB MODELS
# ============================================================

print("=" * 70)
print("TEMPORAL ROBUSTNESS — HGB ENSEMBLE")
print("=" * 70)

valid_indices = X_hgb_valid.index

valid_dates = pd.to_datetime(
    train_v2.loc[valid_indices, "Date"]
)

ensemble_probability = (
    CURRENT_V1_WEIGHT * hgb_valid_probability +
    CURRENT_V2_WEIGHT * hgb_v2_valid_probability +
    CURRENT_V3_WEIGHT * hgb_v3_valid_probability
)

temporal_check = pd.DataFrame({
    "year": valid_dates.dt.year.to_numpy(),
    "y_true": np.asarray(y_hgb_valid),
    "ensemble_prob": ensemble_probability
})

rows = []

for year, group in temporal_check.groupby("year"):

    pred = (
        group["ensemble_prob"] >= CURRENT_THRESHOLD
    ).astype(int)

    rows.append({
        "year": year,
        "rows": len(group),
        "anomalies": int(group["y_true"].sum()),
        "anomaly_rate": group["y_true"].mean(),
        "F1": f1_score(
            group["y_true"],
            pred,
            zero_division=0
        ),
        "precision": precision_score(
            group["y_true"],
            pred,
            zero_division=0
        ),
        "recall": recall_score(
            group["y_true"],
            pred,
            zero_division=0
        )
    })

temporal_results_ensemble = (
    pd.DataFrame(rows)
    .sort_values("year")
    .reset_index(drop=True)
)

display(
    temporal_results_ensemble.style.format({
        "anomaly_rate": "{:.4%}",
        "F1": "{:.4f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}"
    })
)

print(
    "\nMean yearly F1:",
    round(temporal_results_ensemble["F1"].mean(), 6)
)

print(
    "Minimum yearly F1:",
    round(temporal_results_ensemble["F1"].min(), 6)
)

TEMPORAL ROBUSTNESS — HGB ENSEMBLE


,year,rows,anomalies,anomaly_rate,F1,precision,recall
0,2020,3530,85,2.4079%,0.7950,0.8421,0.7529
1,2021,83605,1045,1.2499%,0.8341,0.8758,0.7962
2,2022,84440,737,0.8728%,0.8464,0.8846,0.8114
3,2023,83567,609,0.7288%,0.8376,0.8639,0.8128
4,2024,73334,342,0.4664%,0.8488,0.8987,0.8041



Mean yearly F1: 0.832374
Minimum yearly F1: 0.795031


In [49]:
# ============================================================
# CELL 49 — SECOND GROUP-AWARE VALIDATION SPLIT
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

print("=" * 70)
print("CREATING SECOND INDEPENDENT VALIDATION SPLIT")
print("=" * 70)

# Same grouping principle used previously
second_groups = (
    train_v2["X1"].astype(str) + "_" +
    train_v2["X2"].astype(str) + "_" +
    train_v2["X3"].astype(str) + "_" +
    train_v2["X4"].astype(str) + "_" +
    train_v2["X5"].astype(str) + "_" +
    train_v2["Date"].astype(str)
)

gss2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE + 999
)

idx_train2, idx_valid2 = next(
    gss2.split(
        train_v2,
        train_v2["target"].astype(int),
        groups=second_groups
    )
)

print("Train rows :", len(idx_train2))
print("Valid rows :", len(idx_valid2))

print(
    "Group overlap:",
    len(
        set(second_groups.iloc[idx_train2])
        &
        set(second_groups.iloc[idx_valid2])
    )
)

print(
    "\nValidation positive rate:",
    round(
        train_v2.iloc[idx_valid2]["target"].astype(int).mean(),
        6
    )
)

CREATING SECOND INDEPENDENT VALIDATION SPLIT
Train rows : 1311258
Valid rows : 328166
Group overlap: 0

Validation positive rate: 0.008538


In [50]:
# ============================================================
# CELL 50 — SECOND-SPLIT HGB DATA
# ============================================================

print("=" * 70)
print("BUILDING SECOND-SPLIT DATA")
print("=" * 70)

X_second_train = train_v2.iloc[idx_train2][hgb_features]
X_second_valid = train_v2.iloc[idx_valid2][hgb_features]

y_second_train = (
    train_v2.iloc[idx_train2]["target"]
    .astype(int)
    .to_numpy()
)

y_second_valid = (
    train_v2.iloc[idx_valid2]["target"]
    .astype(int)
    .to_numpy()
)

print("Train:", X_second_train.shape)
print("Valid:", X_second_valid.shape)

print(
    "Train anomaly rate:",
    round(y_second_train.mean(), 6)
)

print(
    "Valid anomaly rate:",
    round(y_second_valid.mean(), 6)
)

# 500k sample again
SECOND_SAMPLE_SIZE = min(
    500_000,
    len(X_second_train)
)

rng_second = np.random.RandomState(
    RANDOM_STATE + 999
)

second_sample_idx = rng_second.choice(
    len(X_second_train),
    size=SECOND_SAMPLE_SIZE,
    replace=False
)

X_second_sample = X_second_train.iloc[second_sample_idx]
y_second_sample = y_second_train[second_sample_idx]

print("\nHGB training sample:", X_second_sample.shape)
print("Positive samples:", int(y_second_sample.sum()))

BUILDING SECOND-SPLIT DATA
Train: (1311258, 50)
Valid: (328166, 50)
Train anomaly rate: 0.008569
Valid anomaly rate: 0.008538

HGB training sample: (500000, 50)
Positive samples: 4306


In [51]:
# ============================================================
# CELL 51 — V3 ON SECOND VALIDATION SPLIT
# ============================================================

print("=" * 70)
print("TRAINING V3 ON INDEPENDENT SPLIT")
print("=" * 70)

hgb_v3_second = HistGradientBoostingClassifier(
    learning_rate=0.04,
    max_iter=650,
    max_leaf_nodes=127,
    max_depth=None,
    min_samples_leaf=35,
    l2_regularization=2.0,
    max_bins=255,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=40,
    random_state=RANDOM_STATE + 200,
    verbose=1
)

hgb_v3_second.fit(
    X_second_sample,
    y_second_sample
)

second_probability = (
    hgb_v3_second
    .predict_proba(X_second_valid)[:, 1]
)

second_best = find_best_threshold(
    y_second_valid,
    second_probability,
    low=0.10,
    high=0.90,
    step=0.005
)

print("\n" + "=" * 70)
print("SECOND-SPLIT V3 RESULT")
print("=" * 70)

print("Best F1   :", round(second_best["f1"], 6))
print("Threshold :", second_best["threshold"])
print("Precision :", round(second_best["precision"], 6))
print("Recall    :", round(second_best["recall"], 6))

print("\nOriginal V3 F1:",
      round(hgb_v3_best["f1"], 6))

print(
    "Difference:",
    round(
        second_best["f1"] - hgb_v3_best["f1"],
        6
    )
)

TRAINING V3 ON INDEPENDENT SPLIT
Binning 0.180 GB of training data: 1.097 s
Binning 0.020 GB of validation data: 0.065 s
Fitting gradient boosted rounds:
Fit 488 trees in 94.901 s, (61976 total leaves)
Time spent computing histograms: 69.953s
Time spent finding best splits:  4.057s
Time spent applying splits:      7.939s
Time spent predicting:           0.940s

SECOND-SPLIT V3 RESULT
Best F1   : 0.830203
Threshold : 0.41000000000000025
Precision : 0.886189
Recall    : 0.780871

Original V3 F1: 0.836033
Difference: -0.00583


In [52]:
# ============================================================
# CELL 52 — THRESHOLD STABILITY
# ============================================================

print("=" * 70)
print("THRESHOLD STABILITY")
print("=" * 70)

threshold_grid = np.arange(
    0.25,
    0.751,
    0.025
)

threshold_rows = []

for threshold in threshold_grid:

    original_pred = (
        ensemble_probability >= threshold
    ).astype(int)

    second_pred = (
        second_probability >= threshold
    ).astype(int)

    threshold_rows.append({
        "threshold": threshold,

        "original_F1": f1_score(
            y_hgb_valid,
            original_pred,
            zero_division=0
        ),

        "second_F1": f1_score(
            y_second_valid,
            second_pred,
            zero_division=0
        ),

        "original_precision": precision_score(
            y_hgb_valid,
            original_pred,
            zero_division=0
        ),

        "second_precision": precision_score(
            y_second_valid,
            second_pred,
            zero_division=0
        ),

        "original_recall": recall_score(
            y_hgb_valid,
            original_pred,
            zero_division=0
        ),

        "second_recall": recall_score(
            y_second_valid,
            second_pred,
            zero_division=0
        )
    })

threshold_stability = pd.DataFrame(
    threshold_rows
)

display(
    threshold_stability.style.format({
        "threshold": "{:.3f}",
        "original_F1": "{:.4f}",
        "second_F1": "{:.4f}",
        "original_precision": "{:.4f}",
        "second_precision": "{:.4f}",
        "original_recall": "{:.4f}",
        "second_recall": "{:.4f}"
    })
)

# Average rank across the two validation splits
threshold_stability["mean_F1"] = (
    threshold_stability["original_F1"] +
    threshold_stability["second_F1"]
) / 2

print("\nBest threshold by mean F1:")

display(
    threshold_stability
    .sort_values("mean_F1", ascending=False)
    .head(10)
)

THRESHOLD STABILITY


,threshold,original_F1,second_F1,original_precision,second_precision,original_recall,second_recall
0,0.250,0.8104,0.8246,0.7479,0.8332,0.8843,0.8162
1,0.275,0.8144,0.8247,0.7616,0.8409,0.8751,0.8091
2,0.300,0.8196,0.8252,0.7769,0.8498,0.8673,0.8019
3,0.325,0.8248,0.8254,0.7913,0.8565,0.8612,0.7966
4,0.350,0.8284,0.8278,0.8058,0.8644,0.8524,0.7941
5,0.375,0.8317,0.8288,0.8186,0.8732,0.8453,0.7887
6,0.400,0.8328,0.8297,0.8306,0.8832,0.8350,0.7823
7,0.425,0.8349,0.8292,0.8401,0.8909,0.8297,0.7755
8,0.450,0.8356,0.8288,0.8502,0.8966,0.8215,0.7705
9,0.475,0.8346,0.8265,0.8585,0.9021,0.8119,0.7627



Best threshold by mean F1:


,threshold,original_F1,second_F1,original_precision,second_precision,original_recall,second_recall,mean_F1
8,0.450,0.835589,0.828791,0.850165,0.896595,0.821505,0.770521,0.832190
7,0.425,0.834851,0.829231,0.840101,0.890939,0.829666,0.775517,0.832041
10,0.500,0.836859,0.826672,0.868652,0.911044,0.807310,0.756602,0.831765
11,0.525,0.837848,0.825490,0.877916,0.916014,0.801278,0.751249,0.831669
6,0.400,0.832773,0.829674,0.830568,0.883159,0.834989,0.782298,0.831224
9,0.475,0.834580,0.826533,0.858537,0.902068,0.811923,0.762670,0.830556
5,0.375,0.831704,0.828802,0.818557,0.873173,0.845280,0.788722,0.830253
12,0.550,0.836302,0.823785,0.886680,0.922566,0.791341,0.744111,0.830044
4,0.350,0.828419,0.827753,0.805770,0.864413,0.852378,0.794076,0.828086
13,0.575,0.834121,0.821258,0.893074,0.928443,0.782470,0.736260,0.827690


In [53]:
# ============================================================
# CELL 53 — FINAL VALIDATION DECISION
# ============================================================

print("=" * 70)
print("VALIDATION DECISION")
print("=" * 70)

best_mean_row = (
    threshold_stability
    .sort_values("mean_F1", ascending=False)
    .iloc[0]
)

print(
    "Current ensemble threshold:",
    CURRENT_THRESHOLD
)

print(
    "Best cross-split threshold:",
    best_mean_row["threshold"]
)

print(
    "Current ensemble F1:",
    round(CURRENT_BEST_F1, 6)
)

print(
    "Original split F1 at robust threshold:",
    round(
        best_mean_row["original_F1"],
        6
    )
)

print(
    "Second split F1 at robust threshold:",
    round(
        best_mean_row["second_F1"],
        6
    )
)

print(
    "Mean cross-split F1:",
    round(
        best_mean_row["mean_F1"],
        6
    )
)

threshold_difference = abs(
    float(best_mean_row["threshold"])
    - float(CURRENT_THRESHOLD)
)

print(
    "\nThreshold difference:",
    round(threshold_difference, 4)
)

if threshold_difference <= 0.05:
    print(
        "\n🔥 GOOD: threshold is reasonably stable."
    )
else:
    print(
        "\n⚠️ WARNING: threshold is unstable."
    )

if best_mean_row["mean_F1"] >= 0.83:
    print(
        "🔥 VERY STRONG: ensemble generalizes well."
    )
elif best_mean_row["mean_F1"] >= 0.80:
    print(
        "✓ Good: model remains competitive."
    )
else:
    print(
        "⚠️ Validation gap detected — be conservative."
    )

VALIDATION DECISION
Current ensemble threshold: 0.5200000000000002
Best cross-split threshold: 0.4500000000000002
Current ensemble F1: 0.838674
Original split F1 at robust threshold: 0.835589
Second split F1 at robust threshold: 0.828791
Mean cross-split F1: 0.83219

Threshold difference: 0.07

⚠️ WARNING: threshold is unstable.
🔥 VERY STRONG: ensemble generalizes well.


In [54]:
# ============================================================
# CELL 54 — COMPETITION-MATCHED DATE SPLIT
# ============================================================

print("=" * 70)
print("COMPETITION-MATCHED 80/20 DATE SPLIT")
print("=" * 70)

# Test appears to contain ~20% of each date.
# Reproduce that structure inside train.

date_sample_valid = (
    train_v2
    .groupby("Date", group_keys=False)
    .sample(frac=0.20, random_state=RANDOM_STATE + 555)
)

matched_valid_idx = date_sample_valid.index

matched_train_mask = ~train_v2.index.isin(matched_valid_idx)

matched_train_idx = train_v2.index[matched_train_mask]

print("Matched train rows :", len(matched_train_idx))
print("Matched valid rows :", len(matched_valid_idx))

print(
    "Validation fraction:",
    round(len(matched_valid_idx) / len(train_v2), 6)
)

print(
    "Train anomaly rate:",
    round(
        train_v2.loc[matched_train_idx, "target"]
        .astype(int)
        .mean(),
        6
    )
)

print(
    "Valid anomaly rate:",
    round(
        train_v2.loc[matched_valid_idx, "target"]
        .astype(int)
        .mean(),
        6
    )
)

# Check date coverage
print(
    "\nUnique train dates:",
    train_v2.loc[matched_train_idx, "Date"].nunique()
)

print(
    "Unique valid dates:",
    train_v2.loc[matched_valid_idx, "Date"].nunique()
)

COMPETITION-MATCHED 80/20 DATE SPLIT
Matched train rows : 1311557
Matched valid rows : 327867
Validation fraction: 0.199989
Train anomaly rate: 0.008568
Valid anomaly rate: 0.00854

Unique train dates: 1432
Unique valid dates: 1432


In [55]:
# ============================================================
# CELL 55 — PREPARE MATCHED DATA
# ============================================================

print("=" * 70)
print("PREPARING MATCHED DATA")
print("=" * 70)

X_match_train = train_v2.loc[
    matched_train_idx,
    hgb_features
]

X_match_valid = train_v2.loc[
    matched_valid_idx,
    hgb_features
]

y_match_train = (
    train_v2.loc[
        matched_train_idx,
        "target"
    ]
    .astype(int)
    .to_numpy()
)

y_match_valid = (
    train_v2.loc[
        matched_valid_idx,
        "target"
    ]
    .astype(int)
    .to_numpy()
)

print("Train:", X_match_train.shape)
print("Valid:", X_match_valid.shape)

print(
    "Train positives:",
    int(y_match_train.sum())
)

print(
    "Valid positives:",
    int(y_match_valid.sum())
)

# Same 500k training budget
MATCH_SAMPLE_SIZE = min(
    500_000,
    len(X_match_train)
)

rng_match = np.random.RandomState(
    RANDOM_STATE + 555
)

match_sample_idx = rng_match.choice(
    len(X_match_train),
    size=MATCH_SAMPLE_SIZE,
    replace=False
)

X_match_sample = X_match_train.iloc[
    match_sample_idx
]

y_match_sample = y_match_train[
    match_sample_idx
]

print(
    "\nHGB sample:",
    X_match_sample.shape
)

print(
    "Sample positives:",
    int(y_match_sample.sum())
)

PREPARING MATCHED DATA
Train: (1311557, 50)
Valid: (327867, 50)
Train positives: 11238
Valid positives: 2800

HGB sample: (500000, 50)
Sample positives: 4266


In [56]:
# ============================================================
# CELL 56 — V3 ON COMPETITION-MATCHED SPLIT
# ============================================================

print("=" * 70)
print("TRAINING V3 — COMPETITION-MATCHED SPLIT")
print("=" * 70)

hgb_v3_match = HistGradientBoostingClassifier(
    learning_rate=0.04,
    max_iter=650,
    max_leaf_nodes=127,
    max_depth=None,
    min_samples_leaf=35,
    l2_regularization=2.0,
    max_bins=255,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=40,
    random_state=RANDOM_STATE + 200,
    verbose=1
)

hgb_v3_match.fit(
    X_match_sample,
    y_match_sample
)

match_probability = (
    hgb_v3_match
    .predict_proba(X_match_valid)[:, 1]
)

match_best = find_best_threshold(
    y_match_valid,
    match_probability,
    low=0.10,
    high=0.90,
    step=0.005
)

print("\n" + "=" * 70)
print("MATCHED-SPLIT V3 RESULT")
print("=" * 70)

print(
    "Best F1     :",
    round(match_best["f1"], 6)
)

print(
    "Threshold   :",
    match_best["threshold"]
)

print(
    "Precision   :",
    round(match_best["precision"], 6)
)

print(
    "Recall      :",
    round(match_best["recall"], 6)
)

TRAINING V3 — COMPETITION-MATCHED SPLIT
Binning 0.180 GB of training data: 2.195 s
Binning 0.020 GB of validation data: 0.202 s
Fitting gradient boosted rounds:
Fit 491 trees in 107.374 s, (62357 total leaves)
Time spent computing histograms: 77.130s
Time spent finding best splits:  4.894s
Time spent applying splits:      9.070s
Time spent predicting:           1.081s

MATCHED-SPLIT V3 RESULT
Best F1     : 0.842027
Threshold   : 0.30500000000000016
Precision   : 0.873369
Recall      : 0.812857


In [57]:
# ============================================================
# CELL 57 — VALIDATION REGIME COMPARISON
# ============================================================

print("=" * 70)
print("VALIDATION REGIME COMPARISON")
print("=" * 70)

comparison = pd.DataFrame([
    [
        "Original group split",
        hgb_v3_best["f1"],
        hgb_v3_best["threshold"]
    ],
    [
        "Independent group split",
        second_best["f1"],
        second_best["threshold"]
    ],
    [
        "Date-matched Kaggle-like split",
        match_best["f1"],
        match_best["threshold"]
    ],
    [
        "3-HGB robust ensemble",
        best_mean_row["mean_F1"],
        best_mean_row["threshold"]
    ]
], columns=[
    "validation",
    "F1",
    "threshold"
])

display(
    comparison.sort_values(
        "F1",
        ascending=False
    ).style.format({
        "F1": "{:.6f}",
        "threshold": "{:.3f}"
    })
)

print("\nCurrent robust ensemble F1:",
      round(best_mean_row["mean_F1"], 6))

print("Matched V3 F1:",
      round(match_best["f1"], 6))

VALIDATION REGIME COMPARISON


,validation,F1,threshold
2,Date-matched Kaggle-like split,0.842027,0.305
0,Original group split,0.836033,0.450
3,3-HGB robust ensemble,0.832190,0.450
1,Independent group split,0.830203,0.410



Current robust ensemble F1: 0.83219
Matched V3 F1: 0.842027


In [58]:
# ============================================================
# CELL 58 — MATCHED SPLIT THRESHOLD STABILITY
# ============================================================

print("=" * 70)
print("THRESHOLD STABILITY — ALL SPLITS")
print("=" * 70)

thresholds_to_test = np.arange(
    0.30,
    0.651,
    0.025
)

rows = []

for threshold in thresholds_to_test:

    # Original ensemble
    original_pred = (
        ensemble_probability >= threshold
    ).astype(int)

    # Independent V3
    second_pred = (
        second_probability >= threshold
    ).astype(int)

    # Matched V3
    matched_pred = (
        match_probability >= threshold
    ).astype(int)

    rows.append({
        "threshold": threshold,

        "original_ensemble": f1_score(
            y_hgb_valid,
            original_pred,
            zero_division=0
        ),

        "independent_V3": f1_score(
            y_second_valid,
            second_pred,
            zero_division=0
        ),

        "matched_V3": f1_score(
            y_match_valid,
            matched_pred,
            zero_division=0
        )
    })

three_split_thresholds = pd.DataFrame(rows)

three_split_thresholds["mean_F1"] = (
    three_split_thresholds[
        [
            "original_ensemble",
            "independent_V3",
            "matched_V3"
        ]
    ].mean(axis=1)
)

display(
    three_split_thresholds
    .sort_values("mean_F1", ascending=False)
    .head(12)
)

THRESHOLD STABILITY — ALL SPLITS


,threshold,original_ensemble,independent_V3,matched_V3,mean_F1
4,0.400,0.832773,0.829674,0.837440,0.833296
5,0.425,0.834851,0.829231,0.835190,0.833091
3,0.375,0.831704,0.828802,0.838207,0.832904
6,0.450,0.835589,0.828791,0.834107,0.832829
7,0.475,0.834580,0.826533,0.833917,0.831676
2,0.350,0.828419,0.827753,0.838443,0.831538
8,0.500,0.836859,0.826672,0.830425,0.831319
1,0.325,0.824809,0.825444,0.841007,0.830420
9,0.525,0.837848,0.825490,0.827246,0.830195
0,0.300,0.819584,0.825193,0.841153,0.828643


In [59]:
# ============================================================
# CELL 59 — ROBUST THREE-SPLIT THRESHOLD
# ============================================================

print("=" * 70)
print("ROBUST THRESHOLD SELECTION")
print("=" * 70)

threshold_ranked = (
    three_split_thresholds
    .sort_values(
        "mean_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

best_robust = threshold_ranked.iloc[0]

print(
    "Best threshold:",
    best_robust["threshold"]
)

print(
    "Mean F1:",
    round(best_robust["mean_F1"], 6)
)

print(
    "Original ensemble F1:",
    round(best_robust["original_ensemble"], 6)
)

print(
    "Independent V3 F1:",
    round(best_robust["independent_V3"], 6)
)

print(
    "Matched V3 F1:",
    round(best_robust["matched_V3"], 6)
)

print("\nTop thresholds:")
display(
    threshold_ranked.head(10)
)

ROBUST THRESHOLD SELECTION
Best threshold: 0.4000000000000001
Mean F1: 0.833296
Original ensemble F1: 0.832773
Independent V3 F1: 0.829674
Matched V3 F1: 0.83744

Top thresholds:


,threshold,original_ensemble,independent_V3,matched_V3,mean_F1
0,0.400,0.832773,0.829674,0.837440,0.833296
1,0.425,0.834851,0.829231,0.835190,0.833091
2,0.375,0.831704,0.828802,0.838207,0.832904
3,0.450,0.835589,0.828791,0.834107,0.832829
4,0.475,0.834580,0.826533,0.833917,0.831676
5,0.350,0.828419,0.827753,0.838443,0.831538
6,0.500,0.836859,0.826672,0.830425,0.831319
7,0.325,0.824809,0.825444,0.841007,0.830420
8,0.525,0.837848,0.825490,0.827246,0.830195
9,0.300,0.819584,0.825193,0.841153,0.828643


In [60]:
# ============================================================
# CELL 60 — DECISION CHECKPOINT
# ============================================================

print("=" * 70)
print("COMPETITION MODEL DECISION")
print("=" * 70)

robust_threshold = float(
    best_robust["threshold"]
)

robust_mean_f1 = float(
    best_robust["mean_F1"]
)

matched_f1 = float(
    best_robust["matched_V3"]
)

print("ROBUST THRESHOLD :", robust_threshold)
print("3-SPLIT MEAN F1  :", round(robust_mean_f1, 6))
print("MATCHED F1       :", round(matched_f1, 6))

print("\nInterpretation:")

if matched_f1 > 0.84:
    print(
        "🔥🔥 VERY STRONG — Kaggle-like split is even better."
    )
elif matched_f1 > 0.83:
    print(
        "🔥 STRONG — V3 is highly competitive."
    )
elif matched_f1 > 0.80:
    print(
        "✓ GOOD — model is stable."
    )
else:
    print(
        "⚠️ Matched split is weaker — investigate before submission."
    )

if 0.40 <= robust_threshold <= 0.50:
    print(
        "\n✓ Threshold is in the stable 0.40–0.50 zone."
    )
else:
    print(
        "\n⚠️ Threshold still looks unstable."
    )

print(
    "\nDO NOT SUBMIT YET."
)

COMPETITION MODEL DECISION
ROBUST THRESHOLD : 0.4000000000000001
3-SPLIT MEAN F1  : 0.833296
MATCHED F1       : 0.83744

Interpretation:
🔥 STRONG — V3 is highly competitive.

✓ Threshold is in the stable 0.40–0.50 zone.

DO NOT SUBMIT YET.


In [61]:
# ============================================================
# CELL 61 — CHECK MATCHED-SPLIT GROUP LEAKAGE
# ============================================================

print("=" * 70)
print("CHECKING MATCHED-SPLIT FOR SENSOR+DATE LEAKAGE")
print("=" * 70)

# Use existing key if available
if "_sensor_date_key" in train_v2.columns:
    sensor_date_key = train_v2["_sensor_date_key"].astype(str)
else:
    sensor_date_key = (
        train_v2["X1"].astype(str) + "_" +
        train_v2["X2"].astype(str) + "_" +
        train_v2["X3"].astype(str) + "_" +
        train_v2["X4"].astype(str) + "_" +
        train_v2["X5"].astype(str) + "_" +
        train_v2["Date"].astype(str)
    )

train_keys_match = set(
    sensor_date_key.loc[matched_train_idx]
)

valid_keys_match = set(
    sensor_date_key.loc[matched_valid_idx]
)

overlap_keys = (
    train_keys_match &
    valid_keys_match
)

overlap_rows_valid = sensor_date_key.loc[
    matched_valid_idx
].isin(overlap_keys).sum()

print("Train unique groups :", len(train_keys_match))
print("Valid unique groups :", len(valid_keys_match))
print("Overlapping groups  :", len(overlap_keys))

print(
    "\nValidation rows affected by leakage:",
    int(overlap_rows_valid)
)

print(
    "Leakage row percentage:",
    round(
        overlap_rows_valid / len(matched_valid_idx) * 100,
        4
    ),
    "%"
)

if overlap_rows_valid == 0:
    print("\n✓ ZERO LEAKAGE")
else:
    print(
        "\n⚠️ Leakage detected — 0.842027 may be optimistic."
    )

CHECKING MATCHED-SPLIT FOR SENSOR+DATE LEAKAGE
Train unique groups : 1273960
Valid unique groups : 325344
Overlapping groups  : 17209

Validation rows affected by leakage: 17891
Leakage row percentage: 5.4568 %

⚠️ Leakage detected — 0.842027 may be optimistic.


In [62]:
# ============================================================
# CELL 62 — CLEAN DATE-STRATIFIED GROUP SPLIT
# ============================================================

print("=" * 70)
print("BUILDING CLEAN DATE-STRATIFIED GROUP SPLIT")
print("=" * 70)

# Every sensor+date group stays entirely in either train or validation.
group_df = pd.DataFrame({
    "group_key": sensor_date_key,
    "Date": train_v2["Date"].values
})

group_sizes = (
    group_df
    .groupby(
        ["Date", "group_key"],
        sort=False
    )
    .size()
    .reset_index(name="rows")
)

# Randomize groups independently within each date
group_sizes["random"] = np.random.RandomState(
    RANDOM_STATE + 777
).rand(len(group_sizes))

group_sizes = (
    group_sizes
    .sort_values(
        ["Date", "random"]
    )
)

# Target approximately 20% of each date
group_sizes["date_total"] = (
    group_sizes
    .groupby("Date")["rows"]
    .transform("sum")
)

group_sizes["target_valid_rows"] = (
    group_sizes["date_total"] * 0.20
)

group_sizes["cumulative_rows"] = (
    group_sizes
    .groupby("Date")["rows"]
    .cumsum()
)

# Select groups whose cumulative count has not crossed target
group_sizes["is_valid_group"] = (
    group_sizes["cumulative_rows"]
    <= group_sizes["target_valid_rows"]
)

valid_group_keys = set(
    group_sizes.loc[
        group_sizes["is_valid_group"],
        "group_key"
    ]
)

clean_valid_mask = sensor_date_key.isin(
    valid_group_keys
)

clean_valid_idx = train_v2.index[
    clean_valid_mask
]

clean_train_idx = train_v2.index[
    ~clean_valid_mask
]

print("Clean train rows :", len(clean_train_idx))
print("Clean valid rows :", len(clean_valid_idx))

print(
    "Validation fraction:",
    round(
        len(clean_valid_idx) / len(train_v2),
        6
    )
)

print(
    "Unique validation groups:",
    len(valid_group_keys)
)

BUILDING CLEAN DATE-STRATIFIED GROUP SPLIT
Clean train rows : 1312167
Clean valid rows : 327257
Validation fraction: 0.199617
Unique validation groups: 315831


In [63]:
# ============================================================
# CELL 63 — VERIFY CLEAN SPLIT
# ============================================================

print("=" * 70)
print("VERIFYING CLEAN SPLIT")
print("=" * 70)

clean_train_keys = set(
    sensor_date_key.loc[clean_train_idx]
)

clean_valid_keys = set(
    sensor_date_key.loc[clean_valid_idx]
)

clean_overlap = (
    clean_train_keys &
    clean_valid_keys
)

print(
    "Train/validation group overlap:",
    len(clean_overlap)
)

clean_train_dates = set(
    train_v2.loc[clean_train_idx, "Date"]
)

clean_valid_dates = set(
    train_v2.loc[clean_valid_idx, "Date"]
)

print(
    "Train dates:",
    len(clean_train_dates)
)

print(
    "Valid dates:",
    len(clean_valid_dates)
)

clean_date_rates = (
    train_v2
    .loc[
        clean_valid_idx,
        ["Date"]
    ]
    .groupby("Date")
    .size()
    /
    train_v2
    .groupby("Date")
    .size()
)

print(
    "\nPer-date validation fraction:"
)

display(
    clean_date_rates.describe()
)

print(
    "\nOverall validation anomaly rate:",
    round(
        train_v2.loc[
            clean_valid_idx,
            "target"
        ].astype(int).mean(),
        6
    )
)

if len(clean_overlap) == 0:
    print("\n🔥 CLEAN SPLIT CONFIRMED — ZERO GROUP LEAKAGE")
else:
    print("\n⚠️ Something is wrong with grouping.")

VERIFYING CLEAN SPLIT
Train/validation group overlap: 0
Train dates: 1432
Valid dates: 1432

Per-date validation fraction:


count    1432.000000
mean        0.199568
std         0.001333
min         0.157895
25%         0.199470
50%         0.199653
75%         0.199827
max         0.200000
dtype: float64


Overall validation anomaly rate: 0.008455

🔥 CLEAN SPLIT CONFIRMED — ZERO GROUP LEAKAGE


In [64]:
# ============================================================
# CELL 64 — PREPARE CLEAN HGB DATA
# ============================================================

print("=" * 70)
print("PREPARING CLEAN HGB DATA")
print("=" * 70)

X_clean_train = train_v2.loc[
    clean_train_idx,
    hgb_features
]

X_clean_valid = train_v2.loc[
    clean_valid_idx,
    hgb_features
]

y_clean_train = (
    train_v2.loc[
        clean_train_idx,
        "target"
    ]
    .astype(int)
    .to_numpy()
)

y_clean_valid = (
    train_v2.loc[
        clean_valid_idx,
        "target"
    ]
    .astype(int)
    .to_numpy()
)

print("Train:", X_clean_train.shape)
print("Valid:", X_clean_valid.shape)

print(
    "Train positives:",
    int(y_clean_train.sum())
)

print(
    "Valid positives:",
    int(y_clean_valid.sum())
)

# Same 500k budget for fair comparison
CLEAN_SAMPLE_SIZE = min(
    500_000,
    len(X_clean_train)
)

rng_clean = np.random.RandomState(
    RANDOM_STATE + 777
)

clean_sample_idx = rng_clean.choice(
    len(X_clean_train),
    size=CLEAN_SAMPLE_SIZE,
    replace=False
)

X_clean_sample = X_clean_train.iloc[
    clean_sample_idx
]

y_clean_sample = y_clean_train[
    clean_sample_idx
]

print(
    "\nTraining sample:",
    X_clean_sample.shape
)

print(
    "Sample positives:",
    int(y_clean_sample.sum())
)

PREPARING CLEAN HGB DATA
Train: (1312167, 50)
Valid: (327257, 50)
Train positives: 11271
Valid positives: 2767

Training sample: (500000, 50)
Sample positives: 4353


In [65]:
# ============================================================
# CELL 65 — V3 ON CLEAN DATE+GROUP SPLIT
# ============================================================

print("=" * 70)
print("TRAINING V3 — CLEAN DATE+GROUP SPLIT")
print("=" * 70)

hgb_v3_clean = HistGradientBoostingClassifier(
    learning_rate=0.04,
    max_iter=650,
    max_leaf_nodes=127,
    max_depth=None,
    min_samples_leaf=35,
    l2_regularization=2.0,
    max_bins=255,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=40,
    random_state=RANDOM_STATE + 200,
    verbose=1
)

hgb_v3_clean.fit(
    X_clean_sample,
    y_clean_sample
)

clean_probability = (
    hgb_v3_clean
    .predict_proba(X_clean_valid)[:, 1]
)

clean_best = find_best_threshold(
    y_clean_valid,
    clean_probability,
    low=0.10,
    high=0.90,
    step=0.005
)

print("\n" + "=" * 70)
print("CLEAN SPLIT RESULT")
print("=" * 70)

print(
    "Best F1     :",
    round(clean_best["f1"], 6)
)

print(
    "Threshold   :",
    clean_best["threshold"]
)

print(
    "Precision   :",
    round(clean_best["precision"], 6)
)

print(
    "Recall      :",
    round(clean_best["recall"], 6)
)

TRAINING V3 — CLEAN DATE+GROUP SPLIT
Binning 0.180 GB of training data: 0.997 s
Binning 0.020 GB of validation data: 0.061 s
Fitting gradient boosted rounds:
Fit 492 trees in 93.702 s, (62484 total leaves)
Time spent computing histograms: 70.843s
Time spent finding best splits:  3.678s
Time spent applying splits:      7.092s
Time spent predicting:           0.871s

CLEAN SPLIT RESULT
Best F1     : 0.826449
Threshold   : 0.4650000000000003
Precision   : 0.891632
Recall      : 0.770148


In [66]:
# ============================================================
# CELL 66 — FINAL VALIDATION REGIME COMPARISON
# ============================================================

print("=" * 70)
print("ALL VALIDATION REGIMES")
print("=" * 70)

all_validation_results = pd.DataFrame([
    [
        "Original group split",
        hgb_v3_best["f1"],
        hgb_v3_best["threshold"]
    ],
    [
        "Independent group split",
        second_best["f1"],
        second_best["threshold"]
    ],
    [
        "Random date-matched split",
        match_best["f1"],
        match_best["threshold"]
    ],
    [
        "Clean date + sensor-group split",
        clean_best["f1"],
        clean_best["threshold"]
    ],
    [
        "3-HGB robust ensemble",
        best_mean_row["mean_F1"],
        best_mean_row["threshold"]
    ]
], columns=[
    "validation",
    "F1",
    "threshold"
])

display(
    all_validation_results
    .sort_values(
        "F1",
        ascending=False
    )
    .style.format({
        "F1": "{:.6f}",
        "threshold": "{:.3f}"
    })
)

print(
    "\nClean split F1:",
    round(clean_best["f1"], 6)
)

print(
    "Random matched F1:",
    round(match_best["f1"], 6)
)

print(
    "Difference caused by possible leakage:",
    round(
        match_best["f1"] - clean_best["f1"],
        6
    )
)

ALL VALIDATION REGIMES


,validation,F1,threshold
2,Random date-matched split,0.842027,0.305
0,Original group split,0.836033,0.450
4,3-HGB robust ensemble,0.832190,0.450
1,Independent group split,0.830203,0.410
3,Clean date + sensor-group split,0.826449,0.465



Clean split F1: 0.826449
Random matched F1: 0.842027
Difference caused by possible leakage: 0.015578


In [67]:
# ============================================================
# CELL 67 — MODEL DECISION GATE
# ============================================================

print("=" * 70)
print("MODEL DECISION GATE")
print("=" * 70)

clean_f1 = float(clean_best["f1"])
clean_threshold = float(clean_best["threshold"])

print("Clean F1       :", round(clean_f1, 6))
print("Clean threshold:", clean_threshold)

print("\nCurrent previous leaderboard:")
print("Kaggle F1      : 0.779661")

print("\nDecision:")

if clean_f1 >= 0.83:

    print(
        "🔥🔥 EXCELLENT — HGB generalizes even without group leakage."
    )

    print(
        "Next: train full-data ensemble and generate a serious submission."
    )

elif clean_f1 >= 0.80:

    print(
        "🔥 STRONG — model remains substantially better than baseline."
    )

    print(
        "Next: one targeted improvement before final submission."
    )

elif clean_f1 >= 0.78:

    print(
        "✓ Competitive, but improvement is needed."
    )

    print(
        "Next: investigate weighting / feature engineering."
    )

else:

    print(
        "⚠️ The random matched result was substantially inflated."
    )

    print(
        "Do not submit yet."
    )

print("\nPrevious best validation:")
print("3-HGB ensemble:", round(best_mean_row["mean_F1"], 6))

print(
    "\nIf clean F1 >= 0.83, we are DONE experimenting with random "
    "model architectures."
)

MODEL DECISION GATE
Clean F1       : 0.826449
Clean threshold: 0.4650000000000003

Current previous leaderboard:
Kaggle F1      : 0.779661

Decision:
🔥 STRONG — model remains substantially better than baseline.
Next: one targeted improvement before final submission.

Previous best validation:
3-HGB ensemble: 0.83219

If clean F1 >= 0.83, we are DONE experimenting with random model architectures.


In [68]:
# ============================================================
# CELL 68 — POSITIVE-PRESERVING SAMPLE
# ============================================================

print("=" * 70)
print("BUILDING POSITIVE-PRESERVING HGB SAMPLE")
print("=" * 70)

# ALL positive examples from clean training partition
positive_mask_clean = y_clean_train == 1
negative_mask_clean = y_clean_train == 0

positive_indices_clean = np.where(
    positive_mask_clean
)[0]

negative_indices_clean = np.where(
    negative_mask_clean
)[0]

print("Available positives:", len(positive_indices_clean))
print("Available negatives:", len(negative_indices_clean))

# Sample 500k negatives
NEGATIVE_SAMPLE_SIZE = min(
    500_000,
    len(negative_indices_clean)
)

rng_pos = np.random.RandomState(
    RANDOM_STATE + 888
)

negative_sample_indices = rng_pos.choice(
    negative_indices_clean,
    size=NEGATIVE_SAMPLE_SIZE,
    replace=False
)

# Combine ALL positives + sampled negatives
pos_preserve_indices = np.concatenate([
    positive_indices_clean,
    negative_sample_indices
])

# Shuffle
rng_shuffle = np.random.RandomState(
    RANDOM_STATE + 889
)

rng_shuffle.shuffle(
    pos_preserve_indices
)

X_pos_preserve = X_clean_train.iloc[
    pos_preserve_indices
]

y_pos_preserve = y_clean_train[
    pos_preserve_indices
]

print("\nFinal sample:", X_pos_preserve.shape)

print(
    "Positives:",
    int(y_pos_preserve.sum())
)

print(
    "Negatives:",
    int((y_pos_preserve == 0).sum())
)

print(
    "Positive rate:",
    round(y_pos_preserve.mean(), 6)
)

BUILDING POSITIVE-PRESERVING HGB SAMPLE
Available positives: 11271
Available negatives: 1300896

Final sample: (511271, 50)
Positives: 11271
Negatives: 500000
Positive rate: 0.022045


In [69]:
# ============================================================
# CELL 69 — V4: ALL POSITIVES + 500K NEGATIVES
# ============================================================

print("=" * 70)
print("TRAINING V4 — POSITIVE-PRESERVING")
print("=" * 70)

hgb_v4_model = HistGradientBoostingClassifier(
    learning_rate=0.04,
    max_iter=650,
    max_leaf_nodes=127,
    max_depth=None,
    min_samples_leaf=35,
    l2_regularization=2.0,
    max_bins=255,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=40,
    random_state=RANDOM_STATE + 300,
    verbose=1
)

hgb_v4_model.fit(
    X_pos_preserve,
    y_pos_preserve
)

v4_clean_probability = (
    hgb_v4_model
    .predict_proba(X_clean_valid)[:, 1]
)

v4_best = find_best_threshold(
    y_clean_valid,
    v4_clean_probability,
    low=0.10,
    high=0.90,
    step=0.005
)

print("\n" + "=" * 70)
print("V4 RESULT")
print("=" * 70)

print(
    "Best F1     :",
    round(v4_best["f1"], 6)
)

print(
    "Threshold   :",
    v4_best["threshold"]
)

print(
    "Precision   :",
    round(v4_best["precision"], 6)
)

print(
    "Recall      :",
    round(v4_best["recall"], 6)
)

TRAINING V4 — POSITIVE-PRESERVING
Binning 0.184 GB of training data: 0.987 s
Binning 0.020 GB of validation data: 0.064 s
Fitting gradient boosted rounds:
Fit 650 trees in 107.746 s, (82550 total leaves)
Time spent computing histograms: 79.814s
Time spent finding best splits:  4.325s
Time spent applying splits:      8.700s
Time spent predicting:           1.061s

V4 RESULT
Best F1     : 0.862433
Threshold   : 0.6350000000000005
Precision   : 0.884207
Recall      : 0.841706


In [70]:
# ============================================================
# CELL 70 — V3 vs V4 COMPARISON
# ============================================================

print("=" * 70)
print("V3 vs V4")
print("=" * 70)

comparison_v4 = pd.DataFrame([
    [
        "V3 — random 500k",
        clean_best["f1"],
        clean_best["threshold"],
        clean_best["precision"],
        clean_best["recall"]
    ],
    [
        "V4 — all positives + 500k negatives",
        v4_best["f1"],
        v4_best["threshold"],
        v4_best["precision"],
        v4_best["recall"]
    ]
], columns=[
    "model",
    "F1",
    "threshold",
    "precision",
    "recall"
])

display(
    comparison_v4.sort_values(
        "F1",
        ascending=False
    ).style.format({
        "F1": "{:.6f}",
        "threshold": "{:.3f}",
        "precision": "{:.6f}",
        "recall": "{:.6f}"
    })
)

print(
    "\nV4 improvement:",
    round(
        v4_best["f1"] - clean_best["f1"],
        6
    )
)

V3 vs V4


,model,F1,threshold,precision,recall
1,V4 — all positives + 500k negatives,0.862433,0.635,0.884207,0.841706
0,V3 — random 500k,0.826449,0.465,0.891632,0.770148



V4 improvement: 0.035983


In [71]:
# ============================================================
# CELL 71 — V3 + V4 ENSEMBLE
# ============================================================

print("=" * 70)
print("V3 + V4 ENSEMBLE")
print("=" * 70)

v3_v4_corr = np.corrcoef(
    clean_probability,
    v4_clean_probability
)[0, 1]

print(
    "Probability correlation:",
    round(v3_v4_corr, 6)
)

best_v3_v4 = {
    "f1": -1
}

for v4_weight in np.arange(
    0.0,
    1.001,
    0.025
):

    v3_weight = 1.0 - v4_weight

    blend_probability = (
        v3_weight * clean_probability +
        v4_weight * v4_clean_probability
    )

    for threshold in np.arange(
        0.20,
        0.801,
        0.005
    ):

        pred = (
            blend_probability >= threshold
        ).astype(int)

        f1 = f1_score(
            y_clean_valid,
            pred,
            zero_division=0
        )

        if f1 > best_v3_v4["f1"]:

            best_v3_v4 = {
                "f1": f1,
                "v3_weight": v3_weight,
                "v4_weight": v4_weight,
                "threshold": threshold,
                "precision": precision_score(
                    y_clean_valid,
                    pred,
                    zero_division=0
                ),
                "recall": recall_score(
                    y_clean_valid,
                    pred,
                    zero_division=0
                )
            }

print("\nBEST V3 + V4:")
print(best_v3_v4)

print(
    "\nImprovement over V3:",
    round(
        best_v3_v4["f1"] - clean_best["f1"],
        6
    )
)

V3 + V4 ENSEMBLE
Probability correlation: 0.939465

BEST V3 + V4:
{'f1': 0.863003663003663, 'v3_weight': np.float64(0.04999999999999993), 'v4_weight': np.float64(0.9500000000000001), 'threshold': np.float64(0.5800000000000003), 'precision': 0.8748607500928333, 'recall': 0.8514636790748102}

Improvement over V3: 0.036554


In [72]:
# ============================================================
# CELL 72 — MODERATE CLASS-WEIGHT EXPERIMENT
# ============================================================

print("=" * 70)
print("MODERATE CLASS WEIGHT EXPERIMENT")
print("=" * 70)

class_weight_results = []

for positive_weight in [1.5, 2.0, 3.0, 4.0]:

    print(
        f"\nTraining positive class weight = {positive_weight}"
    )

    model = HistGradientBoostingClassifier(
        learning_rate=0.04,
        max_iter=550,
        max_leaf_nodes=127,
        max_depth=None,
        min_samples_leaf=35,
        l2_regularization=2.0,
        max_bins=255,
        class_weight={
            0: 1.0,
            1: positive_weight
        },
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=35,
        random_state=RANDOM_STATE + 400 + int(
            positive_weight * 10
        ),
        verbose=0
    )

    model.fit(
        X_clean_sample,
        y_clean_sample
    )

    probability = model.predict_proba(
        X_clean_valid
    )[:, 1]

    result = find_best_threshold(
        y_clean_valid,
        probability,
        low=0.10,
        high=0.90,
        step=0.005
    )

    class_weight_results.append({
        "positive_weight": positive_weight,
        "F1": result["f1"],
        "threshold": result["threshold"],
        "precision": result["precision"],
        "recall": result["recall"]
    })

class_weight_df = pd.DataFrame(
    class_weight_results
).sort_values(
    "F1",
    ascending=False
)

display(
    class_weight_df.style.format({
        "positive_weight": "{:.1f}",
        "F1": "{:.6f}",
        "threshold": "{:.3f}",
        "precision": "{:.6f}",
        "recall": "{:.6f}"
    })
)

print(
    "\nBaseline V3 F1:",
    round(clean_best["f1"], 6)
)

MODERATE CLASS WEIGHT EXPERIMENT

Training positive class weight = 1.5

Training positive class weight = 2.0

Training positive class weight = 3.0

Training positive class weight = 4.0


,positive_weight,F1,threshold,precision,recall
1,2.0,0.826145,0.530,0.870994,0.785688
0,1.5,0.824209,0.405,0.850173,0.799783
3,4.0,0.821519,0.670,0.887909,0.764366
2,3.0,0.818441,0.580,0.861422,0.779545



Baseline V3 F1: 0.826449


In [73]:
# ============================================================
# CELL 73 — CLASS-WEIGHT WINNER
# ============================================================

print("=" * 70)
print("CLASS-WEIGHT WINNER")
print("=" * 70)

best_cw_row = class_weight_df.iloc[0]

best_cw_weight = float(
    best_cw_row["positive_weight"]
)

best_cw_f1 = float(
    best_cw_row["F1"]
)

best_cw_threshold = float(
    best_cw_row["threshold"]
)

print(
    "Best positive weight:",
    best_cw_weight
)

print(
    "F1:",
    round(best_cw_f1, 6)
)

print(
    "Threshold:",
    best_cw_threshold
)

print(
    "Improvement over clean V3:",
    round(
        best_cw_f1 - clean_best["f1"],
        6
    )
)

print("\nFull results:")
display(class_weight_df)

CLASS-WEIGHT WINNER
Best positive weight: 2.0
F1: 0.826145
Threshold: 0.5300000000000004
Improvement over clean V3: -0.000305

Full results:


,positive_weight,F1,threshold,precision,recall
1,2.0,0.826145,0.530,0.870994,0.785688
0,1.5,0.824209,0.405,0.850173,0.799783
3,4.0,0.821519,0.670,0.887909,0.764366
2,3.0,0.818441,0.580,0.861422,0.779545


In [74]:
# ============================================================
# CELL 74 — EXPERIMENT SCOREBOARD
# ============================================================

print("=" * 70)
print("FINAL EXPERIMENT SCOREBOARD")
print("=" * 70)

scoreboard = [
    ["Original XGB + LGB", 0.800956],
    ["HGB V1", 0.815556],
    ["HGB V2", 0.826883],
    ["HGB V3", 0.836033],
    ["3-HGB ensemble", 0.832190],
    ["Clean V3", clean_best["f1"]],
    ["Positive-preserving V4", v4_best["f1"]],
    ["V3 + V4", best_v3_v4["f1"]],
    ["Best class-weight HGB", best_cw_f1]
]

scoreboard_df = pd.DataFrame(
    scoreboard,
    columns=["model", "F1"]
).sort_values(
    "F1",
    ascending=False
).reset_index(drop=True)

display(
    scoreboard_df.style.format({
        "F1": "{:.6f}"
    })
)

champion = scoreboard_df.iloc[0]

print(
    "\n🏆 CURRENT VALIDATION CHAMPION:",
    champion["model"]
)

print(
    "F1:",
    round(champion["F1"], 6)
)

print(
    "\nPrevious Kaggle:",
    0.779661
)

print(
    "Validation improvement over previous Kaggle:",
    round(
        champion["F1"] - 0.779661,
        6
    )
)

FINAL EXPERIMENT SCOREBOARD


,model,F1
0,V3 + V4,0.863004
1,Positive-preserving V4,0.862433
2,HGB V3,0.836033
3,3-HGB ensemble,0.832190
4,HGB V2,0.826883
5,Clean V3,0.826449
6,Best class-weight HGB,0.826145
7,HGB V1,0.815556
8,Original XGB + LGB,0.800956



🏆 CURRENT VALIDATION CHAMPION: V3 + V4
F1: 0.863004

Previous Kaggle: 0.779661
Validation improvement over previous Kaggle: 0.083343


In [75]:
# ============================================================
# CELL 75 — INDEPENDENT SPLIT: POSITIVE-PRESERVING SAMPLE
# ============================================================

print("=" * 70)
print("INDEPENDENT SPLIT — POSITIVE-PRESERVING SAMPLE")
print("=" * 70)

# y_second_train was created in Cell 50

positive_idx_second = np.where(
    y_second_train == 1
)[0]

negative_idx_second = np.where(
    y_second_train == 0
)[0]

print("Available positives:", len(positive_idx_second))
print("Available negatives:", len(negative_idx_second))

NEGATIVE_SAMPLE_SECOND = min(
    500_000,
    len(negative_idx_second)
)

rng_ind_v4 = np.random.RandomState(
    RANDOM_STATE + 999
)

negative_sample_second = rng_ind_v4.choice(
    negative_idx_second,
    size=NEGATIVE_SAMPLE_SECOND,
    replace=False
)

ind_v4_indices = np.concatenate([
    positive_idx_second,
    negative_sample_second
])

rng_ind_v4.shuffle(ind_v4_indices)

X_ind_v4_sample = X_second_train.iloc[
    ind_v4_indices
]

y_ind_v4_sample = y_second_train[
    ind_v4_indices
]

print("\nFinal sample:", X_ind_v4_sample.shape)
print("Positives:", int(y_ind_v4_sample.sum()))
print("Negatives:", int((y_ind_v4_sample == 0).sum()))

print(
    "Positive rate:",
    round(y_ind_v4_sample.mean(), 6)
)

INDEPENDENT SPLIT — POSITIVE-PRESERVING SAMPLE
Available positives: 11236
Available negatives: 1300022

Final sample: (511236, 50)
Positives: 11236
Negatives: 500000
Positive rate: 0.021978


In [76]:
# ============================================================
# CELL 76 — V4 ON INDEPENDENT SPLIT
# ============================================================

print("=" * 70)
print("TRAINING V4 — INDEPENDENT SPLIT")
print("=" * 70)

hgb_v4_second = HistGradientBoostingClassifier(
    learning_rate=0.04,
    max_iter=650,
    max_leaf_nodes=127,
    max_depth=None,
    min_samples_leaf=35,
    l2_regularization=2.0,
    max_bins=255,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=40,
    random_state=RANDOM_STATE + 300,
    verbose=1
)

hgb_v4_second.fit(
    X_ind_v4_sample,
    y_ind_v4_sample
)

v4_second_probability = (
    hgb_v4_second
    .predict_proba(X_second_valid)[:, 1]
)

v4_second_best = find_best_threshold(
    y_second_valid,
    v4_second_probability,
    low=0.20,
    high=0.90,
    step=0.005
)

print("\n" + "=" * 70)
print("INDEPENDENT V4 RESULT")
print("=" * 70)

print(
    "F1        :",
    round(v4_second_best["f1"], 6)
)

print(
    "Threshold :",
    v4_second_best["threshold"]
)

print(
    "Precision :",
    round(v4_second_best["precision"], 6)
)

print(
    "Recall    :",
    round(v4_second_best["recall"], 6)
)

print("\nClean-split V4:",
      round(v4_best["f1"], 6))

print(
    "Cross-split difference:",
    round(
        v4_second_best["f1"] - v4_best["f1"],
        6
    )
)

TRAINING V4 — INDEPENDENT SPLIT
Binning 0.184 GB of training data: 1.013 s
Binning 0.020 GB of validation data: 0.068 s
Fitting gradient boosted rounds:
Fit 650 trees in 111.244 s, (82550 total leaves)
Time spent computing histograms: 81.735s
Time spent finding best splits:  4.603s
Time spent applying splits:      9.352s
Time spent predicting:           1.115s

INDEPENDENT V4 RESULT
F1        : 0.857912
Threshold : 0.5450000000000004
Precision : 0.862554
Recall    : 0.853319

Clean-split V4: 0.862433
Cross-split difference: -0.004521


In [77]:
# ============================================================
# CELL 77 — INDEPENDENT V3 + V4 ENSEMBLE
# ============================================================

print("=" * 70)
print("INDEPENDENT V3 + V4 ENSEMBLE")
print("=" * 70)

v3_v4_ind_corr = np.corrcoef(
    second_probability,
    v4_second_probability
)[0, 1]

print(
    "Probability correlation:",
    round(v3_v4_ind_corr, 6)
)

best_ind_blend = {
    "f1": -1
}

for v4_weight in np.arange(
    0.0,
    1.001,
    0.025
):

    v3_weight = 1.0 - v4_weight

    blend_probability = (
        v3_weight * second_probability +
        v4_weight * v4_second_probability
    )

    for threshold in np.arange(
        0.20,
        0.801,
        0.005
    ):

        pred = (
            blend_probability >= threshold
        ).astype(int)

        f1 = f1_score(
            y_second_valid,
            pred,
            zero_division=0
        )

        if f1 > best_ind_blend["f1"]:

            best_ind_blend = {
                "f1": f1,
                "v3_weight": v3_weight,
                "v4_weight": v4_weight,
                "threshold": threshold,
                "precision": precision_score(
                    y_second_valid,
                    pred,
                    zero_division=0
                ),
                "recall": recall_score(
                    y_second_valid,
                    pred,
                    zero_division=0
                )
            }

print("\nBEST INDEPENDENT BLEND:")
print(best_ind_blend)

print(
    "\nImprovement over independent V4:",
    round(
        best_ind_blend["f1"]
        - v4_second_best["f1"],
        6
    )
)

INDEPENDENT V3 + V4 ENSEMBLE
Probability correlation: 0.94282

BEST INDEPENDENT BLEND:
{'f1': 0.8591065292096219, 'v3_weight': np.float64(0.09999999999999998), 'v4_weight': np.float64(0.9), 'threshold': np.float64(0.5450000000000004), 'precision': 0.8709204253758709, 'recall': 0.8476088508208423}

Improvement over independent V4: 0.001195


In [78]:
# ============================================================
# CELL 78 — V4 THRESHOLD STABILITY
# ============================================================

print("=" * 70)
print("V4 THRESHOLD STABILITY")
print("=" * 70)

v4_threshold_grid = np.arange(
    0.35,
    0.751,
    0.025
)

v4_stability_rows = []

for threshold in v4_threshold_grid:

    clean_pred = (
        v4_clean_probability >= threshold
    ).astype(int)

    independent_pred = (
        v4_second_probability >= threshold
    ).astype(int)

    v4_stability_rows.append({
        "threshold": threshold,

        "clean_F1": f1_score(
            y_clean_valid,
            clean_pred,
            zero_division=0
        ),

        "independent_F1": f1_score(
            y_second_valid,
            independent_pred,
            zero_division=0
        ),

        "clean_precision": precision_score(
            y_clean_valid,
            clean_pred,
            zero_division=0
        ),

        "independent_precision": precision_score(
            y_second_valid,
            independent_pred,
            zero_division=0
        ),

        "clean_recall": recall_score(
            y_clean_valid,
            clean_pred,
            zero_division=0
        ),

        "independent_recall": recall_score(
            y_second_valid,
            independent_pred,
            zero_division=0
        )
    })

v4_stability = pd.DataFrame(
    v4_stability_rows
)

v4_stability["mean_F1"] = (
    v4_stability[
        ["clean_F1", "independent_F1"]
    ].mean(axis=1)
)

display(
    v4_stability
    .sort_values(
        "mean_F1",
        ascending=False
    )
    .head(12)
)

V4 THRESHOLD STABILITY


,threshold,clean_F1,independent_F1,clean_precision,independent_precision,clean_recall,independent_recall,mean_F1
11,0.625,0.862311,0.857353,0.881177,0.884003,0.844236,0.832263,0.859832
12,0.650,0.862402,0.856455,0.888165,0.890258,0.838092,0.825125,0.859429
10,0.600,0.862012,0.856309,0.874349,0.875466,0.850018,0.837973,0.859161
9,0.575,0.861096,0.857143,0.867572,0.867642,0.854716,0.846895,0.859119
8,0.550,0.858588,0.857759,0.858123,0.863341,0.859053,0.852248,0.858173
13,0.675,0.860522,0.854474,0.895312,0.896511,0.828334,0.816203,0.857498
7,0.525,0.858117,0.856787,0.851852,0.854204,0.864474,0.859386,0.857452
15,0.725,0.858613,0.854373,0.908102,0.914158,0.814239,0.801927,0.856493
14,0.700,0.857845,0.853750,0.899326,0.904553,0.820022,0.808351,0.855798
6,0.500,0.855516,0.855277,0.842622,0.846020,0.868811,0.864739,0.855397


In [79]:
# ============================================================
# CELL 79 — WEIGHTED POSITIVE-PRESERVING HGB
# ============================================================

print("=" * 70)
print("WEIGHTED POSITIVE-PRESERVING HGB")
print("=" * 70)

# V4 sample contains:
# ALL positives
# 500k sampled negatives
#
# Give each sampled negative weight equal to:
# total available negatives / sampled negatives

negative_weight = (
    len(negative_indices_clean)
    / len(negative_sample_indices)
)

positive_weight = 1.0

v4_sample_weights = np.where(
    y_pos_preserve == 1,
    positive_weight,
    negative_weight
)

print(
    "Negative sample weight:",
    round(negative_weight, 4)
)

print(
    "Positive sample weight:",
    positive_weight
)

print(
    "Effective negative representation:",
    round(
        negative_weight * 500_000
    )
)

weighted_v4_model = HistGradientBoostingClassifier(
    learning_rate=0.04,
    max_iter=650,
    max_leaf_nodes=127,
    max_depth=None,
    min_samples_leaf=35,
    l2_regularization=2.0,
    max_bins=255,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=40,
    random_state=RANDOM_STATE + 500,
    verbose=1
)

weighted_v4_model.fit(
    X_pos_preserve,
    y_pos_preserve,
    sample_weight=v4_sample_weights
)

weighted_v4_probability = (
    weighted_v4_model
    .predict_proba(X_clean_valid)[:, 1]
)

weighted_v4_best = find_best_threshold(
    y_clean_valid,
    weighted_v4_probability,
    low=0.05,
    high=0.90,
    step=0.005
)

print("\n" + "=" * 70)
print("WEIGHTED V4 RESULT")
print("=" * 70)

print(
    "F1:",
    round(weighted_v4_best["f1"], 6)
)

print(
    "Threshold:",
    weighted_v4_best["threshold"]
)

print(
    "Precision:",
    round(weighted_v4_best["precision"], 6)
)

print(
    "Recall:",
    round(weighted_v4_best["recall"], 6)
)

print(
    "\nUnweighted V4:",
    round(v4_best["f1"], 6)
)

WEIGHTED POSITIVE-PRESERVING HGB
Negative sample weight: 2.6018
Positive sample weight: 1.0
Effective negative representation: 1300896
Binning 0.184 GB of training data: 1.035 s
Binning 0.020 GB of validation data: 0.070 s
Fitting gradient boosted rounds:
Fit 650 trees in 142.396 s, (82550 total leaves)
Time spent computing histograms: 105.104s
Time spent finding best splits:  6.507s
Time spent applying splits:      11.435s
Time spent predicting:           1.298s

WEIGHTED V4 RESULT
F1: 0.865995
Threshold: 0.4349999999999998
Precision: 0.87872
Recall: 0.853632

Unweighted V4: 0.862433


In [80]:
# ============================================================
# CELL 80 — WEIGHTED VS UNWEIGHTED
# ============================================================

print("=" * 70)
print("WEIGHTED VS UNWEIGHTED V4")
print("=" * 70)

weighted_corr = np.corrcoef(
    v4_clean_probability,
    weighted_v4_probability
)[0, 1]

weighted_disagreement = np.mean(
    (
        v4_clean_probability >= v4_best["threshold"]
    )
    !=
    (
        weighted_v4_probability >=
        weighted_v4_best["threshold"]
    )
)

weighted_results = pd.DataFrame([
    [
        "V3",
        clean_best["f1"],
        clean_best["threshold"],
        clean_best["precision"],
        clean_best["recall"]
    ],
    [
        "V4 unweighted",
        v4_best["f1"],
        v4_best["threshold"],
        v4_best["precision"],
        v4_best["recall"]
    ],
    [
        "V4 weighted",
        weighted_v4_best["f1"],
        weighted_v4_best["threshold"],
        weighted_v4_best["precision"],
        weighted_v4_best["recall"]
    ]
], columns=[
    "model",
    "F1",
    "threshold",
    "precision",
    "recall"
])

display(
    weighted_results
    .sort_values(
        "F1",
        ascending=False
    )
)

print(
    "\nV4 probability correlation:",
    round(weighted_corr, 6)
)

print(
    "Prediction disagreement:",
    round(weighted_disagreement, 6)
)

WEIGHTED VS UNWEIGHTED V4


,model,F1,threshold,precision,recall
2,V4 weighted,0.865995,0.435,0.878720,0.853632
1,V4 unweighted,0.862433,0.635,0.884207,0.841706
0,V3,0.826449,0.465,0.891632,0.770148



V4 probability correlation: 0.982378
Prediction disagreement: 0.000807


In [81]:
# ============================================================
# CELL 81 — FINAL V4 CHECKPOINT
# ============================================================

print("=" * 70)
print("FINAL V4 CHECKPOINT")
print("=" * 70)

v4_robust_row = (
    v4_stability
    .sort_values(
        "mean_F1",
        ascending=False
    )
    .iloc[0]
)

print(
    "Clean V4 F1:",
    round(v4_best["f1"], 6)
)

print(
    "Independent V4 F1:",
    round(v4_second_best["f1"], 6)
)

print(
    "Independent V3+V4 F1:",
    round(best_ind_blend["f1"], 6)
)

print(
    "\nBest robust V4 threshold:",
    v4_robust_row["threshold"]
)

print(
    "Mean V4 F1:",
    round(v4_robust_row["mean_F1"], 6)
)

print(
    "\nWeighted V4 F1:",
    round(weighted_v4_best["f1"], 6)
)

print("\n" + "=" * 70)

if v4_robust_row["mean_F1"] >= 0.84:
    print(
        "🔥🔥 V4 IS VALIDATED — VERY STRONG."
    )
    print(
        "We should move toward full-data final training."
    )

elif v4_robust_row["mean_F1"] >= 0.82:
    print(
        "🔥 V4 IS STRONG — likely our best final model."
    )

else:
    print(
        "⚠️ V4 needs more investigation."
    )

print("=" * 70)

FINAL V4 CHECKPOINT
Clean V4 F1: 0.862433
Independent V4 F1: 0.857912
Independent V3+V4 F1: 0.859107

Best robust V4 threshold: 0.6250000000000002
Mean V4 F1: 0.859832

Weighted V4 F1: 0.865995

🔥🔥 V4 IS VALIDATED — VERY STRONG.
We should move toward full-data final training.


In [82]:
# ============================================================
# CELL 82 — LOCK FINAL STRATEGY
# ============================================================

print("=" * 70)
print("FINAL MODEL STRATEGY")
print("=" * 70)

FINAL_V4_NEGATIVES = 500_000
FINAL_V4_MODELS = 3

# Robust threshold from two independent validation splits
FINAL_V4_THRESHOLD = 0.625

# V3/V4 blend was only a small improvement.
# We'll use V4 as the main model and V3 as a small stabilizer.
FINAL_V4_WEIGHT = 0.85
FINAL_V3_WEIGHT = 0.15

print("V4 models              :", FINAL_V4_MODELS)
print("Negatives per V4       :", FINAL_V4_NEGATIVES)
print("All positives          : YES")
print("V4 threshold           :", FINAL_V4_THRESHOLD)
print("V4 ensemble weight     :", FINAL_V4_WEIGHT)
print("V3 stabilizer weight   :", FINAL_V3_WEIGHT)

print("\nValidation evidence:")
print("V4 clean F1            :", round(v4_best["f1"], 6))
print("V4 independent F1      :", round(v4_second_best["f1"], 6))
print("V4 robust mean F1      :", round(v4_robust_row["mean_F1"], 6))
print("V3+V4 independent F1   :", round(best_ind_blend["f1"], 6))

print("\n🔥 FINAL BUILD STARTING")

FINAL MODEL STRATEGY
V4 models              : 3
Negatives per V4       : 500000
All positives          : YES
V4 threshold           : 0.625
V4 ensemble weight     : 0.85
V3 stabilizer weight   : 0.15

Validation evidence:
V4 clean F1            : 0.862433
V4 independent F1      : 0.857912
V4 robust mean F1      : 0.859832
V3+V4 independent F1   : 0.859107

🔥 FINAL BUILD STARTING


In [83]:
# ============================================================
# CELL 83 — FULL DATA POSITIVE-PRESERVING POOL
# ============================================================

print("=" * 70)
print("FULL TRAINING DATA — POSITIVE PRESERVING")
print("=" * 70)

X_full = train_v2[hgb_features]

y_full = (
    train_v2["target"]
    .astype(int)
    .to_numpy()
)

positive_full_idx = np.where(
    y_full == 1
)[0]

negative_full_idx = np.where(
    y_full == 0
)[0]

print("Total rows:", len(y_full))
print("Total positives:", len(positive_full_idx))
print("Total negatives:", len(negative_full_idx))

print(
    "Overall anomaly rate:",
    round(y_full.mean(), 6)
)

print(
    "\nEvery positive example will be used."
)

FULL TRAINING DATA — POSITIVE PRESERVING
Total rows: 1639424
Total positives: 14038
Total negatives: 1625386
Overall anomaly rate: 0.008563

Every positive example will be used.


In [84]:
# ============================================================
# CELL 84 — THREE INDEPENDENT V4 SAMPLES
# ============================================================

print("=" * 70)
print("BUILDING THREE V4 TRAINING SAMPLES")
print("=" * 70)

final_samples = []

for model_number in range(FINAL_V4_MODELS):

    rng = np.random.RandomState(
        RANDOM_STATE + 1000 + model_number
    )

    sampled_negatives = rng.choice(
        negative_full_idx,
        size=FINAL_V4_NEGATIVES,
        replace=False
    )

    sample_idx = np.concatenate([
        positive_full_idx,
        sampled_negatives
    ])

    rng.shuffle(sample_idx)

    final_samples.append(sample_idx)

    sample_positive_rate = y_full[sample_idx].mean()

    print(
        f"Model {model_number + 1}: "
        f"{len(sample_idx):,} rows | "
        f"{int(y_full[sample_idx].sum()):,} positives | "
        f"positive rate={sample_positive_rate:.4%}"
    )

print("\n✓ Three independent negative samples created.")

BUILDING THREE V4 TRAINING SAMPLES
Model 1: 514,038 rows | 14,038 positives | positive rate=2.7309%
Model 2: 514,038 rows | 14,038 positives | positive rate=2.7309%
Model 3: 514,038 rows | 14,038 positives | positive rate=2.7309%

✓ Three independent negative samples created.


In [85]:
# ============================================================
# CELL 85 — TRAIN FINAL V4 BAG
# ============================================================

print("=" * 70)
print("TRAINING FINAL V4 BAG")
print("=" * 70)

final_v4_models = []

for model_number, sample_idx in enumerate(final_samples):

    print("\n" + "-" * 70)
    print(f"TRAINING FINAL V4 MODEL {model_number + 1}/{FINAL_V4_MODELS}")
    print("-" * 70)

    model = HistGradientBoostingClassifier(
        learning_rate=0.04,
        max_iter=650,
        max_leaf_nodes=127,
        max_depth=None,
        min_samples_leaf=35,
        l2_regularization=2.0,
        max_bins=255,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=40,
        random_state=RANDOM_STATE + 300 + model_number,
        verbose=1
    )

    model.fit(
        X_full.iloc[sample_idx],
        y_full[sample_idx]
    )

    final_v4_models.append(model)

    print(
        f"✓ Final V4 model {model_number + 1} trained"
    )

print("\n🔥 ALL FINAL V4 MODELS TRAINED")

TRAINING FINAL V4 BAG

----------------------------------------------------------------------
TRAINING FINAL V4 MODEL 1/3
----------------------------------------------------------------------
Binning 0.185 GB of training data: 1.009 s
Binning 0.021 GB of validation data: 0.072 s
Fitting gradient boosted rounds:
Fit 650 trees in 130.067 s, (82550 total leaves)
Time spent computing histograms: 94.794s
Time spent finding best splits:  6.646s
Time spent applying splits:      10.978s
Time spent predicting:           1.331s
✓ Final V4 model 1 trained

----------------------------------------------------------------------
TRAINING FINAL V4 MODEL 2/3
----------------------------------------------------------------------
Binning 0.185 GB of training data: 1.038 s
Binning 0.021 GB of validation data: 0.064 s
Fitting gradient boosted rounds:
Fit 650 trees in 127.728 s, (82550 total leaves)
Time spent computing histograms: 93.006s
Time spent finding best splits:  5.894s
Time spent applying splits

In [88]:
# ============================================================
# CELL 86A — INSPECT FINAL HGB FEATURES
# ============================================================

print("=" * 70)
print("EXACT HGB FEATURE LIST")
print("=" * 70)

for i, f in enumerate(hgb_features):
    print(f"{i:02d}: {f}")

print("\nTotal:", len(hgb_features))

EXACT HGB FEATURE LIST
00: X1
01: X2
02: X3
03: X4
04: X5
05: year
06: month
07: day
08: day_of_week
09: day_of_year
10: quarter
11: month_sin
12: month_cos
13: day_of_week_sin
14: day_of_week_cos
15: day_of_year_sin
16: day_of_year_cos
17: X1_log_abs
18: X1_log10_abs
19: X1_is_zero
20: X1_is_one
21: X1_large
22: X1_extreme
23: X2_log_abs
24: X2_log10_abs
25: X2_is_zero
26: X2_is_one
27: X2_large
28: X2_extreme
29: X3_log_abs
30: X3_log10_abs
31: X3_is_zero
32: X3_is_one
33: X3_large
34: X3_extreme
35: X4_log_abs
36: X4_log10_abs
37: X4_is_zero
38: X4_is_one
39: X4_large
40: X4_extreme
41: X5_log_abs
42: X5_log10_abs
43: X5_is_zero
44: X5_is_one
45: X5_large
46: X5_extreme
47: _original_row
48: _row_in_date
49: date_target_rate_loo

Total: 50


In [89]:
# ============================================================
# CELL 86B — INSPECT MISSING TRAIN FEATURES
# ============================================================

missing_features = [
    "_original_row",
    "_row_in_date",
    "date_target_rate_loo"
]

print("=" * 70)
print("MISSING FEATURE INSPECTION")
print("=" * 70)

for col in missing_features:
    print(f"\n--- {col} ---")
    
    if col in train_v2.columns:
        s = train_v2[col]
        print("dtype:", s.dtype)
        print("nulls:", s.isna().sum())
        print("unique:", s.nunique())
        print("min:", s.min())
        print("max:", s.max())
        print("mean:", s.mean())
        print("median:", s.median())
        print("head:")
        print(s.head().to_list())
    else:
        print("NOT PRESENT IN train_v2")

MISSING FEATURE INSPECTION

--- _original_row ---
dtype: int32
nulls: 0
unique: 1639424
min: 0
max: 1639423
mean: 819711.5
median: 819711.5
head:
[0, 1, 2, 3, 4]

--- _row_in_date ---
dtype: int64
nulls: 0
unique: 2289
min: 0
max: 2288
mean: 575.3631677955184
median: 574.0
head:
[0, 1, 2, 3, 4]

--- date_target_rate_loo ---
dtype: float64
nulls: 0
unique: 1181
min: 0.0
max: 0.12784398699891658
mean: 0.008558477558706153
median: 0.0021482277121374865
head:
[0.03734439834024896, 0.03333333333333333, 0.03333333333333333, 0.03333333333333333, 0.0375]


In [90]:
# ============================================================
# CELL 86C — TEST DATE STRUCTURE
# ============================================================

print("=" * 70)
print("TEST DATE STRUCTURE")
print("=" * 70)

print("Train rows:", len(train_v2))
print("Test rows :", len(test_v2))

print("\nUnique train dates:", train_v2["Date"].nunique())
print("Unique test dates :", test_v2["Date"].nunique())

print("\nTest rows per date:")
print(
    test_v2.groupby("Date")
    .size()
    .describe()
)

print("\nFirst dates:")
print(test_v2["Date"].head(10).to_list())

TEST DATE STRUCTURE
Train rows: 1639424
Test rows : 409856

Unique train dates: 1432
Unique test dates : 1432

Test rows per date:
count    1432.000000
mean      286.212291
std        26.299908
min         2.000000
25%       277.000000
50%       288.000000
75%       298.000000
max       591.000000
dtype: float64

First dates:
[Timestamp('2020-12-16 00:00:00'), Timestamp('2020-12-16 00:00:00'), Timestamp('2020-12-16 00:00:00'), Timestamp('2020-12-16 00:00:00'), Timestamp('2020-12-16 00:00:00'), Timestamp('2020-12-16 00:00:00'), Timestamp('2020-12-16 00:00:00'), Timestamp('2020-12-16 00:00:00'), Timestamp('2020-12-16 00:00:00'), Timestamp('2020-12-16 00:00:00')]


In [91]:
# ============================================================
# CELL 86D — ROW INDEX RECONSTRUCTION CHECK
# ============================================================

print("=" * 70)
print("ROW INDEX CHECK")
print("=" * 70)

for df_name, df in [
    ("train_v2", train_v2),
    ("test_v2", test_v2)
]:
    print(f"\n{df_name}")
    
    print("index head:", df.index[:10].to_list())
    print("index tail:", df.index[-10:].to_list())
    
    if "_original_row" in df.columns:
        print("_original_row head:",
              df["_original_row"].head(10).to_list())
        print("_original_row tail:",
              df["_original_row"].tail(10).to_list())

ROW INDEX CHECK

train_v2
index head: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
index tail: [1639414, 1639415, 1639416, 1639417, 1639418, 1639419, 1639420, 1639421, 1639422, 1639423]
_original_row head: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
_original_row tail: [1639414, 1639415, 1639416, 1639417, 1639418, 1639419, 1639420, 1639421, 1639422, 1639423]

test_v2
index head: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
index tail: [409846, 409847, 409848, 409849, 409850, 409851, 409852, 409853, 409854, 409855]


In [92]:
# ============================================================
# CELL 86E — ROW-IN-DATE PATTERN
# ============================================================

print("=" * 70)
print("ROW-IN-DATE CHECK")
print("=" * 70)

if "_row_in_date" in train_v2.columns:

    sample_dates = train_v2["Date"].drop_duplicates().head(5)

    for d in sample_dates:

        temp = train_v2[train_v2["Date"] == d]

        print("\nDate:", d)
        print("Rows:", len(temp))

        print(
            temp[
                ["Date", "_original_row", "_row_in_date"]
            ].head(15).to_string(index=False)
        )

else:
    print("_row_in_date not available in train_v2")

ROW-IN-DATE CHECK

Date: 2020-12-16 00:00:00
Rows: 302
      Date  _original_row  _row_in_date
2020-12-16              0             0
2020-12-16              1             1
2020-12-16              2             2
2020-12-16              3             3
2020-12-16              4             4
2020-12-16              5             5
2020-12-16              6             6
2020-12-16              7             7
2020-12-16              8             8
2020-12-16              9             9
2020-12-16             10            10
2020-12-16             11            11
2020-12-16             12            12
2020-12-16             13            13
2020-12-16             14            14

Date: 2020-12-17 00:00:00
Rows: 1125
      Date  _original_row  _row_in_date
2020-12-17            302             0
2020-12-17            303             1
2020-12-17            304             2
2020-12-17            305             3
2020-12-17            306             4
2020-12-17            307  

In [95]:
# ============================================================
# CELL 86F — FIXED DATE TARGET RATE LOO ANALYSIS
# ============================================================

print("=" * 70)
print("DATE TARGET RATE LOO ANALYSIS — FIXED")
print("=" * 70)

# Convert target safely
target_numeric = pd.to_numeric(
    train_v2["target"],
    errors="raise"
).astype(int)

temp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "target": target_numeric.values,
    "date_target_rate_loo": train_v2["date_target_rate_loo"].values
})

date_stats = (
    temp.groupby("Date")
    .agg(
        rows=("target", "size"),
        positives=("target", "sum"),
        actual_rate=("target", "mean"),
        loo_mean=("date_target_rate_loo", "mean"),
        loo_min=("date_target_rate_loo", "min"),
        loo_max=("date_target_rate_loo", "max")
    )
    .reset_index()
)

print("\nDate-level examples:")
print(date_stats.head(10).to_string(index=False))

print("\nTarget dtype:", target_numeric.dtype)

print("\nCorrelation:")
print(
    temp[["target", "date_target_rate_loo"]]
    .corr()
)

DATE TARGET RATE LOO ANALYSIS — FIXED

Date-level examples:
      Date  rows  positives  actual_rate  loo_mean  loo_min  loo_max
2020-12-16   302         11     0.036424  0.037344 0.033333 0.037500
2020-12-17  1125         22     0.019556  0.021111 0.020022 0.021135
2020-12-18  1162          2     0.001721  0.002153 0.001078 0.002155
2020-12-19  1174         21     0.017888  0.019169 0.018124 0.019190
2020-12-20  1140         11     0.009649  0.010965 0.009879 0.010977
2020-12-21  1156          5     0.004325  0.003247 0.002167 0.003250
2020-12-22  1162         22     0.018933  0.018299 0.017241 0.018319
2020-12-23  1153         96     0.083261  0.082430 0.081433 0.082519
2020-12-24  1159         32     0.027610  0.025890 0.024838 0.025918
2020-12-25  1125         46     0.040889  0.040000 0.038932 0.040044

Target dtype: int64

Correlation:
                        target  date_target_rate_loo
target                1.000000              0.159813
date_target_rate_loo  0.159813          

In [97]:
# ============================================================
# CELL 86G — VERIFY EXACT LOO FORMULA — FINAL FIX
# ============================================================

print("=" * 70)
print("REVERSE ENGINEERING LOO FORMULA")
print("=" * 70)

# Convert target ONCE to numeric
target_numeric = pd.to_numeric(
    train_v2["target"],
    errors="raise"
).astype(np.int64)

# Date-wise positive count
date_positive_sum = (
    pd.DataFrame({
        "Date": train_v2["Date"].values,
        "_target_num": target_numeric.values
    })
    .groupby("Date")["_target_num"]
    .transform("sum")
)

# Date-wise row count
date_count = (
    train_v2.groupby("Date")["target"]
    .transform("count")
)

# ------------------------------------------------------------
# Exact leave-one-out formula
#
# Remove the CURRENT ROW'S target from its date's
# positive count, then divide by remaining rows.
# ------------------------------------------------------------

expected_loo = (
    date_positive_sum - target_numeric
) / (
    date_count - 1
)

actual_loo = train_v2[
    "date_target_rate_loo"
].astype(float)

diff = (
    expected_loo - actual_loo
).abs()

print("\nRESULT")
print("-" * 70)

print(f"Max absolute difference  : {diff.max():.15f}")
print(f"Mean absolute difference : {diff.mean():.15f}")
print(f"Median difference        : {diff.median():.15f}")
print(f"Exact matches            : {(diff == 0).sum():,}")
print(f"Total rows               : {len(diff):,}")

print("\nSample comparison:")

check = pd.DataFrame({
    "target": target_numeric.head(20).to_numpy(),
    "actual_loo": actual_loo.head(20).to_numpy(),
    "expected_loo": expected_loo.head(20).to_numpy(),
    "difference": diff.head(20).to_numpy()
})

print(check.to_string(index=False))

print("\n" + "=" * 70)

if diff.max() < 1e-12:
    print("✅ EXACT MATCH — LOO FORMULA CONFIRMED")
else:
    print("⚠️ FORMULA DOES NOT EXACTLY MATCH")
    
print("=" * 70)

REVERSE ENGINEERING LOO FORMULA

RESULT
----------------------------------------------------------------------
Max absolute difference  : 0.005154270169661
Mean absolute difference : 0.000437930960712
Median difference        : 0.000218340611354
Exact matches            : 623,910
Total rows               : 1,639,424

Sample comparison:
 target  actual_loo  expected_loo  difference
      0    0.037344      0.036545    0.000800
      1    0.033333      0.033223    0.000111
      1    0.033333      0.033223    0.000111
      1    0.033333      0.033223    0.000111
      0    0.037500      0.036545    0.000955
      0    0.037344      0.036545    0.000800
      0    0.037500      0.036545    0.000955
      0    0.037500      0.036545    0.000955
      0    0.037500      0.036545    0.000955
      0    0.037500      0.036545    0.000955
      1    0.037344      0.033223    0.004122
      1    0.033333      0.033223    0.000111
      0    0.037500      0.036545    0.000955
      0    0.03750

In [98]:
# ============================================================
# CELL 86H — REVERSE ENGINEER DATE TARGET FEATURE
# ============================================================

print("=" * 70)
print("DATE TARGET FEATURE — DEEP REVERSE ENGINEERING")
print("=" * 70)

target_numeric = pd.to_numeric(
    train_v2["target"],
    errors="raise"
).astype(int)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "target": target_numeric.values,
    "feature": train_v2["date_target_rate_loo"].astype(float).values
})

date_info = (
    tmp.groupby("Date")
    .agg(
        n=("target", "size"),
        positives=("target", "sum"),
        actual_rate=("target", "mean"),
        feature_mean=("feature", "mean"),
        feature_min=("feature", "min"),
        feature_max=("feature", "max"),
        feature_unique=("feature", "nunique")
    )
    .reset_index()
)

# How much does the feature vary inside a date?
date_info["feature_range"] = (
    date_info["feature_max"] -
    date_info["feature_min"]
)

print("\nFirst 20 dates:")
print(
    date_info.head(20).to_string(index=False)
)

print("\nFeature variation within dates:")
print(
    date_info["feature_range"].describe()
)

print("\nDates where feature is constant:")
print(
    (date_info["feature_range"] == 0).sum(),
    "of",
    len(date_info)
)

print("\nDates where feature varies:")
print(
    (date_info["feature_range"] > 0).sum(),
    "of",
    len(date_info)
)

DATE TARGET FEATURE — DEEP REVERSE ENGINEERING

First 20 dates:
      Date    n  positives  actual_rate  feature_mean  feature_min  feature_max  feature_unique  feature_range
2020-12-16  302         11     0.036424      0.037344     0.033333     0.037500               3       0.004167
2020-12-17 1125         22     0.019556      0.021111     0.020022     0.021135               3       0.001112
2020-12-18 1162          2     0.001721      0.002153     0.001078     0.002155               3       0.001078
2020-12-19 1174         21     0.017888      0.019169     0.018124     0.019190               3       0.001066
2020-12-20 1140         11     0.009649      0.010965     0.009879     0.010977               3       0.001098
2020-12-21 1156          5     0.004325      0.003247     0.002167     0.003250               3       0.001083
2020-12-22 1162         22     0.018933      0.018299     0.017241     0.018319               3       0.001078
2020-12-23 1153         96     0.083261      0.0

In [99]:
# ============================================================
# CELL 86I — CHECK DATE + TARGET DEPENDENCY
# ============================================================

print("=" * 70)
print("DATE + TARGET DEPENDENCY")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "target": pd.to_numeric(
        train_v2["target"],
        errors="raise"
    ).astype(int).values,
    "feature": train_v2["date_target_rate_loo"].astype(float).values
})

# Number of unique feature values for each Date + target
check = (
    tmp.groupby(["Date", "target"])["feature"]
    .agg(
        unique="nunique",
        minimum="min",
        maximum="max"
    )
    .reset_index()
)

check["range"] = check["maximum"] - check["minimum"]

print("\nDate + target groups:", len(check))

print("\nGroups with multiple feature values:")
print(
    (check["unique"] > 1).sum()
)

print("\nGroups with constant feature:")
print(
    (check["unique"] == 1).sum()
)

print("\nMaximum within-group range:")
print(check["range"].max())

print("\nExamples:")
print(
    check.head(20).to_string(index=False)
)

DATE + TARGET DEPENDENCY

Date + target groups: 2320

Groups with multiple feature values:
1509

Groups with constant feature:
811

Maximum within-group range:
0.005319148936170213

Examples:
      Date  target  unique  minimum  maximum    range
2020-12-16       0       2 0.037344 0.037500 0.000156
2020-12-16       1       2 0.033333 0.037344 0.004011
2020-12-17       0       2 0.021111 0.021135 0.000023
2020-12-17       1       2 0.020022 0.021111 0.001089
2020-12-18       0       2 0.002153 0.002155 0.000002
2020-12-18       1       1 0.001078 0.001078 0.000000
2020-12-19       0       2 0.019169 0.019190 0.000020
2020-12-19       1       2 0.018124 0.019169 0.001046
2020-12-20       0       2 0.010965 0.010977 0.000012
2020-12-20       1       2 0.009879 0.010965 0.001086
2020-12-21       0       2 0.003247 0.003250 0.000004
2020-12-21       1       2 0.002167 0.003247 0.001080
2020-12-22       0       2 0.018299 0.018319 0.000020
2020-12-22       1       2 0.017241 0.018299 0.00105

In [100]:
# ============================================================
# CELL 86J — EXACT FIRST DATE INVESTIGATION
# ============================================================

print("=" * 70)
print("EXACT FIRST DATE INVESTIGATION")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "target": pd.to_numeric(
        train_v2["target"],
        errors="raise"
    ).astype(int).values,
    "feature": train_v2["date_target_rate_loo"].astype(float).values,
    "row": train_v2["_original_row"].values,
    "row_in_date": train_v2["_row_in_date"].values
})

first_date = tmp["Date"].iloc[0]

d = tmp[tmp["Date"] == first_date].copy()

print("Date:", first_date)
print("Rows:", len(d))
print("Positive:", d["target"].sum())
print("Actual rate:", d["target"].mean())

print("\nFeature values by target:")
print(
    d.groupby("target")["feature"]
    .agg(["count", "min", "max", "mean", "nunique"])
)

print("\nFirst 40 rows:")
print(
    d.head(40).to_string(index=False)
)

EXACT FIRST DATE INVESTIGATION
Date: 2020-12-16 00:00:00
Rows: 302
Positive: 11
Actual rate: 0.03642384105960265

Feature values by target:
        count       min       max      mean  nunique
target                                              
0         291  0.037344  0.037500  0.037468        2
1          11  0.033333  0.037344  0.034063        2

First 40 rows:
      Date  target  feature  row  row_in_date
2020-12-16       0 0.037344    0            0
2020-12-16       1 0.033333    1            1
2020-12-16       1 0.033333    2            2
2020-12-16       1 0.033333    3            3
2020-12-16       0 0.037500    4            4
2020-12-16       0 0.037344    5            5
2020-12-16       0 0.037500    6            6
2020-12-16       0 0.037500    7            7
2020-12-16       0 0.037500    8            8
2020-12-16       0 0.037500    9            9
2020-12-16       1 0.037344   10           10
2020-12-16       1 0.033333   11           11
2020-12-16       0 0.037500   12  

In [101]:
# ============================================================
# CELL 86K — TEST COMMON TARGET ENCODING FORMULAS
# ============================================================

print("=" * 70)
print("TESTING COMMON FORMULAS")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "y": pd.to_numeric(
        train_v2["target"],
        errors="raise"
    ).astype(float).values,
    "actual": train_v2["date_target_rate_loo"].astype(float).values
})

g = tmp.groupby("Date")["y"]

S = g.transform("sum")
N = g.transform("count")
Y = tmp["y"]

global_rate = tmp["y"].mean()

candidates = {}

# Simple LOO
candidates["simple_loo"] = (S - Y) / (N - 1)

# Date rate
candidates["date_rate"] = S / N

# Add-one smoothing
candidates["add1"] = (S - Y + 1) / N

# Laplace smoothing
candidates["laplace"] = (S - Y + 1) / (N + 2)

# Jeffreys-style smoothing
candidates["jeffreys"] = (S - Y + 0.5) / (N - 1 + 1)

# Global-rate smoothing, alpha=1
candidates["global_alpha1"] = (
    (S - Y) + global_rate
) / N

# Global-rate smoothing, alpha=5
candidates["global_alpha5"] = (
    (S - Y) + 5 * global_rate
) / (N - 1 + 5)

# Global-rate smoothing, alpha=10
candidates["global_alpha10"] = (
    (S - Y) + 10 * global_rate
) / (N - 1 + 10)

results = []

actual = tmp["actual"].to_numpy()

for name, pred in candidates.items():

    arr = np.asarray(pred, dtype=float)

    diff = np.abs(arr - actual)

    results.append({
        "formula": name,
        "MAE": diff.mean(),
        "median_abs_error": np.median(diff),
        "max_abs_error": diff.max(),
        "exact": np.sum(diff < 1e-12)
    })

formula_results = (
    pd.DataFrame(results)
    .sort_values("MAE")
)

print(
    formula_results.to_string(index=False)
)

TESTING COMMON FORMULAS
       formula      MAE  median_abs_error  max_abs_error  exact
    simple_loo 0.000438          0.000218       0.005154 623910
     date_rate 0.000441          0.000219       0.008511 623795
 global_alpha1 0.000441          0.000214       0.005089      0
 global_alpha5 0.000453          0.000197       0.005177      0
global_alpha10 0.000469          0.000187       0.005341      0
      jeffreys 0.000640          0.000436       0.026316   5446
       laplace 0.000968          0.000864       0.047619      0
          add1 0.000977          0.000865       0.052632   1631


In [102]:
# ============================================================
# CELL 86L — CHECK ROW-POSITION DEPENDENCY
# ============================================================

print("=" * 70)
print("ROW POSITION DEPENDENCY")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "target": pd.to_numeric(
        train_v2["target"],
        errors="raise"
    ).astype(int).values,
    "row_in_date": train_v2["_row_in_date"].values,
    "feature": train_v2["date_target_rate_loo"].astype(float).values
})

# Correlation with row position
print(
    "\nCorrelation(feature, row_in_date):",
    tmp["feature"].corr(tmp["row_in_date"])
)

# First 10 vs last 10 positions
tmp["position_bucket"] = pd.cut(
    tmp["row_in_date"],
    bins=10,
    labels=False
)

bucket_stats = (
    tmp.groupby("position_bucket", observed=True)["feature"]
    .agg(["mean", "median", "std", "min", "max"])
)

print("\nFeature by position bucket:")
print(
    bucket_stats.to_string()
)

print("\nFeature correlation with target:")
print(
    tmp["feature"].corr(tmp["target"])
)

ROW POSITION DEPENDENCY

Correlation(feature, row_in_date): -0.0011666836709024692

Feature by position bucket:
                     mean    median       std  min       max
position_bucket                                             
0                0.008567  0.002148  0.015212  0.0  0.127844
1                0.008570  0.002148  0.015218  0.0  0.127844
2                0.008551  0.002148  0.015210  0.0  0.127844
3                0.008555  0.002148  0.015209  0.0  0.127844
4                0.008568  0.002148  0.015222  0.0  0.127844
5                0.008670  0.001088  0.015344  0.0  0.127844
6                0.000000  0.000000  0.000000  0.0  0.000000
7                0.000000  0.000000  0.000000  0.0  0.000000
8                0.000000  0.000000  0.000000  0.0  0.000000
9                0.000000  0.000000  0.000000  0.0  0.000000

Feature correlation with target:
0.15981267448264014


In [103]:
# ============================================================
# CELL 86M — CHECK SHIFTED DATE INFORMATION
# ============================================================

print("=" * 70)
print("CHECK SHIFTED / NEIGHBOR DATE RELATIONSHIPS")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "target": pd.to_numeric(
        train_v2["target"],
        errors="raise"
    ).astype(float).values,
    "feature": train_v2["date_target_rate_loo"].astype(float).values
})

daily = (
    tmp.groupby("Date")
    .agg(
        target_rate=("target", "mean"),
        feature_mean=("feature", "mean")
    )
    .sort_index()
)

# Previous / next day target rates
daily["prev_rate"] = daily["target_rate"].shift(1)
daily["next_rate"] = daily["target_rate"].shift(-1)

daily["prev2_rate"] = daily["target_rate"].shift(2)
daily["next2_rate"] = daily["target_rate"].shift(-2)

print("\nCorrelations:")

for col in [
    "target_rate",
    "prev_rate",
    "next_rate",
    "prev2_rate",
    "next2_rate"
]:
    print(
        f"feature vs {col:15s}: "
        f"{daily['feature_mean'].corr(daily[col]):.6f}"
    )

print("\nDaily sample:")
print(
    daily.head(15).to_string()
)

CHECK SHIFTED / NEIGHBOR DATE RELATIONSHIPS

Correlations:
feature vs target_rate    : 0.998727
feature vs prev_rate      : 0.219424
feature vs next_rate      : 0.218558
feature vs prev2_rate     : 0.110149
feature vs next2_rate     : 0.107947

Daily sample:
            target_rate  feature_mean  prev_rate  next_rate  prev2_rate  next2_rate
Date                                                                               
2020-12-16     0.036424      0.037344        NaN   0.019556         NaN    0.001721
2020-12-17     0.019556      0.021111   0.036424   0.001721         NaN    0.017888
2020-12-18     0.001721      0.002153   0.019556   0.017888    0.036424    0.009649
2020-12-19     0.017888      0.019169   0.001721   0.009649    0.019556    0.004325
2020-12-20     0.009649      0.010965   0.017888   0.004325    0.001721    0.018933
2020-12-21     0.004325      0.003247   0.009649   0.018933    0.017888    0.083261
2020-12-22     0.018933      0.018299   0.004325   0.083261    0.0096

In [104]:
# ============================================================
# CELL 86N — FEATURE VALUE FORENSICS
# ============================================================

print("=" * 70)
print("FEATURE VALUE FORENSICS")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "target": pd.to_numeric(
        train_v2["target"],
        errors="raise"
    ).astype(int).values,
    "feature": train_v2["date_target_rate_loo"].astype(float).values
})

daily = (
    tmp.groupby("Date")
    .agg(
        N=("target", "size"),
        P=("target", "sum"),
        rate=("target", "mean"),
        feature_min=("feature", "min"),
        feature_max=("feature", "max"),
        feature_mean=("feature", "mean")
    )
    .reset_index()
)

# Compare feature to simple rational possibilities
daily["P_div_N"] = daily["P"] / daily["N"]
daily["P_minus1_div_N_minus1"] = (
    (daily["P"] - 1) /
    (daily["N"] - 1)
)

daily["P_div_N_minus1"] = (
    daily["P"] /
    (daily["N"] - 1)
)

daily["P_plus1_div_N_plus1"] = (
    (daily["P"] + 1) /
    (daily["N"] + 1)
)

daily["P_plus1_div_N_plus2"] = (
    (daily["P"] + 1) /
    (daily["N"] + 2)
)

print("\nExamples with largest feature deviation from date rate:")

daily["rate_diff"] = (
    daily["feature_mean"] -
    daily["rate"]
).abs()

print(
    daily.sort_values(
        "rate_diff",
        ascending=False
    ).head(20).to_string(index=False)
)

print("\nFeature unique values:", tmp["feature"].nunique())
print("Date count:", daily["Date"].nunique())

print("\nMost common feature values:")
print(
    tmp["feature"]
    .value_counts()
    .head(20)
)

FEATURE VALUE FORENSICS

Examples with largest feature deviation from date rate:
      Date    N  P     rate  feature_min  feature_max  feature_mean  P_div_N  P_minus1_div_N_minus1  P_div_N_minus1  P_plus1_div_N_plus1  P_plus1_div_N_plus2  rate_diff
2023-01-16 1146 97 0.084642     0.078689     0.079781      0.079694 0.084642               0.083843        0.084716             0.085440             0.085366   0.004948
2022-05-25 1167 33 0.028278     0.022532     0.023605      0.023580 0.028278               0.027444        0.028302             0.029110             0.029085   0.004698
2022-04-05 1149 20 0.017406     0.020697     0.021786      0.021763 0.017406               0.016551        0.017422             0.018261             0.018245   0.004356
2021-12-27 1143 54 0.047244     0.050383     0.051479      0.051422 0.047244               0.046410        0.047285             0.048077             0.048035   0.004178
2021-01-14 1155 49 0.042424     0.045504     0.046587      0.046537 0.0424

In [105]:
# ============================================================
# CELL 86O — NEIGHBOR / ROLLING BEHAVIOR
# ============================================================

print("=" * 70)
print("NEIGHBOR / ROLLING BEHAVIOR")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "target": pd.to_numeric(train_v2["target"], errors="raise").astype(int).values,
    "feature": train_v2["date_target_rate_loo"].astype(float).values,
    "row_in_date": train_v2["_row_in_date"].values
})

# Inspect the first date carefully
d = tmp[tmp["Date"] == tmp["Date"].iloc[0]].copy()

print("Date:", d["Date"].iloc[0])
print("Rows:", len(d))

# Show feature changes
d["feature_change"] = d["feature"].diff()

print("\nFirst 100 rows where feature changes:")
print(
    d.loc[
        d["feature_change"].fillna(0) != 0,
        ["row_in_date", "target", "feature", "feature_change"]
    ].head(100).to_string(index=False)
)

print("\nNumber of feature changes:", d["feature_change"].ne(0).sum())

print("\nUnique feature values:")
print(
    d["feature"].value_counts().sort_index().to_string()
)

NEIGHBOR / ROLLING BEHAVIOR
Date: 2020-12-16 00:00:00
Rows: 302

First 100 rows where feature changes:
 row_in_date  target  feature  feature_change
           1       1 0.033333       -0.004011
           4       0 0.037500        0.004167
           5       0 0.037344       -0.000156
           6       0 0.037500        0.000156
          10       1 0.037344       -0.000156
          11       1 0.033333       -0.004011
          12       0 0.037500        0.004167
          15       0 0.037344       -0.000156
          16       1 0.033333       -0.004011
          18       0 0.037500        0.004167
          20       0 0.037344       -0.000156
          21       0 0.037500        0.000156
          25       0 0.037344       -0.000156
          26       0 0.037500        0.000156
          30       0 0.037344       -0.000156
          31       0 0.037500        0.000156
          33       1 0.033333       -0.004167
          35       1 0.037344        0.004011
          36       0 0.

In [106]:
# ============================================================
# CELL 86P — TEST ROLLING WINDOWS
# ============================================================

print("=" * 70)
print("TESTING ROLLING-WINDOW FORMULAS")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "target": pd.to_numeric(train_v2["target"], errors="raise").astype(float).values,
    "actual": train_v2["date_target_rate_loo"].astype(float).values
})

# Keep original row order
tmp["_row"] = np.arange(len(tmp))

windows = [10, 20, 30, 50, 80, 100, 150, 200, 300, 500]

results = []

for w in windows:

    # Previous w rows within each date, excluding current row
    rolling_prev = (
        tmp.groupby("Date", sort=False)["target"]
        .transform(
            lambda x: x.shift(1).rolling(
                window=w,
                min_periods=1
            ).mean()
        )
    )

    diff = (
        rolling_prev.fillna(0).to_numpy() -
        tmp["actual"].to_numpy()
    )

    results.append({
        "window": w,
        "MAE": np.abs(diff).mean(),
        "median_abs": np.median(np.abs(diff)),
        "max_abs": np.abs(diff).max()
    })

print(
    pd.DataFrame(results)
    .sort_values("MAE")
    .to_string(index=False)
)

TESTING ROLLING-WINDOW FORMULAS
 window      MAE  median_abs  max_abs
    500 0.007735    0.001802 0.995652
    300 0.009341    0.002162 0.995652
    200 0.010828    0.002160 0.995652
    150 0.011744    0.002157 0.995652
    100 0.012783    0.002155 0.995652
     80 0.013241    0.002155 0.995652
     50 0.014062    0.002151 0.995652
     30 0.014802    0.002148 0.995652
     20 0.015252    0.002148 0.995652
     10 0.015787    0.002148 0.995652


In [107]:
# ============================================================
# CELL 86Q — CENTERED / FORWARD WINDOWS
# ============================================================

print("=" * 70)
print("CENTERED / FORWARD WINDOW TEST")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "target": pd.to_numeric(train_v2["target"], errors="raise").astype(float).values,
    "actual": train_v2["date_target_rate_loo"].astype(float).values
})

windows = [10, 20, 30, 50, 80, 100, 150, 200]

results = []

for w in windows:

    # Centered rolling
    centered = (
        tmp.groupby("Date", sort=False)["target"]
        .transform(
            lambda x: x.rolling(
                window=w,
                min_periods=1,
                center=True
            ).mean()
        )
    )

    # Forward-looking rolling
    forward = (
        tmp.iloc[::-1]
        .groupby("Date", sort=False)["target"]
        .transform(
            lambda x: x.shift(1).rolling(
                window=w,
                min_periods=1
            ).mean()
        )
        .iloc[::-1]
    )

    centered_diff = (
        centered.to_numpy() -
        tmp["actual"].to_numpy()
    )

    forward_diff = (
        forward.to_numpy() -
        tmp["actual"].to_numpy()
    )

    results.append({
        "window": w,
        "centered_MAE": np.nanmean(np.abs(centered_diff)),
        "forward_MAE": np.nanmean(np.abs(forward_diff))
    })

print(
    pd.DataFrame(results).to_string(index=False)
)

CENTERED / FORWARD WINDOW TEST
 window  centered_MAE  forward_MAE
     10      0.015791     0.015784
     20      0.015255     0.015245
     30      0.014802     0.014792
     50      0.014055     0.014058
     80      0.013228     0.013256
    100      0.012774     0.012823
    150      0.011819     0.011928
    200      0.011092     0.011278


In [108]:
# ============================================================
# CELL 86R — ROLLING CORRELATION SEARCH
# ============================================================

print("=" * 70)
print("ROLLING CORRELATION SEARCH")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "target": pd.to_numeric(train_v2["target"], errors="raise").astype(float).values,
    "actual": train_v2["date_target_rate_loo"].astype(float).values
})

windows = [
    5, 10, 15, 20, 25, 30, 40, 50,
    60, 75, 80, 100, 125, 150, 200,
    250, 300, 400, 500, 750, 1000
]

results = []

for w in windows:

    prev = (
        tmp.groupby("Date", sort=False)["target"]
        .transform(
            lambda x: x.shift(1).rolling(
                w,
                min_periods=1
            ).mean()
        )
    )

    corr = (
        tmp["actual"]
        .corr(prev)
    )

    mae = np.nanmean(
        np.abs(
            tmp["actual"] -
            prev
        )
    )

    results.append({
        "window": w,
        "correlation": corr,
        "MAE": mae
    })

res = pd.DataFrame(results)

print(
    res.sort_values(
        "correlation",
        ascending=False
    ).head(15).to_string(index=False)
)

ROLLING CORRELATION SEARCH
 window  correlation      MAE
    400     0.471936 0.008387
    300     0.469221 0.009342
    500     0.466588 0.007734
    250     0.462097 0.010022
    200     0.447885 0.010830
    750     0.436134 0.006770
    150     0.424530 0.011747
   1000     0.412424 0.006380
    125     0.407367 0.012248
    100     0.385119 0.012787
     80     0.363001 0.013246
     75     0.356700 0.013367
     60     0.335524 0.013763
     50     0.318811 0.014067
     40     0.299554 0.014413


In [109]:
# ============================================================
# CELL 86S — RATIONAL DENOMINATOR ANALYSIS
# ============================================================

print("=" * 70)
print("RATIONAL DENOMINATOR ANALYSIS")
print("=" * 70)

from fractions import Fraction

actual = (
    train_v2["date_target_rate_loo"]
    .astype(float)
)

unique_values = (
    actual
    .value_counts()
    .head(100)
    .index
)

print("\nTop feature values and rational approximations:")

for value in unique_values[:50]:

    frac = Fraction(
        float(value)
    ).limit_denominator(5000)

    print(
        f"{value:.12f}  ->  "
        f"{frac.numerator}/{frac.denominator}  "
        f"(count={int((actual == value).sum()):,})"
    )

RATIONAL DENOMINATOR ANALYSIS

Top feature values and rational approximations:
0.000000000000  ->  0/1  (count=655,347)
0.001097694841  ->  1/911  (count=8,431)
0.001090512541  ->  1/917  (count=7,117)
0.001088139282  ->  1/919  (count=6,672)
0.002148227712  ->  2/931  (count=6,291)
0.001089324619  ->  1/918  (count=6,208)
0.002185792350  ->  2/915  (count=6,177)
0.001086956522  ->  1/920  (count=5,987)
0.001084598698  ->  1/922  (count=5,773)
0.001085776330  ->  1/921  (count=5,760)
0.001094091904  ->  1/914  (count=5,721)
0.001079913607  ->  1/926  (count=4,866)
0.003236245955  ->  1/309  (count=4,857)
0.001092896175  ->  1/915  (count=4,817)
0.002155172414  ->  1/464  (count=4,647)
0.002166847237  ->  2/923  (count=4,627)
0.003250270856  ->  3/923  (count=4,605)
0.002176278564  ->  2/919  (count=4,601)
0.002178649237  ->  1/459  (count=4,594)
0.005434782609  ->  1/184  (count=4,586)
0.001095290252  ->  1/913  (count=4,567)
0.003260869565  ->  3/920  (count=4,368)
0.002157497303  -> 

In [110]:
# ============================================================
# CELL 86T — POSITION + TARGET INTERACTION
# ============================================================

print("=" * 70)
print("POSITION + TARGET INTERACTION")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "target": pd.to_numeric(
        train_v2["target"],
        errors="raise"
    ).astype(int).values,
    "row_in_date": train_v2["_row_in_date"].values,
    "feature": train_v2["date_target_rate_loo"].astype(float).values
})

# Position bins based on actual quantiles
tmp["pos_bin"] = pd.qcut(
    tmp["row_in_date"],
    q=20,
    duplicates="drop"
)

summary = (
    tmp.groupby(
        ["pos_bin", "target"],
        observed=True
    )["feature"]
    .agg(
        count="size",
        mean="mean",
        median="median",
        min="min",
        max="max",
        unique="nunique"
    )
    .reset_index()
)

print(
    summary.to_string(index=False)
)

POSITION + TARGET INTERACTION
         pos_bin  target  count     mean   median      min      max  unique
  (-0.001, 57.0]       0  82736 0.008467 0.002144 0.000000 0.127844     938
  (-0.001, 57.0]       1    246 0.038756 0.038251 0.000000 0.081897      45
   (57.0, 114.0]       0  81432 0.008534 0.002148 0.000000 0.127844     938
   (57.0, 114.0]       1     78 0.038898 0.047220 0.000000 0.081897      19
  (114.0, 171.0]       0  81452 0.008569 0.002148 0.000000 0.127844     938
  (114.0, 171.0]       1      1 0.046537 0.046537 0.046537 0.046537       1
  (171.0, 229.0]       0  82797 0.008579 0.002148 0.000000 0.127844     938
  (229.0, 286.0]       0  81214 0.008586 0.002148 0.000000 0.127844     938
  (229.0, 286.0]       1     16 0.025240 0.027027 0.001098 0.037611       6
  (286.0, 344.0]       0  82051 0.008540 0.002144 0.000000 0.127844     937
  (286.0, 344.0]       1    556 0.012875 0.005593 0.000000 0.081897     248
  (344.0, 402.0]       0  82207 0.008491 0.002146 0.000000

In [111]:
# ============================================================
# CELL 86U — HARD CUTOFF / RESET ANALYSIS
# ============================================================

print("=" * 70)
print("HARD CUTOFF / RESET ANALYSIS")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "target": pd.to_numeric(
        train_v2["target"],
        errors="raise"
    ).astype(int).values,
    "row_in_date": train_v2["_row_in_date"].values,
    "feature": train_v2["date_target_rate_loo"].astype(float).values
})

# For every row position, calculate fraction of rows
# whose feature is exactly zero.
position_stats = (
    tmp.groupby("row_in_date")["feature"]
    .agg(
        count="size",
        mean="mean",
        zero_rate=lambda x: (x == 0).mean(),
        unique="nunique"
    )
    .reset_index()
)

print("\nPositions with highest zero rate:")

print(
    position_stats
    .sort_values(
        "zero_rate",
        ascending=False
    )
    .head(50)
    .to_string(index=False)
)

print("\nFirst position where zero rate exceeds 50%:")

high_zero = position_stats[
    position_stats["zero_rate"] > 0.5
]

if len(high_zero):
    print(
        high_zero.iloc[0].to_string()
    )
else:
    print("No position exceeds 50% zero rate.")

print("\nLast 20 positions:")
print(
    position_stats.tail(20).to_string(index=False)
)

HARD CUTOFF / RESET ANALYSIS

Positions with highest zero rate:
 row_in_date  count  mean  zero_rate  unique
        1198      1   0.0        1.0       1
        1199      1   0.0        1.0       1
        1200      1   0.0        1.0       1
        1201      1   0.0        1.0       1
        2242      1   0.0        1.0       1
        2243      1   0.0        1.0       1
        2244      1   0.0        1.0       1
        2245      1   0.0        1.0       1
        2246      1   0.0        1.0       1
        2247      1   0.0        1.0       1
        2248      1   0.0        1.0       1
        2249      1   0.0        1.0       1
        2250      1   0.0        1.0       1
        2251      1   0.0        1.0       1
        2252      1   0.0        1.0       1
        2253      1   0.0        1.0       1
        2254      1   0.0        1.0       1
        2255      1   0.0        1.0       1
        2256      1   0.0        1.0       1
        2257      1   0.0        1.0

In [112]:
# ============================================================
# CELL 86V — EXPANDING TARGET RATE
# ============================================================

print("=" * 70)
print("EXPANDING TARGET RATE TEST")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "target": pd.to_numeric(train_v2["target"], errors="raise").astype(float).values,
    "actual": train_v2["date_target_rate_loo"].astype(float).values
})

# Previous observations only
exp_prev = (
    tmp.groupby("Date", sort=False)["target"]
    .transform(
        lambda x: x.shift(1).expanding(min_periods=1).mean()
    )
)

# Previous observations including current
exp_current = (
    tmp.groupby("Date", sort=False)["target"]
    .transform(
        lambda x: x.expanding(min_periods=1).mean()
    )
)

for name, pred in [
    ("expanding_previous", exp_prev),
    ("expanding_current", exp_current)
]:
    diff = np.abs(
        pred.to_numpy() -
        tmp["actual"].to_numpy()
    )

    print(
        f"\n{name}"
        f"\n  MAE       : {np.nanmean(diff):.8f}"
        f"\n  Median    : {np.nanmedian(diff):.8f}"
        f"\n  Max       : {np.nanmax(diff):.8f}"
        f"\n  <1e-6     : {(diff < 1e-6).sum():,}"
    )

EXPANDING TARGET RATE TEST

expanding_previous
  MAE       : 0.00631216
  Median    : 0.00106496
  Max       : 0.99565217
  <1e-6     : 637,277

expanding_current
  MAE       : 0.00630812
  Median    : 0.00106496
  Max       : 0.99565689
  <1e-6     : 637,720


In [113]:
# ============================================================
# CELL 86W — CUMULATIVE COUNT FORMULAS
# ============================================================

print("=" * 70)
print("CUMULATIVE POSITIVE COUNT TEST")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "y": pd.to_numeric(train_v2["target"], errors="raise").astype(float).values,
    "actual": train_v2["date_target_rate_loo"].astype(float).values
})

g = tmp.groupby("Date", sort=False)

cum_pos = g["y"].cumsum()
cum_n = g.cumcount() + 1

candidates = {
    "previous_rate": (cum_pos - tmp["y"]) / (cum_n - 1),
    "current_rate": cum_pos / cum_n,
    "previous_plus_global": (
        cum_pos - tmp["y"] + tmp["y"].mean()
    ) / cum_n,
    "current_plus_global": (
        cum_pos + tmp["y"].mean()
    ) / (cum_n + 1),
}

for name, pred in candidates.items():

    diff = np.abs(
        pred.to_numpy() -
        tmp["actual"].to_numpy()
    )

    print(
        f"{name:25s} "
        f"MAE={np.nanmean(diff):.8f} "
        f"MAX={np.nanmax(diff):.8f}"
    )

CUMULATIVE POSITIVE COUNT TEST
previous_rate             MAE=0.00631216 MAX=0.99565217
current_rate              MAE=0.00630812 MAX=0.99565689
previous_plus_global      MAE=0.00629711 MAX=0.85049604
current_plus_global       MAE=0.00628979 MAX=0.85158894


In [114]:
# ============================================================
# CELL 86X — GLOBAL DATE-ORDERED EXPANDING TEST
# ============================================================

print("=" * 70)
print("GLOBAL DATE-ORDERED EXPANDING TEST")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "y": pd.to_numeric(train_v2["target"], errors="raise").astype(float).values,
    "actual": train_v2["date_target_rate_loo"].astype(float).values
})

# Sort by Date while preserving row order within each date
tmp["_order"] = np.arange(len(tmp))

ordered = tmp.sort_values(
    ["Date", "_order"]
).copy()

ordered["global_prev_rate"] = (
    ordered["y"]
    .shift(1)
    .expanding(min_periods=1)
    .mean()
)

# Restore original order
ordered = ordered.sort_index()

diff = np.abs(
    ordered["global_prev_rate"].to_numpy() -
    ordered["actual"].to_numpy()
)

print("Global previous expanding MAE :", np.nanmean(diff))
print("Median error                  :", np.nanmedian(diff))
print("Max error                     :", np.nanmax(diff))
print("< 1e-4                        :", (diff < 1e-4).sum())
print("< 1e-5                        :", (diff < 1e-5).sum())

GLOBAL DATE-ORDERED EXPANDING TEST
Global previous expanding MAE : 0.011298323736461603
Median error                  : 0.009615072634761449
Max error                     : 0.7125
< 1e-4                        : 10390
< 1e-5                        : 2075


In [115]:
# ============================================================
# CELL 86Y — GLOBAL ROW-LEVEL LOO TEST
# ============================================================

print("=" * 70)
print("GLOBAL ROW-LEVEL LOO TEST")
print("=" * 70)

y = pd.to_numeric(
    train_v2["target"],
    errors="raise"
).astype(float)

actual = train_v2["date_target_rate_loo"].astype(float)

total_positive = y.sum()
total_rows = len(y)

global_loo = (
    total_positive - y
) / (
    total_rows - 1
)

diff = np.abs(
    global_loo.to_numpy() -
    actual.to_numpy()
)

print("MAE          :", diff.mean())
print("Median error :", np.median(diff))
print("Max error    :", diff.max())
print("Correlation  :", np.corrcoef(
    global_loo.to_numpy(),
    actual.to_numpy()
)[0, 1])

GLOBAL ROW-LEVEL LOO TEST
MAE          : 0.010077216653240297
Median error : 0.008562768730217888
Max error    : 0.1192812182686987
Correlation  : -0.15981267448263076


In [116]:
# ============================================================
# CELL 86Z — FEATURE PRECISION ANALYSIS
# ============================================================

print("=" * 70)
print("FEATURE PRECISION ANALYSIS")
print("=" * 70)

actual = train_v2["date_target_rate_loo"].astype(float)

print("Raw examples:")
print(
    actual.head(30).to_string(index=False)
)

print("\nRounded uniqueness:")

for decimals in [3, 4, 5, 6, 7, 8, 9, 10, 12]:

    unique_count = (
        actual.round(decimals)
        .nunique()
    )

    print(
        f"{decimals:2d} decimals -> "
        f"{unique_count:,} unique values"
    )

print("\nMost common exact values:")
print(
    actual.value_counts()
    .head(30)
    .to_string()
)

FEATURE PRECISION ANALYSIS
Raw examples:
0.037344
0.033333
0.033333
0.033333
0.037500
0.037344
0.037500
0.037500
0.037500
0.037500
0.037344
0.033333
0.037500
0.037500
0.037500
0.037344
0.033333
0.033333
0.037500
0.037500
0.037344
0.037500
0.037500
0.037500
0.037500
0.037344
0.037500
0.037500
0.037500
0.037500

Rounded uniqueness:
 3 decimals -> 90 unique values
 4 decimals -> 415 unique values
 5 decimals -> 996 unique values
 6 decimals -> 1,181 unique values
 7 decimals -> 1,181 unique values
 8 decimals -> 1,181 unique values
 9 decimals -> 1,181 unique values
10 decimals -> 1,181 unique values
12 decimals -> 1,181 unique values

Most common exact values:
date_target_rate_loo
0.000000    655347
0.001098      8431
0.001091      7117
0.001088      6672
0.002148      6291
0.001089      6208
0.002186      6177
0.001087      5987
0.001085      5773
0.001086      5760
0.001094      5721
0.001080      4866
0.003236      4857
0.001093      4817
0.002155      4647
0.002167      4627
0.003250

In [117]:
# ============================================================
# CELL 86AA — PREVIOUS TARGET HISTORY TEST
# ============================================================

print("=" * 70)
print("PREVIOUS TARGET HISTORY TEST")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "y": pd.to_numeric(train_v2["target"], errors="raise").astype(int).values,
    "actual": train_v2["date_target_rate_loo"].astype(float).values
})

# Previous target indicators
tmp["prev1"] = tmp.groupby("Date")["y"].shift(1)
tmp["prev2"] = tmp.groupby("Date")["y"].shift(2)
tmp["prev3"] = tmp.groupby("Date")["y"].shift(3)

for col in ["prev1", "prev2", "prev3"]:

    valid = tmp[col].notna()

    print(
        f"\nCorrelation(actual, {col}): "
        f"{tmp.loc[valid, 'actual'].corr(tmp.loc[valid, col]):.8f}"
    )

# Feature changes after target transitions
tmp["target_change"] = (
    tmp.groupby("Date")["y"]
    .diff()
)

print("\nFeature mean after target transitions:")

print(
    tmp.groupby("target_change", dropna=False)["actual"]
    .agg(["count", "mean", "median"])
    .to_string()
)

PREVIOUS TARGET HISTORY TEST

Correlation(actual, prev1): 0.16121416

Correlation(actual, prev2): 0.16192968

Correlation(actual, prev3): 0.16239399

Feature mean after target transitions:
                 count      mean    median
target_change                             
-1.0              3655  0.029164  0.022272
 0.0           1630682  0.008468  0.002144
 1.0              3655  0.028326  0.021668
 NaN              1432  0.008544  0.002146


In [118]:
# ============================================================
# CELL 86AB — SYSTEMATIC DATE COUNT SEARCH
# ============================================================

print("=" * 70)
print("SYSTEMATIC DATE COUNT FORMULA SEARCH")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "y": pd.to_numeric(train_v2["target"], errors="raise").astype(float).values,
    "actual": train_v2["date_target_rate_loo"].astype(float).values
})

g = tmp.groupby("Date", sort=False)

P = g["y"].transform("sum")
N = g["y"].transform("count")
Y = tmp["y"]

actual = tmp["actual"].to_numpy()

candidates = {}

# Different numerator adjustments
for a in [-2, -1, 0, 1, 2]:
    # Different denominator adjustments
    for b in [-2, -1, 0, 1, 2]:

        denominator = N + b

        valid = denominator > 0

        pred = np.full(len(tmp), np.nan)

        pred[valid] = (
            (P + a - Y)[valid] /
            denominator[valid]
        )

        candidates[
            f"(P+{a}-Y)/(N+{b})"
        ] = pred

results = []

for name, pred in candidates.items():

    diff = np.abs(pred - actual)

    results.append({
        "formula": name,
        "MAE": np.nanmean(diff),
        "median": np.nanmedian(diff),
        "max": np.nanmax(diff)
    })

result_df = (
    pd.DataFrame(results)
    .sort_values("MAE")
)

print(
    result_df.head(15).to_string(index=False)
)

SYSTEMATIC DATE COUNT FORMULA SEARCH
        formula      MAE   median      max
 (P+0-Y)/(N+-2) 0.000438 0.000218 0.005228
 (P+0-Y)/(N+-1) 0.000438 0.000218 0.005154
  (P+0-Y)/(N+0) 0.000439 0.000219 0.005081
  (P+0-Y)/(N+1) 0.000439 0.000220 0.005094
  (P+0-Y)/(N+2) 0.000440 0.000221 0.005134
  (P+1-Y)/(N+2) 0.000968 0.000864 0.047619
  (P+1-Y)/(N+1) 0.000972 0.000864 0.050000
  (P+1-Y)/(N+0) 0.000977 0.000865 0.052632
 (P+1-Y)/(N+-1) 0.000982 0.000867 0.055556
 (P+1-Y)/(N+-2) 0.000987 0.000869 0.058824
(P+-1-Y)/(N+-2) 0.000987 0.000873 0.058824
(P+-1-Y)/(N+-1) 0.000990 0.000873 0.055556
 (P+-1-Y)/(N+0) 0.000994 0.000873 0.052632
 (P+-1-Y)/(N+1) 0.000997 0.000873 0.050000
 (P+-1-Y)/(N+2) 0.001001 0.000872 0.047619


In [120]:
# ============================================================
# CELL 86AC — FIND ORIGINAL FEATURE-ENGINEERING OBJECTS
# ============================================================

print("=" * 70)
print("SEARCHING NOTEBOOK STATE")
print("=" * 70)

keywords = [
    "date",
    "loo",
    "target",
    "rate",
    "feature",
    "engineer",
    "encode"
]

matches = []

# IMPORTANT: snapshot keys first
global_names = list(globals().keys())

for name in global_names:

    lname = str(name).lower()

    if any(k in lname for k in keywords):
        matches.append(name)

for name in sorted(matches):

    try:
        obj = globals()[name]

        print(
            f"{name:45s} | "
            f"{type(obj).__name__}"
        )

    except Exception:
        pass

SEARCHING NOTEBOOK STATE
TRAIN_ONLY_FEATURES                           | list
actual_loo                                    | Series
add_date_features                             | function
add_sensor_regime_features                    | function
best_date_te                                  | Series
best_sensor_date                              | Series
best_sensor_lookup                            | Series
candidates                                    | dict
clean_date_rates                              | Series
clean_train_dates                             | set
clean_valid_dates                             | set
date_count                                    | Series
date_features                                 | list
date_info                                     | DataFrame
date_mad                                      | Series
date_positive_sum                             | Series
date_relative_features                        | DataFrame
date_relative_results                     

In [121]:
# ============================================================
# CELL 86AD — FIND FEATURE ENGINEERING FUNCTIONS
# ============================================================

print("=" * 70)
print("SEARCHING FUNCTIONS")
print("=" * 70)

import inspect

# Snapshot first — prevents dictionary-changed-size error
global_items = list(globals().items())

for name, obj in global_items:

    try:
        is_function = inspect.isfunction(obj)
    except Exception:
        is_function = False

    if not is_function:
        continue

    lname = str(name).lower()

    if any(
        k in lname
        for k in [
            "feature",
            "engineer",
            "date",
            "target",
            "encode",
            "loo"
        ]
    ):

        print("\n" + "=" * 70)
        print("FUNCTION:", name)
        print("=" * 70)

        try:
            source = inspect.getsource(obj)
            print(source[:10000])
        except Exception as e:
            print("Source unavailable:", e)

SEARCHING FUNCTIONS

FUNCTION: add_date_features
def add_date_features(df):
    
    df = df.copy()
    
    # Make sure Date is datetime
    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="coerce"
    )
    
    # Calendar features
    df["year"] = df["Date"].dt.year.astype("int16")
    df["month"] = df["Date"].dt.month.astype("int8")
    df["day"] = df["Date"].dt.day.astype("int8")
    df["day_of_week"] = (
        df["Date"]
        .dt.dayofweek
        .astype("int8")
    )
    
    df["day_of_year"] = (
        df["Date"]
        .dt.dayofyear
        .astype("int16")
    )
    
    df["quarter"] = (
        df["Date"]
        .dt.quarter
        .astype("int8")
    )
    
    
    # Cyclic encoding
    df["month_sin"] = np.sin(
        2 * np.pi * df["month"] / 12
    )
    
    df["month_cos"] = np.cos(
        2 * np.pi * df["month"] / 12
    )
    
    df["day_of_week_sin"] = np.sin(
        2 * np.pi * df["day_of_week"] / 7
    )
    
    df["day_of_week_cos

In [122]:
# ============================================================
# CELL 86AE — DATAFRAME METADATA
# ============================================================

print("=" * 70)
print("TRAIN_V2 METADATA")
print("=" * 70)

print("Shape:", train_v2.shape)

print("\nColumns:")
for i, c in enumerate(train_v2.columns):
    print(f"{i:02d}: {c}")

print("\nDtypes:")
print(train_v2.dtypes.to_string())

print("\nDataFrame attrs:")
print(list(train_v2.attrs.keys()))

print("\nIndex:")
print(type(train_v2.index))
print("Index name:", train_v2.index.name)

print("\nTarget dtype:")
print(train_v2["target"].dtype)

print("\nDate dtype:")
print(train_v2["Date"].dtype)

TRAIN_V2 METADATA
Shape: (1639424, 54)

Columns:
00: Date
01: X1
02: X2
03: X3
04: X4
05: X5
06: target
07: year
08: month
09: day
10: day_of_week
11: day_of_year
12: quarter
13: month_sin
14: month_cos
15: day_of_week_sin
16: day_of_week_cos
17: day_of_year_sin
18: day_of_year_cos
19: X1_log_abs
20: X1_log10_abs
21: X1_is_zero
22: X1_is_one
23: X1_large
24: X1_extreme
25: X2_log_abs
26: X2_log10_abs
27: X2_is_zero
28: X2_is_one
29: X2_large
30: X2_extreme
31: X3_log_abs
32: X3_log10_abs
33: X3_is_zero
34: X3_is_one
35: X3_large
36: X3_extreme
37: X4_log_abs
38: X4_log10_abs
39: X4_is_zero
40: X4_is_one
41: X4_large
42: X4_extreme
43: X5_log_abs
44: X5_log10_abs
45: X5_is_zero
46: X5_is_one
47: X5_large
48: X5_extreme
49: _original_row
50: _row_in_date
51: date_target_rate_loo
52: _sensor_key
53: _sensor_date_key

Dtypes:
Date                    datetime64[ns]
X1                             float64
X2                             float64
X3                             float64
X4        

In [123]:
# ============================================================
# CELL 86AF — FEATURE RELATIONSHIP SEARCH
# ============================================================

print("=" * 70)
print("RELATIONSHIP SEARCH")
print("=" * 70)

target_feature = train_v2[
    "date_target_rate_loo"
].astype(float)

numeric_cols = train_v2.select_dtypes(
    include=[np.number]
).columns.tolist()

numeric_cols = [
    c for c in numeric_cols
    if c != "date_target_rate_loo"
]

results = []

for col in numeric_cols:

    x = pd.to_numeric(
        train_v2[col],
        errors="coerce"
    )

    valid = (
        x.notna() &
        target_feature.notna()
    )

    if valid.sum() > 100:

        corr = x[valid].corr(
            target_feature[valid]
        )

        results.append({
            "feature": col,
            "correlation": corr
        })

corr_df = (
    pd.DataFrame(results)
    .sort_values(
        "correlation",
        key=lambda x: x.abs(),
        ascending=False
    )
)

print(
    corr_df.to_string(index=False)
)

RELATIONSHIP SEARCH
        feature  correlation
day_of_year_cos     0.224671
      month_cos     0.222106
  _original_row    -0.210216
           year    -0.201090
       X4_large     0.143350
   X4_log10_abs     0.140856
     X4_log_abs     0.140832
     X4_extreme     0.137111
day_of_week_sin     0.110882
      X3_is_one    -0.110234
   X3_log10_abs     0.099712
     X3_log_abs     0.098738
       X3_large     0.097467
     X3_extreme     0.093629
             X5     0.076629
     X5_log_abs     0.056346
    day_of_week    -0.052291
      month_sin     0.047862
             X1     0.044960
     X1_log_abs     0.043023
   X1_log10_abs     0.041383
      X4_is_one    -0.038144
     X5_is_zero     0.029478
   X5_log10_abs    -0.023463
        quarter     0.018328
          month     0.016910
    day_of_year     0.015978
            day    -0.014595
day_of_year_sin     0.014442
day_of_week_cos    -0.010887
      X1_is_one    -0.009695
             X4     0.005354
             X2     0.0

In [124]:
# ============================================================
# CELL 86AG — ROW POSITION MATHEMATICAL RELATIONSHIP
# ============================================================

print("=" * 70)
print("ROW POSITION MATHEMATICAL RELATIONSHIP")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "y": pd.to_numeric(
        train_v2["target"],
        errors="raise"
    ).astype(int).values,
    "row": train_v2["_row_in_date"].values,
    "feature": train_v2[
        "date_target_rate_loo"
    ].astype(float).values
})

tmp["date_N"] = (
    tmp.groupby("Date")["y"]
    .transform("size")
)

tmp["date_P"] = (
    tmp.groupby("Date")["y"]
    .transform("sum")
)

tmp["N_minus_row"] = (
    tmp["date_N"] -
    tmp["row"]
)

tmp["P_minus_y"] = (
    tmp["date_P"] -
    tmp["y"]
)

candidates = {
    "P_minus_y / (N-row-1)": (
        tmp["P_minus_y"] /
        (tmp["N_minus_row"] - 1)
    ),

    "P_minus_y / (N-row)": (
        tmp["P_minus_y"] /
        tmp["N_minus_row"]
    ),

    "P / (N-row)": (
        tmp["date_P"] /
        tmp["N_minus_row"]
    ),

    "P / (N-row+1)": (
        tmp["date_P"] /
        (tmp["N_minus_row"] + 1)
    )
}

actual = tmp["feature"].to_numpy()

results = []

for name, candidate in candidates.items():

    arr = candidate.to_numpy()

    diff = np.abs(
        arr - actual
    )

    results.append({
        "formula": name,
        "MAE": np.nanmean(diff),
        "median": np.nanmedian(diff),
        "max": np.nanmax(diff)
    })

print(
    pd.DataFrame(results)
    .sort_values("MAE")
    .to_string(index=False)
)

ROW POSITION MATHEMATICAL RELATIONSHIP
              formula      MAE   median        max
        P / (N-row+1) 0.048200 0.001114  73.372156
  P_minus_y / (N-row) 0.056704 0.001116 146.872156
          P / (N-row) 0.056755 0.001119 146.872156
P_minus_y / (N-row-1)      inf 0.001124        inf


In [125]:
# ============================================================
# CELL 86AH — GLOBAL SEQUENCE POSITION SEARCH
# ============================================================

print("=" * 70)
print("SEQUENCE POSITION SEARCH")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "y": pd.to_numeric(
        train_v2["target"],
        errors="raise"
    ).astype(float).values,
    "feature": train_v2[
        "date_target_rate_loo"
    ].astype(float).values
})

cum_y = tmp["y"].cumsum()
cum_n = np.arange(1, len(tmp) + 1)

candidates = {
    "global_previous":
        (cum_y - tmp["y"]) /
        np.maximum(cum_n - 1, 1),

    "global_current":
        cum_y / cum_n,

    "global_previous_plus_one":
        (cum_y - tmp["y"] + 1) /
        np.maximum(cum_n, 1)
}

actual = tmp["feature"].to_numpy()

results = []

for name, candidate in candidates.items():

    arr = candidate.to_numpy()

    diff = np.abs(
        arr - actual
    )

    results.append({
        "formula": name,
        "MAE": np.nanmean(diff),
        "correlation": np.corrcoef(
            arr,
            actual
        )[0, 1]
    })

print(
    pd.DataFrame(results)
    .sort_values("MAE")
    .to_string(index=False)
)

SEQUENCE POSITION SEARCH
                 formula      MAE  correlation
          global_current 0.011395     0.192971
         global_previous 0.011395     0.192974
global_previous_plus_one 0.011401     0.184767


In [126]:
# ============================================================
# CELL 86AI — EXACT FEATURE VALUE COMPARISON
# ============================================================

print("=" * 70)
print("EXACT FEATURE VALUE COMPARISON")
print("=" * 70)

tmp = pd.DataFrame({
    "Date": train_v2["Date"].values,
    "y": pd.to_numeric(
        train_v2["target"],
        errors="raise"
    ).astype(int).values,
    "row": train_v2["_row_in_date"].values,
    "feature": train_v2[
        "date_target_rate_loo"
    ].astype(float).values
})

stats = (
    tmp.groupby("Date")
    .agg(
        N=("y", "size"),
        P=("y", "sum"),
        rate=("y", "mean")
    )
)

tmp = tmp.join(
    stats,
    on="Date",
    rsuffix="_date"
)

first_date = tmp["Date"].iloc[0]

d = tmp[
    tmp["Date"] == first_date
].copy()

print("Date:", first_date)
print(
    "N =", d["N"].iloc[0],
    "P =", d["P"].iloc[0],
    "rate =", d["rate"].iloc[0]
)

print("\nAll unique feature values:")
print(
    sorted(d["feature"].unique())
)

print("\nValue counts by target:")
print(
    d.groupby(
        ["y", "feature"]
    ).size()
    .reset_index(name="count")
    .to_string(index=False)
)

print("\nRows 0-100:")
print(
    d[
        ["row", "y", "feature"]
    ].head(100).to_string(index=False)
)

EXACT FEATURE VALUE COMPARISON
Date: 2020-12-16 00:00:00
N = 302 P = 11 rate = 0.03642384105960265

All unique feature values:
[np.float64(0.03333333333333333), np.float64(0.03734439834024896), np.float64(0.0375)]

Value counts by target:
 y  feature  count
 0 0.037344     59
 0 0.037500    232
 1 0.033333      9
 1 0.037344      2

Rows 0-100:
 row  y  feature
   0  0 0.037344
   1  1 0.033333
   2  1 0.033333
   3  1 0.033333
   4  0 0.037500
   5  0 0.037344
   6  0 0.037500
   7  0 0.037500
   8  0 0.037500
   9  0 0.037500
  10  1 0.037344
  11  1 0.033333
  12  0 0.037500
  13  0 0.037500
  14  0 0.037500
  15  0 0.037344
  16  1 0.033333
  17  1 0.033333
  18  0 0.037500
  19  0 0.037500
  20  0 0.037344
  21  0 0.037500
  22  0 0.037500
  23  0 0.037500
  24  0 0.037500
  25  0 0.037344
  26  0 0.037500
  27  0 0.037500
  28  0 0.037500
  29  0 0.037500
  30  0 0.037344
  31  0 0.037500
  32  0 0.037500
  33  1 0.033333
  34  1 0.033333
  35  1 0.037344
  36  0 0.037500
  37  0

In [127]:
# ============================================================
# FINAL FAST TRACK — CELL 1/7
# LOCK 47-FEATURE V4 STRATEGY
# ============================================================

print("=" * 70)
print("FINAL 47-FEATURE V4 STRATEGY")
print("=" * 70)

# The 3 unavailable / target-derived columns
EXCLUDE_FINAL = [
    "_original_row",
    "_row_in_date",
    "date_target_rate_loo"
]

# Exact features available in BOTH train and test
FINAL_FEATURES = [
    f for f in hgb_features
    if f not in EXCLUDE_FINAL
    and f in train_v2.columns
    and f in test_v2.columns
]

print("Original features :", len(hgb_features))
print("Final features    :", len(FINAL_FEATURES))
print("Excluded          :", EXCLUDE_FINAL)

assert len(FINAL_FEATURES) == 47

print("\nFinal feature list:")
for i, f in enumerate(FINAL_FEATURES):
    print(f"{i:02d}: {f}")

print("\n✅ 47-feature configuration locked.")

FINAL 47-FEATURE V4 STRATEGY
Original features : 50
Final features    : 47
Excluded          : ['_original_row', '_row_in_date', 'date_target_rate_loo']

Final feature list:
00: X1
01: X2
02: X3
03: X4
04: X5
05: year
06: month
07: day
08: day_of_week
09: day_of_year
10: quarter
11: month_sin
12: month_cos
13: day_of_week_sin
14: day_of_week_cos
15: day_of_year_sin
16: day_of_year_cos
17: X1_log_abs
18: X1_log10_abs
19: X1_is_zero
20: X1_is_one
21: X1_large
22: X1_extreme
23: X2_log_abs
24: X2_log10_abs
25: X2_is_zero
26: X2_is_one
27: X2_large
28: X2_extreme
29: X3_log_abs
30: X3_log10_abs
31: X3_is_zero
32: X3_is_one
33: X3_large
34: X3_extreme
35: X4_log_abs
36: X4_log10_abs
37: X4_is_zero
38: X4_is_one
39: X4_large
40: X4_extreme
41: X5_log_abs
42: X5_log10_abs
43: X5_is_zero
44: X5_is_one
45: X5_large
46: X5_extreme

✅ 47-feature configuration locked.


In [128]:
# ============================================================
# FINAL FAST TRACK — CELL 2/7
# BUILD FULL TRAINING POOL
# ============================================================

print("=" * 70)
print("BUILDING FINAL TRAINING POOL")
print("=" * 70)

X_final_full = train_v2[FINAL_FEATURES].copy()

y_final_full = pd.to_numeric(
    train_v2["target"],
    errors="raise"
).astype(np.int8).to_numpy()

# Indices
positive_final_idx = np.flatnonzero(
    y_final_full == 1
)

negative_final_idx = np.flatnonzero(
    y_final_full == 0
)

print("Total rows       :", len(X_final_full))
print("Positive rows    :", len(positive_final_idx))
print("Negative rows    :", len(negative_final_idx))
print("Positive rate    :", y_final_full.mean())
print("Feature count    :", X_final_full.shape[1])

print("\nNull values:")
print(X_final_full.isna().sum().sum())

print("\nInfinite values:")
print(np.isinf(X_final_full.to_numpy()).sum())

assert X_final_full.shape[1] == 47
assert len(positive_final_idx) == 14038

print("\n✅ Training pool ready.")

BUILDING FINAL TRAINING POOL
Total rows       : 1639424
Positive rows    : 14038
Negative rows    : 1625386
Positive rate    : 0.00856276350718301
Feature count    : 47

Null values:
0

Infinite values:
0

✅ Training pool ready.


In [129]:
# ============================================================
# FINAL FAST TRACK — CELL 3/7
# CREATE 3 FINAL TRAINING SAMPLES
# ============================================================

print("=" * 70)
print("CREATING FINAL V4 SAMPLES")
print("=" * 70)

FINAL_NEGATIVES = 500_000
FINAL_MODELS = 3

final_47_samples = []

for model_number in range(FINAL_MODELS):

    seed = RANDOM_STATE + 5000 + model_number

    rng = np.random.default_rng(seed)

    sampled_negatives = rng.choice(
        negative_final_idx,
        size=FINAL_NEGATIVES,
        replace=False
    )

    sample_idx = np.concatenate([
        positive_final_idx,
        sampled_negatives
    ])

    # Shuffle
    rng.shuffle(sample_idx)

    final_47_samples.append(sample_idx)

    sample_positive = (
        y_final_full[sample_idx].sum()
    )

    print(
        f"Model {model_number + 1}: "
        f"rows={len(sample_idx):,} | "
        f"positives={sample_positive:,} | "
        f"positive_rate={sample_positive / len(sample_idx):.4%}"
    )

print("\n✅ 3 samples created.")

CREATING FINAL V4 SAMPLES
Model 1: rows=514,038 | positives=14,038 | positive_rate=2.7309%
Model 2: rows=514,038 | positives=14,038 | positive_rate=2.7309%
Model 3: rows=514,038 | positives=14,038 | positive_rate=2.7309%

✅ 3 samples created.


In [131]:
# ============================================================
# FINAL FAST TRACK — CELL 4/7 — FIXED
# TRAIN 3 V4 MODELS
# ============================================================

import time

print("=" * 70)
print("TRAINING FINAL 47-FEATURE V4 ENSEMBLE")
print("=" * 70)

final_47_models = []

for model_number, sample_idx in enumerate(
    final_47_samples,
    start=1
):

    print(
        f"\n{'=' * 60}\n"
        f"TRAINING MODEL {model_number}/{FINAL_MODELS}\n"
        f"{'=' * 60}"
    )

    model = HistGradientBoostingClassifier(
        learning_rate=0.04,
        max_iter=650,
        max_leaf_nodes=127,
        max_depth=None,
        min_samples_leaf=35,
        l2_regularization=2.0,
        max_bins=255,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=40,
        random_state=RANDOM_STATE + 6000 + model_number,
        verbose=1
    )

    X_sample = X_final_full.iloc[sample_idx]
    y_sample = y_final_full[sample_idx]

    start_time = time.time()

    model.fit(
        X_sample,
        y_sample
    )

    elapsed = time.time() - start_time

    final_47_models.append(model)

    print(
        f"\n✅ Model {model_number} completed "
        f"in {elapsed:.1f} seconds"
    )

    print(
        "Iterations:",
        getattr(model, "n_iter_", "N/A")
    )

print("\n" + "=" * 70)
print("✅ ALL 3 MODELS TRAINED")
print("=" * 70)

TRAINING FINAL 47-FEATURE V4 ENSEMBLE

TRAINING MODEL 1/3
Binning 0.174 GB of training data: 0.948 s
Binning 0.019 GB of validation data: 0.068 s
Fitting gradient boosted rounds:
Fit 650 trees in 98.941 s, (82550 total leaves)
Time spent computing histograms: 70.901s
Time spent finding best splits:  4.251s
Time spent applying splits:      8.178s
Time spent predicting:           0.998s

✅ Model 1 completed in 98.9 seconds
Iterations: 650

TRAINING MODEL 2/3
Binning 0.174 GB of training data: 0.942 s
Binning 0.019 GB of validation data: 0.062 s
Fitting gradient boosted rounds:
Fit 650 trees in 98.914 s, (82550 total leaves)
Time spent computing histograms: 70.142s
Time spent finding best splits:  4.153s
Time spent applying splits:      9.126s
Time spent predicting:           1.002s

✅ Model 2 completed in 98.9 seconds
Iterations: 650

TRAINING MODEL 3/3
Binning 0.174 GB of training data: 0.925 s
Binning 0.019 GB of validation data: 0.070 s
Fitting gradient boosted rounds:
Fit 650 trees i

In [132]:
# ============================================================
# FINAL FAST TRACK — CELL 5/7
# TEST PREDICTIONS
# ============================================================

print("=" * 70)
print("PREDICTING TEST SET")
print("=" * 70)

X_final_test = test_v2[FINAL_FEATURES].copy()

assert X_final_test.shape[1] == 47
assert X_final_test.shape[0] == 409856

test_probabilities_47 = []

for i, model in enumerate(
    final_47_models,
    start=1
):

    print(
        f"\nPredicting model {i}/{FINAL_MODELS}..."
    )

    prob = model.predict_proba(
        X_final_test
    )[:, 1]

    test_probabilities_47.append(prob)

    print(
        f"Mean probability : {prob.mean():.8f}"
    )

    print(
        f"Max probability  : {prob.max():.8f}"
    )

# Ensemble
final_47_probability = np.mean(
    np.vstack(test_probabilities_47),
    axis=0
)

print("\n" + "=" * 70)
print("ENSEMBLE SUMMARY")
print("=" * 70)

print(
    "Rows       :",
    len(final_47_probability)
)

print(
    "Mean       :",
    f"{final_47_probability.mean():.8f}"
)

print(
    "Median     :",
    f"{np.median(final_47_probability):.8f}"
)

print(
    "90th pct   :",
    f"{np.percentile(final_47_probability, 90):.8f}"
)

print(
    "95th pct   :",
    f"{np.percentile(final_47_probability, 95):.8f}"
)

print(
    "99th pct   :",
    f"{np.percentile(final_47_probability, 99):.8f}"
)

print(
    "Max        :",
    f"{final_47_probability.max():.8f}"
)

print("\n✅ Test prediction complete.")

PREDICTING TEST SET

Predicting model 1/3...
Mean probability : 0.01238321
Max probability  : 0.99999297

Predicting model 2/3...
Mean probability : 0.01230402
Max probability  : 0.99999565

Predicting model 3/3...
Mean probability : 0.01228467
Max probability  : 0.99998924

ENSEMBLE SUMMARY
Rows       : 409856
Mean       : 0.01232397
Median     : 0.00005802
90th pct   : 0.00489312
95th pct   : 0.01765217
99th pct   : 0.41540949
Max        : 0.99998845

✅ Test prediction complete.


In [133]:
# ============================================================
# FINAL FAST TRACK — CELL 6/7
# THRESHOLD ANALYSIS
# ============================================================

print("=" * 70)
print("FINAL THRESHOLD ANALYSIS")
print("=" * 70)

thresholds = [
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.575,
    0.60,
    0.625,
    0.65,
    0.675,
    0.70,
    0.75
]

print(
    f"{'Threshold':>12} "
    f"{'Anomalies':>12} "
    f"{'Rate %':>12}"
)

print("-" * 40)

for threshold in thresholds:

    pred = final_47_probability >= threshold

    print(
        f"{threshold:>12.3f} "
        f"{pred.sum():>12,} "
        f"{pred.mean()*100:>11.4f}%"
    )

print("\n" + "=" * 70)
print("TRAIN REFERENCE")
print("=" * 70)

print(
    f"Train anomalies : "
    f"{y_final_full.sum():,}"
)

print(
    f"Train rate      : "
    f"{y_final_full.mean()*100:.4f}%"
)

print(
    f"Expected test-scale anomalies at train rate: "
    f"{int(len(test_v2) * y_final_full.mean()):,}"
)

print("=" * 70)

FINAL THRESHOLD ANALYSIS
   Threshold    Anomalies       Rate %
----------------------------------------
       0.300        4,862      1.1863%
       0.350        4,482      1.0936%
       0.400        4,181      1.0201%
       0.450        3,925      0.9577%
       0.500        3,720      0.9076%
       0.550        3,503      0.8547%
       0.575        3,404      0.8305%
       0.600        3,302      0.8056%
       0.625        3,226      0.7871%
       0.650        3,141      0.7664%
       0.675        3,048      0.7437%
       0.700        2,959      0.7220%
       0.750        2,786      0.6798%

TRAIN REFERENCE
Train anomalies : 14,038
Train rate      : 0.8563%
Expected test-scale anomalies at train rate: 3,509


In [134]:
# ============================================================
# FINAL FAST TRACK — CELL 7/7
# CREATE FINAL KAGGLE SUBMISSION
# ============================================================

print("=" * 70)
print("CREATING FINAL SUBMISSION")
print("=" * 70)

# ------------------------------------------------------------
# LOCK FINAL THRESHOLD
# ------------------------------------------------------------

FINAL_THRESHOLD = 0.625

final_prediction = (
    final_47_probability >= FINAL_THRESHOLD
).astype(int)

# ------------------------------------------------------------
# Build submission using TEST IDs
# ------------------------------------------------------------

submission = pd.DataFrame({
    "ID": test_v2["ID"].to_numpy(),
    "target": final_prediction
})

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert len(submission) == 409856
assert list(submission.columns) == ["ID", "target"]

assert submission["ID"].is_unique
assert submission["ID"].equals(test_v2["ID"])

assert set(submission["target"].unique()).issubset({0, 1})

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

submission_path = "/kaggle/working/submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

print("\nFINAL MODEL")
print("-" * 70)
print("Model              : 3-model V4 HGB ensemble")
print("Features           : 47")
print("Training positives : 14,038")
print("Negatives/sample   : 500,000 per model")
print("Threshold           :", FINAL_THRESHOLD)

print("\nPREDICTIONS")
print("-" * 70)

print(
    f"Total test rows    : {len(submission):,}"
)

print(
    f"Anomalies          : {final_prediction.sum():,}"
)

print(
    f"Normal             : {(final_prediction == 0).sum():,}"
)

print(
    f"Anomaly rate       : "
    f"{final_prediction.mean()*100:.4f}%"
)

print("\nSUBMISSION")
print("-" * 70)

print("Shape:", submission.shape)
print("Columns:", submission.columns.tolist())
print("File:", submission_path)

print("\nFirst 10 rows:")
print(submission.head(10).to_string(index=False))

print("\nLast 10 rows:")
print(submission.tail(10).to_string(index=False))

print("\nTarget distribution:")
print(submission["target"].value_counts().sort_index())

print("\n" + "=" * 70)
print("✅ SUBMISSION READY — UPLOAD submission.csv TO KAGGLE")
print("=" * 70)

CREATING FINAL SUBMISSION

FINAL MODEL
----------------------------------------------------------------------
Model              : 3-model V4 HGB ensemble
Features           : 47
Training positives : 14,038
Negatives/sample   : 500,000 per model
Threshold           : 0.625

PREDICTIONS
----------------------------------------------------------------------
Total test rows    : 409,856
Anomalies          : 3,226
Normal             : 406,630
Anomaly rate       : 0.7871%

SUBMISSION
----------------------------------------------------------------------
Shape: (409856, 2)
Columns: ['ID', 'target']
File: /kaggle/working/submission.csv

First 10 rows:
 ID  target
  0       0
  1       0
  2       0
  3       0
  4       0
  5       0
  6       0
  7       0
  8       1
  9       0

Last 10 rows:
    ID  target
409846       0
409847       0
409848       0
409849       0
409850       0
409851       0
409852       0
409853       0
409854       0
409855       0

Target distribution:
target
0    4

In [135]:
# ============================================================
# VERIFY 47-FEATURE V4 VALIDATION F1
# ============================================================

print("=" * 70)
print("VERIFYING 47-FEATURE V4 F1")
print("=" * 70)

import time

# ------------------------------------------------------------
# Use the clean date-stratified group split created earlier
# ------------------------------------------------------------

# Check available variables
required = [
    "clean_train_idx",
    "clean_valid_idx"
]

missing = [
    x for x in required
    if x not in globals()
]

print("Missing split variables:", missing)

if missing:
    print("\nAvailable split-related variables:")
    for name in globals():
        if "valid" in name.lower() or "clean" in name.lower():
            print(name)
    
    raise NameError(
        "Clean validation indices are not available in notebook state."
    )

# ------------------------------------------------------------
# Build validation data
# ------------------------------------------------------------

X_train_check = train_v2.iloc[
    clean_train_idx
][FINAL_FEATURES]

y_train_check = pd.to_numeric(
    train_v2.iloc[
        clean_train_idx
    ]["target"],
    errors="raise"
).astype(int).to_numpy()

X_valid_check = train_v2.iloc[
    clean_valid_idx
][FINAL_FEATURES]

y_valid_check = pd.to_numeric(
    train_v2.iloc[
        clean_valid_idx
    ]["target"],
    errors="raise"
).astype(int).to_numpy()

print("\nTrain:", X_train_check.shape)
print("Valid:", X_valid_check.shape)
print("Validation positives:", y_valid_check.sum())

# ------------------------------------------------------------
# Positive-preserving training sample
# ALL positives + 500k negatives
# ------------------------------------------------------------

train_pos_idx = np.flatnonzero(
    y_train_check == 1
)

train_neg_idx = np.flatnonzero(
    y_train_check == 0
)

rng = np.random.default_rng(
    RANDOM_STATE + 9000
)

n_neg = min(
    500_000,
    len(train_neg_idx)
)

sample_neg_idx = rng.choice(
    train_neg_idx,
    size=n_neg,
    replace=False
)

sample_idx = np.concatenate([
    train_pos_idx,
    sample_neg_idx
])

rng.shuffle(sample_idx)

X_check_sample = X_train_check.iloc[
    sample_idx
]

y_check_sample = y_train_check[
    sample_idx
]

print(
    "\nTraining sample:",
    X_check_sample.shape
)

print(
    "Training positives:",
    y_check_sample.sum()
)

# ------------------------------------------------------------
# Train V4
# ------------------------------------------------------------

model_check = HistGradientBoostingClassifier(
    learning_rate=0.04,
    max_iter=650,
    max_leaf_nodes=127,
    max_depth=None,
    min_samples_leaf=35,
    l2_regularization=2.0,
    max_bins=255,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=40,
    random_state=RANDOM_STATE + 9001,
    verbose=1
)

start = time.time()

model_check.fit(
    X_check_sample,
    y_check_sample
)

print(
    f"\nTraining time: {time.time() - start:.1f}s"
)

# ------------------------------------------------------------
# Validation probabilities
# ------------------------------------------------------------

valid_prob = model_check.predict_proba(
    X_valid_check
)[:, 1]

# ------------------------------------------------------------
# Search threshold
# ------------------------------------------------------------

threshold_grid = np.arange(
    0.20,
    0.801,
    0.005
)

best_f1 = -1
best_threshold = None
best_precision = None
best_recall = None

for threshold in threshold_grid:

    pred = (
        valid_prob >= threshold
    ).astype(int)

    tp = np.sum(
        (pred == 1) &
        (y_valid_check == 1)
    )

    fp = np.sum(
        (pred == 1) &
        (y_valid_check == 0)
    )

    fn = np.sum(
        (pred == 0) &
        (y_valid_check == 1)
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    if f1 > best_f1:

        best_f1 = f1
        best_threshold = threshold
        best_precision = precision
        best_recall = recall

# ------------------------------------------------------------
# Result
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("47-FEATURE V4 VALIDATION RESULT")
print("=" * 70)

print(
    f"Best F1        : {best_f1:.6f}"
)

print(
    f"Best threshold : {best_threshold:.3f}"
)

print(
    f"Precision      : {best_precision:.6f}"
)

print(
    f"Recall         : {best_recall:.6f}"
)

print(
    f"Previous 50-feature V4 F1 : 0.862433"
)

print("=" * 70)

if best_f1 >= 0.85:
    print("🔥 EXCELLENT — 47-feature model is still ~0.86 F1")
elif best_f1 >= 0.82:
    print("✅ GOOD — model remains strong")
else:
    print("⚠️ 47-feature model lost significant validation performance")

print("=" * 70)

VERIFYING 47-FEATURE V4 F1
Missing split variables: []

Train: (1312167, 47)
Valid: (327257, 47)
Validation positives: 2767

Training sample: (511271, 47)
Training positives: 11271
Binning 0.173 GB of training data: 0.936 s
Binning 0.019 GB of validation data: 0.070 s
Fitting gradient boosted rounds:
Fit 650 trees in 98.346 s, (82550 total leaves)
Time spent computing histograms: 70.522s
Time spent finding best splits:  4.104s
Time spent applying splits:      8.318s
Time spent predicting:           0.987s

Training time: 98.4s

47-FEATURE V4 VALIDATION RESULT
Best F1        : 0.805136
Best threshold : 0.580
Precision      : 0.843021
Recall         : 0.770510
Previous 50-feature V4 F1 : 0.862433
⚠️ 47-feature model lost significant validation performance


In [138]:
print("Hello World")

Hello World


In [140]:
# SAFE SEARCH — does NOT copy the entire notebook history
matches = []

start = max(1, len(In) - 100)

for i in range(start, len(In)):
    cell = In[i]
    if "date_target_rate_loo" in cell:
        matches.append(i)

print("Notebook history length:", len(In))
print("Checked cells:", start, "to", len(In) - 1)
print("Matching cells:", matches)

for i in matches:
    print("\n" + "="*80)
    print("CELL", i)
    print("="*80)
    print(In[i][:5000])

Notebook history length: 141
Checked cells: 41 to 140
Matching cells: [87, 89, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 123, 124, 125, 126, 127, 136, 137, 139, 140]

CELL 87
# ============================================================
# CELL 86 — FINAL V4 TEST PREDICTIONS (FIXED)
# ============================================================

print("=" * 70)
print("FINAL V4 TEST PREDICTIONS")
print("=" * 70)

# ------------------------------------------------------------
# IMPORTANT:
# hgb_features contains a few train-only / target-derived columns
# that cannot legitimately be constructed for test.
# Remove them for prediction.
# ------------------------------------------------------------

TRAIN_ONLY_FEATURES = [
    "_original_row",
    "_row_in_date",
    "date_target_rate_loo"
]

# Features actually available in BOTH train and test
final_test_features = [
    f for f in hgb_features
    if f not in

In [141]:
# ============================================================
# SAFE RECOVERY — CHECK EARLY NOTEBOOK CELLS ONLY
# ============================================================

print("=" * 80)
print("SEARCHING CELLS 1–40 FOR ORIGINAL FEATURE CREATION")
print("=" * 80)

matches = []

for i in range(1, min(41, len(In))):
    cell = In[i]

    if (
        "date_target_rate_loo" in cell
        or "target_rate" in cell
        or "target encoding" in cell.lower()
        or "target rate" in cell.lower()
        or "groupby" in cell.lower() and "target" in cell.lower()
    ):
        matches.append(i)

print("Matching cells:", matches)

for i in matches:
    print("\n" + "=" * 80)
    print(f"🔥 POSSIBLE ORIGINAL CELL {i}")
    print("=" * 80)
    print(In[i])

SEARCHING CELLS 1–40 FOR ORIGINAL FEATURE CREATION
Matching cells: [6, 7, 9, 12, 15, 16, 17, 18, 23]

🔥 POSSIBLE ORIGINAL CELL 6
# ============================================================
# V2 - CELL 6
# COMPETITION-LIKE DATE-WISE VALIDATION SPLIT
# ============================================================

print("=" * 70)
print("CREATING DATE-WISE VALIDATION SPLIT")
print("=" * 70)


# ------------------------------------------------------------
# Preserve original row order
# ------------------------------------------------------------

train_v2["_original_row"] = np.arange(
    len(train_v2),
    dtype=np.int32
)


# ------------------------------------------------------------
# Position within each date
# ------------------------------------------------------------

train_v2["_row_in_date"] = (
    train_v2
    .groupby("Date", sort=False)
    .cumcount()
)


# ------------------------------------------------------------
# Deterministic 80/20 split
# ------------------------

In [142]:
# ============================================================
# CELL 1/7 — RESTORE THE ORIGINAL 50-FEATURE PIPELINE
# ============================================================

print("=" * 80)
print("RESTORING FULL 50-FEATURE PIPELINE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Ensure Date is datetime
# ------------------------------------------------------------

train_v2["Date"] = pd.to_datetime(train_v2["Date"])
test_v2["Date"] = pd.to_datetime(test_v2["Date"])

# ------------------------------------------------------------
# 2. Reconstruct original row position
# ------------------------------------------------------------

train_v2["_original_row"] = np.arange(
    len(train_v2),
    dtype=np.int32
)

test_v2["_original_row"] = np.arange(
    len(test_v2),
    dtype=np.int32
)

# ------------------------------------------------------------
# 3. Reconstruct row position within each date
# ------------------------------------------------------------

train_v2["_row_in_date"] = (
    train_v2
    .groupby("Date", sort=False)
    .cumcount()
)

test_v2["_row_in_date"] = (
    test_v2
    .groupby("Date", sort=False)
    .cumcount()
)

# ------------------------------------------------------------
# 4. FULL-TRAIN date statistics
#
# For final model:
# training date statistics come from ALL available train rows.
# ------------------------------------------------------------

y_full = (
    train_v2["target"]
    .astype(str)
    .astype(int)
)

date_stats_full = (
    pd.DataFrame({
        "Date": train_v2["Date"].values,
        "_target": y_full.values
    })
    .groupby("Date")["_target"]
    .agg(
        date_anomaly_count="sum",
        date_observation_count="count"
    )
)

date_stats_full["date_anomaly_rate"] = (
    date_stats_full["date_anomaly_count"]
    /
    date_stats_full["date_observation_count"]
)

global_anomaly_rate_full = y_full.mean()

# ------------------------------------------------------------
# 5. Training LOO feature
# ------------------------------------------------------------

train_date_counts = (
    train_v2["Date"]
    .map(date_stats_full["date_observation_count"])
    .to_numpy()
)

train_date_anomaly_counts = (
    train_v2["Date"]
    .map(date_stats_full["date_anomaly_count"])
    .to_numpy()
)

train_date_target = y_full.to_numpy()

train_v2["date_target_rate_loo"] = (
    train_date_anomaly_counts - train_date_target
) / np.maximum(
    train_date_counts - 1,
    1
)

# ------------------------------------------------------------
# 6. Test date feature
#
# Test has no target, so use full training-date anomaly rate.
# ------------------------------------------------------------

test_v2["date_target_rate_loo"] = (
    test_v2["Date"]
    .map(date_stats_full["date_anomaly_rate"])
    .fillna(global_anomaly_rate_full)
    .astype(float)
    .values
)

# ------------------------------------------------------------
# 7. Final feature verification
# ------------------------------------------------------------

FINAL_FEATURES_50 = [
    f for f in hgb_features
    if f in train_v2.columns
    and f in test_v2.columns
]

print("\nFeature count:", len(FINAL_FEATURES_50))

print("\nMissing from TRAIN:")
print([
    f for f in hgb_features
    if f not in train_v2.columns
])

print("\nMissing from TEST:")
print([
    f for f in hgb_features
    if f not in test_v2.columns
])

print("\nDate TE:")
print("Global train anomaly rate:",
      f"{global_anomaly_rate_full:.8f}")

print("Train date TE range:",
      f"{train_v2['date_target_rate_loo'].min():.8f}",
      "to",
      f"{train_v2['date_target_rate_loo'].max():.8f}")

print("Test date TE range:",
      f"{test_v2['date_target_rate_loo'].min():.8f}",
      "to",
      f"{test_v2['date_target_rate_loo'].max():.8f}")

assert len(FINAL_FEATURES_50) == 50

print("\n🔥 FULL 50-FEATURE PIPELINE RESTORED.")

RESTORING FULL 50-FEATURE PIPELINE

Feature count: 50

Missing from TRAIN:
[]

Missing from TEST:
[]

Date TE:
Global train anomaly rate: 0.00856276
Train date TE range: 0.00000000 to 0.12738302
Test date TE range: 0.00000000 to 0.12727273

🔥 FULL 50-FEATURE PIPELINE RESTORED.


In [143]:
# ============================================================
# CELL 2/7 — FINAL TRAINING SAMPLE
# ALL POSITIVES + 500K NEGATIVES
# ============================================================

print("=" * 80)
print("BUILDING FINAL V4 TRAINING SAMPLE")
print("=" * 80)

# Numeric target
y_full_numeric = (
    train_v2["target"]
    .astype(str)
    .astype(int)
    .to_numpy()
)

positive_idx = np.flatnonzero(y_full_numeric == 1)
negative_idx = np.flatnonzero(y_full_numeric == 0)

print("Total training rows:", len(train_v2))
print("Total positives:", len(positive_idx))
print("Total negatives:", len(negative_idx))

# Keep ALL positives
# Sample 500K negatives
NEGATIVE_SAMPLE_SIZE = 500_000

rng_final = np.random.default_rng(RANDOM_STATE + 999)

sampled_negative_idx = rng_final.choice(
    negative_idx,
    size=NEGATIVE_SAMPLE_SIZE,
    replace=False
)

final_train_idx = np.concatenate([
    positive_idx,
    sampled_negative_idx
])

# Shuffle final sample
rng_final.shuffle(final_train_idx)

X_final_train = train_v2.iloc[
    final_train_idx
][FINAL_FEATURES_50].to_numpy(dtype=np.float64)

y_final_train = y_full_numeric[final_train_idx]

X_final_test = test_v2[
    FINAL_FEATURES_50
].to_numpy(dtype=np.float64)

print("\nFinal training shape:", X_final_train.shape)
print("Final test shape:", X_final_test.shape)

print("\nFinal training class distribution:")
print(pd.Series(y_final_train).value_counts().sort_index())

print(
    "\nFinal positive rate:",
    f"{y_final_train.mean():.6%}"
)

assert X_final_train.shape[1] == 50
assert X_final_test.shape[1] == 50
assert len(X_final_train) == len(y_final_train)

print("\n🔥 FINAL TRAINING DATA READY.")

BUILDING FINAL V4 TRAINING SAMPLE
Total training rows: 1639424
Total positives: 14038
Total negatives: 1625386

Final training shape: (514038, 50)
Final test shape: (409856, 50)

Final training class distribution:
0    500000
1     14038
Name: count, dtype: int64

Final positive rate: 2.730927%

🔥 FINAL TRAINING DATA READY.


In [144]:
# ============================================================
# CELL 3/7 — V4 MODEL #1
# ============================================================

print("=" * 80)
print("TRAINING V4 MODEL #1")
print("=" * 80)

from sklearn.ensemble import HistGradientBoostingClassifier
import time

v4_model_1 = HistGradientBoostingClassifier(
    learning_rate=0.04,
    max_iter=650,
    max_leaf_nodes=127,
    max_depth=None,
    min_samples_leaf=35,
    l2_regularization=2.0,
    max_bins=255,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=40,
    random_state=RANDOM_STATE + 300,
    verbose=1
)

start_time = time.time()

v4_model_1.fit(
    X_final_train,
    y_final_train
)

elapsed = time.time() - start_time

print("\n" + "=" * 80)
print("V4 MODEL #1 COMPLETE")
print("=" * 80)
print(f"Training time: {elapsed:.1f} seconds")
print("Iterations:", v4_model_1.n_iter_)
print("Training score:", v4_model_1.score(X_final_train, y_final_train))

TRAINING V4 MODEL #1
Binning 0.185 GB of training data: 1.030 s
Binning 0.021 GB of validation data: 0.070 s
Fitting gradient boosted rounds:
Fit 650 trees in 125.795 s, (82550 total leaves)
Time spent computing histograms: 94.082s
Time spent finding best splits:  4.931s
Time spent applying splits:      11.044s
Time spent predicting:           1.098s

V4 MODEL #1 COMPLETE
Training time: 125.8 seconds
Iterations: 650
Training score: 0.9993502425890692


In [145]:
# ============================================================
# CELL 4/7 — V4 MODEL #2
# ============================================================

print("=" * 80)
print("TRAINING V4 MODEL #2")
print("=" * 80)

v4_model_2 = HistGradientBoostingClassifier(
    learning_rate=0.04,
    max_iter=650,
    max_leaf_nodes=127,
    max_depth=None,
    min_samples_leaf=35,
    l2_regularization=2.0,
    max_bins=255,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=40,
    random_state=RANDOM_STATE + 400,
    verbose=1
)

start_time = time.time()

v4_model_2.fit(
    X_final_train,
    y_final_train
)

elapsed = time.time() - start_time

print("\n" + "=" * 80)
print("V4 MODEL #2 COMPLETE")
print("=" * 80)
print(f"Training time: {elapsed:.1f} seconds")
print("Iterations:", v4_model_2.n_iter_)
print("Training score:", v4_model_2.score(X_final_train, y_final_train))

TRAINING V4 MODEL #2
Binning 0.185 GB of training data: 1.025 s
Binning 0.021 GB of validation data: 0.066 s
Fitting gradient boosted rounds:
Fit 650 trees in 125.630 s, (82550 total leaves)
Time spent computing histograms: 95.114s
Time spent finding best splits:  5.115s
Time spent applying splits:      9.357s
Time spent predicting:           1.135s

V4 MODEL #2 COMPLETE
Training time: 125.6 seconds
Iterations: 650
Training score: 0.9992821542376245


In [146]:
# ============================================================
# CELL 5/7 — V4 MODEL #3
# ============================================================

print("=" * 80)
print("TRAINING V4 MODEL #3")
print("=" * 80)

v4_model_3 = HistGradientBoostingClassifier(
    learning_rate=0.04,
    max_iter=650,
    max_leaf_nodes=127,
    max_depth=None,
    min_samples_leaf=35,
    l2_regularization=2.0,
    max_bins=255,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=40,
    random_state=RANDOM_STATE + 500,
    verbose=1
)

start_time = time.time()

v4_model_3.fit(
    X_final_train,
    y_final_train
)

elapsed = time.time() - start_time

print("\n" + "=" * 80)
print("V4 MODEL #3 COMPLETE")
print("=" * 80)
print(f"Training time: {elapsed:.1f} seconds")
print("Iterations:", v4_model_3.n_iter_)
print("Training score:", v4_model_3.score(X_final_train, y_final_train))

TRAINING V4 MODEL #3
Binning 0.185 GB of training data: 1.029 s
Binning 0.021 GB of validation data: 0.069 s
Fitting gradient boosted rounds:
Fit 650 trees in 126.202 s, (82550 total leaves)
Time spent computing histograms: 95.272s
Time spent finding best splits:  5.069s
Time spent applying splits:      9.938s
Time spent predicting:           1.111s

V4 MODEL #3 COMPLETE
Training time: 126.2 seconds
Iterations: 650
Training score: 0.9993833140740568


In [147]:
# ============================================================
# CELL 6/7 — FINAL ENSEMBLE PREDICTIONS + THRESHOLD ANALYSIS
# ============================================================

print("=" * 80)
print("GENERATING FINAL V4 ENSEMBLE PREDICTIONS")
print("=" * 80)

start_time = time.time()

p1 = v4_model_1.predict_proba(X_final_test)[:, 1]
p2 = v4_model_2.predict_proba(X_final_test)[:, 1]
p3 = v4_model_3.predict_proba(X_final_test)[:, 1]

test_pred_prob = (
    p1 + p2 + p3
) / 3.0

elapsed = time.time() - start_time

print(f"\nPrediction time: {elapsed:.1f} seconds")

print("\nProbability distribution:")
print("Mean   :", f"{np.mean(test_pred_prob):.8f}")
print("Median :", f"{np.median(test_pred_prob):.8f}")
print("P95    :", f"{np.percentile(test_pred_prob, 95):.8f}")
print("P99    :", f"{np.percentile(test_pred_prob, 99):.8f}")
print("P99.5  :", f"{np.percentile(test_pred_prob, 99.5):.8f}")
print("P99.9  :", f"{np.percentile(test_pred_prob, 99.9):.8f}")
print("Max    :", f"{np.max(test_pred_prob):.8f}")

# ------------------------------------------------------------
# Threshold analysis
# ------------------------------------------------------------

thresholds = [
    0.50,
    0.525,
    0.55,
    0.575,
    0.60,
    0.625,
    0.65,
    0.675,
    0.70,
    0.725,
    0.75
]

print("\n" + "-" * 80)
print("THRESHOLD ANALYSIS")
print("-" * 80)

for threshold in thresholds:
    count = int(np.sum(test_pred_prob >= threshold))
    rate = count / len(test_pred_prob)

    print(
        f"Threshold {threshold:.3f} | "
        f"Anomalies: {count:5d} | "
        f"Rate: {rate:.6%}"
    )

# Based on robust clean + independent validation
FINAL_THRESHOLD = 0.625

final_anomaly_count = int(
    np.sum(test_pred_prob >= FINAL_THRESHOLD)
)

print("\n" + "=" * 80)
print("SELECTED FINAL THRESHOLD:", FINAL_THRESHOLD)
print("Predicted anomalies:", final_anomaly_count)
print(
    "Predicted anomaly rate:",
    f"{final_anomaly_count / len(test_pred_prob):.6%}"
)
print("=" * 80)

GENERATING FINAL V4 ENSEMBLE PREDICTIONS

Prediction time: 49.9 seconds

Probability distribution:
Mean   : 0.00843332
Median : 0.00001080
P95    : 0.00574587
P99    : 0.23930571
P99.5  : 0.62566695
P99.9  : 0.99926369
Max    : 0.99998950

--------------------------------------------------------------------------------
THRESHOLD ANALYSIS
--------------------------------------------------------------------------------
Threshold 0.500 | Anomalies:  2509 | Rate: 0.612166%
Threshold 0.525 | Anomalies:  2407 | Rate: 0.587279%
Threshold 0.550 | Anomalies:  2307 | Rate: 0.562881%
Threshold 0.575 | Anomalies:  2216 | Rate: 0.540678%
Threshold 0.600 | Anomalies:  2130 | Rate: 0.519695%
Threshold 0.625 | Anomalies:  2051 | Rate: 0.500420%
Threshold 0.650 | Anomalies:  1977 | Rate: 0.482365%
Threshold 0.675 | Anomalies:  1907 | Rate: 0.465285%
Threshold 0.700 | Anomalies:  1844 | Rate: 0.449914%
Threshold 0.725 | Anomalies:  1785 | Rate: 0.435519%
Threshold 0.750 | Anomalies:  1716 | Rate: 0.4186

In [149]:
# ============================================================
# CELL 7/7 — FINAL KAGGLE SUBMISSION
# ============================================================

print("=" * 80)
print("CREATING FINAL SUBMISSION")
print("=" * 80)

submission = pd.DataFrame({
    "ID": test_v2["ID"].values,
    "target": (
        test_pred_prob >= FINAL_THRESHOLD
    ).astype(int)
})

# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------

assert submission.shape == (409856, 2)
assert submission.columns.tolist() == ["ID", "target"]
assert submission["ID"].is_unique
assert submission["target"].isin([0, 1]).all()

print("\nSubmission shape:", submission.shape)

print("\nTarget distribution:")
print(submission["target"].value_counts().sort_index())

print(
    "\nPredicted anomaly count:",
    int(submission["target"].sum())
)

print(
    "Predicted anomaly rate:",
    f"{submission['target'].mean():.6%}"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

submission_path = "/kaggle/working/submission_v4_50f.csv"

submission.to_parquet(
    submission_path,
    index=False
)

print("\nSaved to:")
print(submission_path)

print("\nSubmission preview:")
display(submission.head(10))

print("\n🔥 FINAL 50-FEATURE V4 SUBMISSION READY.")

CREATING FINAL SUBMISSION

Submission shape: (409856, 2)

Target distribution:
target
0    407805
1      2051
Name: count, dtype: int64

Predicted anomaly count: 2051
Predicted anomaly rate: 0.500420%

Saved to:
/kaggle/working/submission_v4_50f.csv

Submission preview:


,ID,target
0,0,0
1,1,1
2,2,0
3,3,0
4,4,1
5,5,1
6,6,0
7,7,0
8,8,1
9,9,1



🔥 FINAL 50-FEATURE V4 SUBMISSION READY.


In [150]:
# ============================================================
# EMERGENCY THRESHOLD CHECK
# ============================================================

thresholds = np.arange(0.20, 0.701, 0.025)

print("=" * 80)
print("50-FEATURE MODEL — WIDE THRESHOLD ANALYSIS")
print("=" * 80)

for t in thresholds:
    count = int((test_pred_prob >= t).sum())
    rate = count / len(test_pred_prob)

    print(
        f"Threshold {t:.3f} | "
        f"Anomalies {count:5d} | "
        f"Rate {rate:.6%}"
    )

50-FEATURE MODEL — WIDE THRESHOLD ANALYSIS
Threshold 0.200 | Anomalies  4528 | Rate 1.104778%
Threshold 0.225 | Anomalies  4236 | Rate 1.033534%
Threshold 0.250 | Anomalies  3991 | Rate 0.973757%
Threshold 0.275 | Anomalies  3784 | Rate 0.923251%
Threshold 0.300 | Anomalies  3593 | Rate 0.876649%
Threshold 0.325 | Anomalies  3435 | Rate 0.838099%
Threshold 0.350 | Anomalies  3281 | Rate 0.800525%
Threshold 0.375 | Anomalies  3127 | Rate 0.762951%
Threshold 0.400 | Anomalies  2973 | Rate 0.725377%
Threshold 0.425 | Anomalies  2855 | Rate 0.696586%
Threshold 0.450 | Anomalies  2723 | Rate 0.664380%
Threshold 0.475 | Anomalies  2613 | Rate 0.637541%
Threshold 0.500 | Anomalies  2509 | Rate 0.612166%
Threshold 0.525 | Anomalies  2407 | Rate 0.587279%
Threshold 0.550 | Anomalies  2307 | Rate 0.562881%
Threshold 0.575 | Anomalies  2216 | Rate 0.540678%
Threshold 0.600 | Anomalies  2130 | Rate 0.519695%
Threshold 0.625 | Anomalies  2051 | Rate 0.500420%
Threshold 0.650 | Anomalies  1977 | Rat